# GLiNER Interview-Anonymisierung (v8.17, PHONE-Resolver + v8.16-Basis)

> **Google-Colab-Portierung:** Alle Azure-/Databricks-spezifischen Teile (Key Vault, Blob Storage, dbutils, DBFS) sind auskommentiert und kommentiert. LLM-Aufrufe laufen über die OpenAI-API (API-Key via Colab Secrets: `OPENAI_API_KEY`). Verzeichnisse liegen relativ zum Startverzeichnis: `./gliner_input` (Input), `./gliner_output` (Output), `./models` (GLiNER-Cache). Laufzeit-Parameter (`entity_types`, `runtime_mode`, `input_file_path`) werden über Colab-Formularfelder in Zelle 2 gesetzt. `runtime_mode=prod` wirft einen Fehler.


Anonymisierung langer (deutscher / mehrsprachiger) Interview-Transkripte mit **GLiNER PII**.

## Änderungen in v8.17

- **PHONE-Resolver als führendes Bewertungssystem**:
  - GLiNER und PHONE-Regex-Recall liefern nur noch Evidenz.
  - Die finale PHONE-Entscheidung erfolgt vorkommensgenau über `start`/`end`, Kontext und Quellenlage.
  - Dadurch wird dieselbe Ziffernfolge je nach Kontext unterschiedlich behandelt, z. B. `Rufnummer 4443 ...` anonymisieren, `Kreditkartennummer 4443 ...` nicht anonymisieren.
- **Konfliktauflösung für PHONE**:
  - `GLiNER=True / Regex=False` und `GLiNER=False / Regex=True` werden nicht mehr als zwei finale Bewertungen ausgewiesen.
  - Stattdessen erzeugt der Resolver eine finale Entscheidung pro konkretem Textvorkommen.
- **PHONE-LLM bleibt auf Grauzonen beschränkt**:
  - Klare Telefonnummern und klare Nicht-Telefonnummern werden deterministisch entschieden.
  - Nur ambige Kandidaten gehen optional ans PHONE-LLM.
- **HTML-/Diagnoseausgabe verbessert**:
  - Score-Übersicht zeigt für PHONE Resolver-Entscheidung, Quellen, Start/Ende und Filterdetails.
  - Entfernte PHONE-Einträge zeigen ebenfalls die finale Resolver-Entscheidung statt isolierter GLiNER-/Regex-Einzelbewertungen.
- **v8.16-Basis beibehalten**:
  - Runtime-Schalter `local`/`prod` bleibt erhalten.
  - MAIL-Syntaxfilter bleibt unverändert.
  - PERSON-Pipeline, PERSON-Propagation und PERSON-LLM-Adjudication bleiben unverändert.

## Betriebsmodus

Das Notebook hat einen expliziten Umschalter `runtime_mode`:

- `local` = Test-/Weiterentwicklungsbetrieb
  - keine Blob-Schreibzugriffe
  - RexGuard-TXT wird lokal in `gliner_output` geschrieben
  - optionales Vergleichs-HTML wird lokal in `gliner_output` geschrieben
- `prod` = Produktivbetrieb / v7-kompatibler Backend-Contract
  - RexGuard-TXT wird lokal im DBFS-Arbeitsverzeichnis geschrieben und anschließend in Container `data` hochgeladen
  - optionales Vergleichs-HTML wird lokal im DBFS-Arbeitsverzeichnis geschrieben und anschließend nach `logs/html/YYYY-MM-DD/...` hochgeladen
  - Container-Check/-Erstellung ist nur in diesem Modus aktiv

Der Standard ist bewusst `local`, damit Tests und Weiterentwicklung keine Daten in den Blob Storage schreiben. Für realen Betrieb den Notebook-Parameter `runtime_mode` auf `prod` setzen.

## Änderungen gegenüber v6 (Integration in Azure/Databricks Backend)


- **Kein Übersetzungs-Step mehr**: GLiNER ist mehrsprachig → **Azure OpenAI / GPT-4.1-mini wird nicht benötigt**.
- **Input bleibt v7-kompatibel**:
  - Wenn `input_file_path` leer ist, wird wie in v7 automatisch der neueste Blob aus Container **`data`** verarbeitet.
  - Ein expliziter `input_file_path` bleibt als optionaler Debug-/Sonderfall erhalten.
- **Output abhängig von `runtime_mode`**:
  - `local`: Output lokal in `gliner_output`, keine Blob-Uploads.
  - `prod`: Output wieder zurück in Container **`data`**, HTML optional nach **`logs/html/YYYY-MM-DD/...`**.
- **Entitätensteuerung über Databricks Notebook-Parameter `entity_types`** (wie vom Backend gesendet):
  - Mapping: `1=PER, 2=ORG, 3=LOC, 4=PHO, 5=MAIL`
  - Feste Abarbeitungsreihenfolge: **PER → ORG → LOC → PHO → MAIL**
- **Modell-Laden „wie DeBERTa“ (lokal / DBFS)**:
  - lädt bevorzugt von `/dbfs/my_models/...` (wenn vorhanden), sonst Fallback auf HuggingFace-ID.
- **Vergleichs-HTML bleibt optional**.
- **RexGuard-Output als Sterne-Maskierung (`*`)** (wie bisher erwartet).

## Beibehaltene Features

- **Chunking unverändert** (satzbasiert, max. Wörter/Chunk, Overlap)
- Separate Inferenz-Durchläufe pro Label-Gruppe
- Per-Label Thresholds
- **PER-Negativ-Labels unverändert** (essentiell für bessere Disambiguierung)
- Pronomen-Blocklist + Whitelist (wortbasiert)
- Konfidenz-Warnung + Vergleichs-HTML (optional)

## Workflow

1. Input-Text wie v7 aus dem neuesten Blob im Container `data` laden; optional expliziter `input_file_path` für Debug/Sonderfälle
2. Satzbasiertes Chunking (max ~250 Wörter, Overlap)
3. Pro aktivierter Entity-Gruppe ein eigener Inferenz-Durchlauf (Labels + optionale Negativ-Labels)
4. Merge + Konfliktauflösung
5. Post-Processing: Threshold, Mindestlänge, Pronomen-Blocklist, interne Whitelist
6. Name-Propagation aus sicheren Vollnamen
7. Externe aktive Entity-Whitelist laden und anwenden
8. LLM-Adjudication nur für ambige PERSON-Kandidaten
9. Anonymisierung (Entities → `*`-Maskierung)
10. Output abhängig von `runtime_mode`: lokal nach `gliner_output` oder produktiv in Blob-Container `data` plus optional HTML nach `logs`


**v8.11:** Die LLM-Exemption für sichere PERSON-Namen basiert nicht mehr allein auf Score >= 0.85, sondern auf einem temporären Speaker-Roster aus Sprecherzeilen des aktuellen Transkripts.


**v8.12:** LLM-Decision-Labels umbenannt: `KEEP` → `ANON`, `DROP` → `WHITEL`; `REVIEW` bleibt für unsichere Fälle, die anonymisiert und als Kandidaten geprüft werden.


**v8.13 Runtime-Switch:** Standardmäßig lokaler Test-/Weiterentwicklungsbetrieb ohne Blob-Schreibzugriffe; Produktivbetrieb per `runtime_mode=prod` wieder v7-kompatibel mit Upload in `data`/`logs`.


## Zelle 1: Installation
**Jede Zelle einzeln ausführen, dann Kernel neu starten.**

In [1]:
%env PYTHONIOENCODING=utf-8
%env PYTORCH_ENABLE_MPS_FALLBACK=1

env: PYTHONIOENCODING=utf-8
env: PYTORCH_ENABLE_MPS_FALLBACK=1


In [2]:
!pip install openai -q
!pip install --upgrade openai -q
# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# !pip install azure-identity -q
# !pip install azure-keyvault-secrets -q

# torch, accelerate NICHT upgraden — Cluster-Versionen beibehalten!
# Azure-Pakete (azure-identity, azure-keyvault-secrets, azure-storage-blob) aus der
# Installationszeile entfernt — werden in Colab nicht benötigt:
%pip install gliner seqeval tqdm transformers==4.44.2

from openai import OpenAI
import pandas as pd
import re
from pathlib import Path

# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# from openai import AzureOpenAI
# from azure.identity import DefaultAzureCredential, get_bearer_token_provider
# from azure.keyvault.secrets import SecretClient


Note: you may need to restart the kernel to use updated packages.


In [3]:
# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# dbutils.library.restartPython()
#
# Hinweis Colab: Nach der Paketinstallation in Zelle 1 ggf. die Laufzeit
# manuell neu starten (Menü: Laufzeit > Sitzung neu starten), falls
# Importfehler wegen bereits geladener Paketversionen auftreten.


In [4]:
import os

# ============================================================
# LLM-Client: lokales Ollama (OpenAI-kompatibler Endpoint)  [Mac-Portierung]
# ============================================================
# Entity-Judge = qwen2.5 (Projektplan). Kein API-Key noetig.
# Voraussetzung: `ollama serve` laeuft und das Modell ist gepullt.
# Steuerbar ueber Env-Vars OLLAMA_API_BASE und ENTITY_JUDGE_MODEL.
from openai import OpenAI

OLLAMA_API_BASE = os.getenv("OLLAMA_API_BASE", "http://127.0.0.1:11434")
deployment = os.getenv("ENTITY_JUDGE_MODEL", "qwen2.5:32b")

client = OpenAI(base_url=f"{OLLAMA_API_BASE}/v1", api_key="ollama")  # api_key: Pflichtfeld, Wert beliebig
print(f"LLM-Client (Ollama) initialisiert. Modell: {deployment}  @ {OLLAMA_API_BASE}/v1")

# --- Frueher: OpenAI-Cloud via Colab Secrets (deaktiviert, zur Referenz) ---
# from google.colab import userdata
# _openai_api_key = userdata.get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY", "")
# deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
# client = OpenAI(api_key=_openai_api_key)

# ============================================================
# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# Ursprüngliche Azure-OpenAI-Initialisierung mit Key Vault:
# ============================================================
# from openai import AzureOpenAI
# from azure.identity import DefaultAzureCredential, get_bearer_token_provider
# from azure.keyvault.secrets import SecretClient
#
# vault_url = "https://anonymization.vault.azure.net/"
# # secret_name = "open-api-key-GPT-4o-mini"
# # secret_name = "openai-api-key" # GPT 4o
# secret_name = "open-api-key-GPT-4-1"
# # secret_name = "open-api-key-GPT-4-1-mini"
# # secret_name = "open-api-key-GPT-5-mini"
#
# credential = DefaultAzureCredential()
# secret_client = SecretClient(vault_url=vault_url, credential=credential)
# retrieved_openai_key = secret_client.get_secret(secret_name)
#
# # endpoint = os.getenv("ENDPOINT_URL", "https://open-ai-studio-access.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-08-01-preview")
# # endpoint = os.getenv("ENDPOINT_URL", "https://rg-france-central-gpt4-name.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-08-01-preview")
# # endpoint = os.getenv("ENDPOINT_URL", "https://mso1k-m7h3689g-swedencentral.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-08-01-preview")
# endpoint = os.getenv("ENDPOINT_URL", "https://mso1k-m7h3689g-swedencentral.cognitiveservices.azure.com/")
# #  # api_version="2024-12-01-preview" for GPT4.1
#
# # Initialize Azure OpenAI Service client with Entra ID authentication
# token_provider = get_bearer_token_provider(
#     DefaultAzureCredential(),
#     "https://cognitiveservices.azure.com/.default"
# )
#
# client = AzureOpenAI(
#     azure_endpoint=endpoint,
#     api_key=retrieved_openai_key.value,
#     api_version="2024-12-01-preview"
# )
# # print(retrieved_openai_key.value)


# ##Beispiel für Aufruf später:
    # for attempt in range(max_retries):
    #     try:
    #         response = client.chat.completions.create(
    #             model=deployment,
    #             messages=[
    #                 {
    #                     "role": "system",
    #                     "content": r"""You are an expert in chat analysis and chat classification.
    #                     ...
    #             ],
    #             ### GPT-4.1 ###
    #             temperature=0.1,    # GPT-4.1...
    #             max_tokens=3000,    # GPT-4.1...
    #             top_p=0.2,    # GPT-4.1...
    #             presence_penalty=0.0,    # GPT-4.1...
    #             frequency_penalty=0.1,    # GPT-4.1...
    #             response_format={"type": "json_object"} # 4.1 + 5...
    #         )


LLM-Client (Ollama) initialisiert. Modell: qwen2.5:32b  @ http://127.0.0.1:11434/v1


## Zelle 2: Konfiguration

**LABEL_CONFIG Felder:**
- **Key** (z.B. `"person"`): Positive GLiNER-Labels, deren Treffer behalten werden.
- **tag**: Ausgabe-Tag im anonymisierten Text bzw. Vergleichs-HTML, z.B. `[PERSON]`.
- **threshold**: Technischer GLiNER-Erkennungsschwellenwert. Für PERSON bewusst eher recall-orientiert; die fachliche Schärfung passiert danach im Post-Processing.
- **negative_labels** *(optional)*: Wird weiterhin unterstützt, ist für PERSON aber deaktiviert, weil es im aktuellen Dokument keinen Lösungsbeitrag liefert.

**PERSON-Strategie:**
GLiNER sucht nur nach `person`. Unsichere Treffer werden anschließend differenziert bewertet:
- Mehrwortige Namen dürfen einen niedrigeren Score haben.
- Einwortige Namen brauchen entweder höheren Score oder starken Kontext.
- Code-/Abkürzungsformen wie `SGP DM` werden entfernt.


In [5]:
import os
import torch
from datetime import datetime

# ============================================================
# BACKEND / DATABRICKS PARAMETER (Notebook-Params)
# ============================================================
# Das Backend übergibt `entity_types` als kommaseparierten String:
#   1=PER, 2=ORG, 3=LOC, 4=PHO, 5=MAIL
# Die Abarbeitungsreihenfolge MUSS stabil sein:
#   PER → ORG → LOC → PHO → MAIL
# --- COLAB: Laufzeit-Parameter als Formularfeld (ersetzt dbutils.widgets) ---
entity_types = "1"  #@param {type:"string"}
_entity_types_raw = entity_types if str(entity_types).strip() else os.getenv("ENTITY_TYPES", "1")

# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# try:
#     dbutils.widgets.text(
#         "entity_types",
#         "",
#         "Entity Types (1=PER,2=ORG,3=LOC,4=PHO,5=MAIL)"
#     )
#     _entity_types_raw = dbutils.widgets.get("entity_types")
# except Exception:
#     # Fallback (z.B. interaktives Testen außerhalb geplanter Jobs)
#     _entity_types_raw = os.getenv("ENTITY_TYPES", "1")

entity_types_numeric = [x.strip() for x in str(_entity_types_raw).split(",") if x.strip()]

ENTITY_MAPPING = {'1': 'PER', '2': 'ORG', '3': 'LOC', '4': 'PHO', '5': 'MAIL'}
ENTITY_ORDER = ['PER', 'ORG', 'LOC', 'PHO', 'MAIL']

entity_types_selected = []
for code in entity_types_numeric:
    if code in ENTITY_MAPPING:
        entity_types_selected.append(ENTITY_MAPPING[code])
    else:
        print(f"Warning: Entity type '{code}' is not recognized and will be ignored.")

# Eindeutige Auswahl + feste Reihenfolge
_entity_set = set(entity_types_selected)
ENTITY_TYPES_ACTIVE = [et for et in ENTITY_ORDER if et in _entity_set]

print("Aktivierte Entity Types (geordnet):", ENTITY_TYPES_ACTIVE)

# ============================================================
# RUNTIME-MODUS + AZURE BLOB STORAGE + OUTPUT-PFADE
# ============================================================
# runtime_mode steuert, ob dieses Notebook im Test-/Weiterentwicklungsbetrieb
# oder im produktiven Backend-Contract läuft:
#   local = keine Blob-Schreibzugriffe, Output lokal nach gliner_output
#   prod  = v7-kompatibel: TXT nach Container data, HTML nach logs/html/YYYY-MM-DD/...
#
# Input bleibt v7-kompatibel: Wenn input_file_path leer ist, wird der neueste
# Blob aus Container data gelesen. Ein expliziter input_file_path bleibt als
# Debug-/Sonderfall erhalten.
# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# from azure.identity import DefaultAzureCredential
# from azure.keyvault.secrets import SecretClient
# from azure.storage.blob import BlobServiceClient

# --- COLAB: Laufzeit-Parameter als Formularfeld (ersetzt dbutils.widgets) ---
runtime_mode = "local"  #@param ["local", "prod"]
RUNTIME_MODE = runtime_mode

# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# try:
#     dbutils.widgets.text(
#         "runtime_mode",
#         "local",
#         "Runtime mode: local=no Blob writes, prod=Blob/DBFS output"
#     )
#     RUNTIME_MODE = dbutils.widgets.get("runtime_mode")
# except Exception:
#     RUNTIME_MODE = os.getenv("RUNTIME_MODE", "local")

RUNTIME_MODE = str(RUNTIME_MODE).strip().lower()
if RUNTIME_MODE not in {"local", "prod"}:
    raise ValueError("runtime_mode must be 'local' or 'prod'.")

# --- COLAB: prod ist ohne Azure Blob Storage nicht lauffähig ---
if RUNTIME_MODE == "prod":
    raise RuntimeError(
        "runtime_mode='prod' wird in der Colab-Portierung nicht unterstützt "
        "(Azure Blob Storage ist deaktiviert). Bitte runtime_mode='local' verwenden."
    )

ALLOW_BLOB_WRITES = RUNTIME_MODE == "prod"

# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# VAULT_URL = "https://anonymization.vault.azure.net/"
# DATA_CONTAINER_NAME = "data"
# LOGS_CONTAINER_NAME = "logs"
# HTML_PREFIX = "html"

# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# # credential / secret_client wurden i.d.R. bereits in der Azure-OpenAI-Zelle erzeugt.
# # Für robuste Einzel-/Debug-Ausführung werden sie hier bei Bedarf erneut initialisiert.
# try:
#     credential
# except NameError:
#     credential = DefaultAzureCredential()
#
# try:
#     secret_client
# except NameError:
#     secret_client = SecretClient(vault_url=VAULT_URL, credential=credential)
#
# CONNECTION_STRING = secret_client.get_secret("ConnectionString").value
# blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)

# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# Das DBFS-Arbeitsverzeichnis wurde nur für Blob-Downloads/-Uploads im
# Produktivmodus benötigt und entfällt in Colab.
# LOCAL_WORK_DIR = "/dbfs/FileStore/tables"
# os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# Lokale Artefakte und der Test-/Entwicklungs-Output bleiben im Notebook-Arbeitsbereich.
# Dazu gehören aktive Whitelist, REVIEW-Kandidaten, optionale Diagnose-CSVs und
# im runtime_mode=local auch der RexGuard-TXT-Output plus optionales HTML.
# --- COLAB: Verzeichnisse relativ zum Startverzeichnis des Notebooks ---
LOCAL_ARTIFACT_ROOT = os.getcwd()
# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# LOCAL_ARTIFACT_ROOT = "/Workspace/General/Productive model sources"
LOCAL_INPUT_DIR = os.path.join(LOCAL_ARTIFACT_ROOT, "gliner_input")
LOCAL_OUTPUT_DIR = os.path.join(LOCAL_ARTIFACT_ROOT, "gliner_output")
os.makedirs(LOCAL_INPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

# Primärer lokaler Ablageort für die Ergebnisdateien:
# - local: gliner_output
# Colab-Portierung: prod wirft bereits oben einen Fehler, daher immer gliner_output.
PRIMARY_OUTPUT_DIR = LOCAL_OUTPUT_DIR
# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# PRIMARY_OUTPUT_DIR = LOCAL_OUTPUT_DIR if RUNTIME_MODE == "local" else LOCAL_WORK_DIR
os.makedirs(PRIMARY_OUTPUT_DIR, exist_ok=True)

# --- COLAB: Laufzeit-Parameter als Formularfeld (ersetzt dbutils.widgets) ---
# Leer = neueste Datei aus dem lokalen Input-Verzeichnis gliner_input.
input_file_path = ""  #@param {type:"string"}
INPUT_FILE_PATH = input_file_path if str(input_file_path).strip() else os.getenv("INPUT_FILE_PATH", "")

# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# try:
#     dbutils.widgets.text(
#         "input_file_path",
#         "",
#         "Optional input file path; leer = neuester Blob aus data wie v7"
#     )
#     INPUT_FILE_PATH = dbutils.widgets.get("input_file_path")
# except Exception:
#     INPUT_FILE_PATH = os.getenv("INPUT_FILE_PATH", "")

# ============================================================
# MODELL (lokal / DBFS bevorzugt, sonst HF-ID)
# ============================================================
MODEL_HF_ID = "urchade/gliner_multi_pii-v1"
MODEL_LOCAL_PATH = os.path.join(os.getcwd(), "models", "gliner_multi_pii-v1")  # ggf. anpassen
# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# MODEL_LOCAL_PATH = "/dbfs/my_models/gliner_multi_pii-v1"  # ggf. anpassen
MODEL_LOAD_SOURCE = MODEL_LOCAL_PATH if os.path.isdir(MODEL_LOCAL_PATH) else MODEL_HF_ID

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# LABEL-KONFIGURATION (MASTER) + Aktivierung über entity_types
# ============================================================
# Jedes Label wird als EIGENER Inferenz-Durchlauf ausgeführt.
#
# Key:             Kommaseparierte POSITIV-Labels → werden behalten
# tag:             Ausgabe-Tag im Vergleichs-HTML
# threshold:       Individueller Confidence-Schwellenwert
# negative_labels: Optionale NEGATIV-Labels → laufen im selben
#                  Durchlauf mit, Treffer werden aber verworfen.
#
# PERSON: GLiNER sucht bewusst nur nach dem Label "person".
# Die Schärfung gegen False Positives erfolgt im PERSON-spezifischen
# Post-Processing weiter unten. Zusätzliche Labels/Negativ-Labels sind
# für das aktuelle Dokument deaktiviert, weil sie die Trefferqualität
# verschlechtert bzw. nicht verbessert haben.
LABEL_CONFIG_MASTER = {
    # 1) PER
    "person": {
        "_entity_type": "PER",
        "tag": "[PERSON]",
        "threshold": 0.5,
        "negative_labels": [],
    },
    # 2) ORG (im Backend später deaktiviert, bleibt hier der Vollständigkeit halber)
    "organization": {
        "_entity_type": "ORG",
        "tag": "[ORG]",
        "threshold": 0.635,
        "negative_labels": [],
    },
    # 3) LOC (im Backend später deaktiviert, bleibt hier der Vollständigkeit halber)
    "location": {
        "_entity_type": "LOC",
        "tag": "[LOC]",
        "threshold": 0.89,
        "negative_labels": [],
    },
    # 4) PHO
    "phone number": {
        "_entity_type": "PHO",
        "tag": "[PHONE]",
        "threshold": 0.5,
        "negative_labels": [],
    },
    # 5) MAIL
    "email": {
        "_entity_type": "MAIL",
        "tag": "[MAIL]",
        "threshold": 0.5,
        "negative_labels": [],
    },
}

# Zuordnung Entity → Config Keys (erlaubt später mehrere Varianten pro Entity)
ENTITY_TO_CONFIG_KEYS = {
    "PER": ["person"],
    "ORG": ["organization"],
    "LOC": ["location"],
    "PHO": ["phone number"],
    "MAIL": ["email"],
}

# Aktive LABEL_CONFIG in stabiler Reihenfolge aufbauen
LABEL_CONFIG = {}
for et in ENTITY_ORDER:
    if et not in ENTITY_TYPES_ACTIVE:
        continue
    for ck in ENTITY_TO_CONFIG_KEYS.get(et, []):
        if ck in LABEL_CONFIG_MASTER:
            cfg = dict(LABEL_CONFIG_MASTER[ck])
            cfg.pop("_entity_type", None)
            LABEL_CONFIG[ck] = cfg

print("Aktive Inferenz-Durchläufe (LABEL_CONFIG Keys):", list(LABEL_CONFIG.keys()))

# ============================================================
# PHONE-POST-PROCESSING
# ============================================================
# PHONE wird bewusst hybrid behandelt:
# 1) Regex-Recall-Pass für klar strukturierte Telefonnummern
# 2) harte Syntax-/Kontextvalidierung gegen semantische Telefonbegriffe
# 3) LLM nur für ambige Zahlenkandidaten, nicht für klare Treffer/Nichttreffer
PHONE_ADD_REGEX_RECALL_PASS = True
PHONE_REQUIRE_VALID_NUMBER_SYNTAX = True
PHONE_USE_LLM_FOR_AMBIGUOUS = True

# Ambige PHONE-Kandidaten bleiben bei REVIEW anonymisiert. Das ist im
# Testbetrieb datenschutzorientiert und zugleich diagnostisch auswertbar.
# Erlaubte Werte: "llm_or_review", "llm_or_keep", "llm_or_drop".
PHONE_AMBIGUOUS_POLICY = "llm_or_review"

# E.164 erlaubt max. 15 Ziffern. Kürzere Durchwahlen werden nur bei starkem
# Telefonkontext akzeptiert.
PHONE_MIN_DIGITS = 7
PHONE_MAX_DIGITS = 15
PHONE_SHORT_EXTENSION_MIN_DIGITS = 3
PHONE_SHORT_EXTENSION_MAX_DIGITS = 6
PHONE_CONTEXT_CHARS = 80
PHONE_LLM_BATCH_SIZE = 30

# ============================================================
# MAIL-POST-PROCESSING
# ============================================================
# Test-/Produktiv-unabhängiger Präzisionsfilter für MAIL:
# GLiNER darf weiterhin Kandidaten liefern, final behalten werden aber
# nur echte E-Mail-Adressen mit localpart@domain.tld.
# Dadurch werden generische Begriffe wie "E-Mail-Prozess" oder
# "E-Mail-Vorlagen" nicht anonymisiert.
MAIL_REQUIRE_VALID_ADDRESS_SYNTAX = True

# ============================================================
# HTML-ARTEFAKT (optional)
# ============================================================
# HTML wird optional erzeugt. Ablage/Upload hängt vom runtime_mode ab.
GENERATE_HTML = True  # True: Vergleichs-HTML erzeugen

# Diagnose-CSV-Ausgaben optional.
# Intern werden die DataFrames weiter aufgebaut; standardmäßig werden aber
# keine <input>_entity_list.csv und <input>_removed_entities.csv geschrieben.
# Für Debugging bei Bedarf auf True setzen.
WRITE_ENTITY_DIAGNOSTIC_CSVS = False

# ============================================================
# LLM-ADJUDICATION FÜR PERSON-FALSE-POSITIVES
# ============================================================
# GLiNER bleibt Entity-Kandidatengenerator. Nur ambige PERSON-
# Kandidaten werden zusätzlich per LLM adjudiziert. Sichere Namen
# bleiben direkt in entities_filtered und werden anonymisiert.
USE_LLM_ENTITY_ADJUDICATION = True
LLM_ENTITY_TYPES = {"PER"}

# In dieser Version darf die externe Whitelist ausschließlich PERSON betreffen.
# Damit bleiben ORG/LOC/PHONE/MAIL auch dann unverändert, wenn sie über entity_types aktiviert sind.
EXTERNAL_WHITELIST_ENTITY_TYPES = {"PER"}

# Score-Schwelle bleibt sichtbar, reicht aber ab v8.11 nicht mehr allein für LLM-Exemption.
# Exempt wird nur, wenn zusätzlich ein harter Strukturbeweis vorliegt, z. B. Speaker-Roster.
LLM_EXEMPT_PERSON_MULTI_TOKEN_MIN_SCORE = 0.85

# Temporärer Speaker-Roster pro Transkript:
# Aus Sprecherzeilen wie "Mustermann Max (AbC/DEF3.2)" werden sichere Sprecher-Namen abgeleitet.
# Diese Namen und ihre Namensbestandteile dürfen LLM-exempt bleiben.
USE_SPEAKER_ROSTER_EXEMPTION = True
SPEAKER_ROSTER_MIN_TOKENS = 2
SPEAKER_ROSTER_MAX_TOKENS = 4
LLM_CONTEXT_CHARS_LEFT = 240
LLM_CONTEXT_CHARS_RIGHT = 240
LLM_MAX_CONTEXTS_PER_CANDIDATE = 3
LLM_BATCH_SIZE = 30

EXTERNAL_ENTITY_WHITELIST_PATH = os.path.join(
    LOCAL_OUTPUT_DIR, "entity_whitelist.csv"
)

WHITELIST_CANDIDATES_PATH = os.path.join(
    LOCAL_OUTPUT_DIR, "whitelist_candidates.csv"
)


# ============================================================
# CHUNKING-PARAMETER
# ============================================================
MAX_WORDS_PER_CHUNK = 250   # Max Wörter pro Chunk (GLiNER Limit: 384)
OVERLAP_SENTENCES = 2       # Überlappende Sätze an Chunk-Grenzen

# ============================================================
# SHORT-SEGMENT RECALL-PASS
# ============================================================
# Optionaler zusätzlicher GLiNER-Lauf auf kurzen Segmenten des echten raw_text.
# Ziel: Entities finden, die im langen, verrauschten Produktions-Chunk untergehen.
# Der Recall-Pass entscheidet NICHT selbst über Anonymisierung. Er erzeugt nur
# zusätzliche Kandidaten, die danach durch dieselbe Pipeline laufen:
# Postprocessing -> Whitelist -> LLM -> finale Anonymisierung.
USE_SHORT_SEGMENT_RECALL_PASS = False

# Aktuell auf PER/ORG/LOC vorbereitet. PHONE/MAIL bleiben im normalen Pfad;
# sie sind eher Kandidaten für harte Validierungslogik als für Segment-Recall.
SHORT_SEGMENT_RECALL_ENTITY_TYPES = {"PER", "ORG", "LOC"}

# Sichtbare Stellhebel. Schwellen kommen standardmäßig aus LABEL_CONFIG.
# Overrides sind optional, z.B. {"PER": 0.45, "organization": 0.60}.
SHORT_SEGMENT_RECALL_MAX_CHARS = 320
SHORT_SEGMENT_RECALL_MIN_CHARS = 8
SHORT_SEGMENT_RECALL_BATCH_SIZE = 16
SHORT_SEGMENT_RECALL_THRESHOLD_OVERRIDES = {}

# ============================================================
# POST-PROCESSING FILTER
# ============================================================

# --- 1. Mindestlänge für Entities ---
MIN_ENTITY_LENGTH = 3

# --- 2. Pronomen-Blocklist ---
PRONOUN_BLOCKLIST = {
    # Personalpronomen
    "ich", "du", "er", "sie", "es", "wir", "ihr",
    "mich", "dich", "sich", "uns", "euch",
    "mir", "dir", "ihm", "ihnen",
    "mein", "dein", "sein", "unser", "euer",
    "meine", "deine", "seine", "ihre", "unsere", "eure",
    "meinen", "deinen", "seinen", "ihren", "unseren", "euren",
    "meinem", "deinem", "seinem", "ihrem", "unserem", "eurem",
    "meiner", "deiner", "seiner", "ihrer", "unserer", "eurer",
    # Demonstrativ- und Relativpronomen
    "dieser", "diese", "dieses", "jener", "jene", "jenes",
    "der", "die", "das", "welcher", "welche", "welches", "eine", "einer",
    # Indefinitpronomen
    "man", "jemand", "niemand", "alle", "jeder", "jede", "jedes",
    # Höflichkeitsform
    "Sie",
    # Fragepronomen
    "wer", "wen", "wem", "wessen",
}
PRONOUN_BLOCKLIST_LOWER = {p.lower() for p in PRONOUN_BLOCKLIST}

# --- 3. Konfidenz-Warnung ---
LOW_CONFIDENCE_WARN = 0.6

# ============================================================
# PERSON-SPEZIFISCHES POST-PROCESSING
# ============================================================
# Ziel: Recall-orientierte Anonymisierung von Interviews.
# Grundregel: Wenn GLiNER mit Label "person" etwas findet und der
# technische Schwellenwert erreicht ist, wird es im Zweifel behalten.
# Entfernt werden nur harte Non-Person-Muster wie Codes, IDs,
# reine Abkürzungen oder technische Fragmente.
#
# Hintergrund: In Interviewdaten sind echte Einzelnamen wie "Anna",
# "Anne" oder "Birgit" oft niedriger bewertet als projekt-/toolartige
# False Positives. Deshalb ist eine hohe Einwort-Score-Schwelle hier
# kontraproduktiv.

# Mehrwortige Personennamen wie "Tom Richter" oder "Mustermann Max".
# Recall-orientiert: gleicher Mindestscore wie GLiNER-PER-Threshold.
PERSON_MULTI_TOKEN_MIN_SCORE = 0.5

# Einwortige Treffer werden ebenfalls recall-orientiert behalten.
# Der Kontext dient nicht mehr als Pflichtbedingung, sondern nur noch
# als Diagnose/Begründung in filter_detail.
PERSON_SINGLE_TOKEN_MIN_SCORE = 0.5
PERSON_SINGLE_TOKEN_CONTEXT_MIN_SCORE = 0.5

# Breiterer Kontextausschnitt für Diagnose und spätere Nachanalyse.
PERSON_CONTEXT_CHARS = 160

# Optionaler manueller Allowlist-Hebel für bekannte Interviewteilnehmer.
PERSON_SINGLE_TOKEN_ALLOWLIST = set()

# ============================================================
# NAMENSBESTANDTEILE AUS SICHEREN VOLLNAMEN PROPAGIEREN
# ============================================================
# Wenn GLiNER z.B. "Mustermann Max" oder "Tom Richter" erkennt,
# werden daraus zusätzlich "Mustermann", "Max", "Tom" und "Richter"
# abgeleitet und im gesamten Text anonymisiert. Das deckt Interviewfälle ab,
# in denen eine Person zuerst vollständig und später nur noch mit Vorname
# oder Nachname genannt wird.
PROPAGATE_PERSON_NAME_PARTS = True
PERSON_PROPAGATION_MIN_FULL_NAME_SCORE = 0.5
PERSON_PROPAGATION_MIN_TOKEN_LENGTH = 3

# Tokens, die nicht aus Mehrwort-Treffern propagiert werden sollen.
# Wichtig für Anreden, Rollen und Organisations-/Klammerzusätze.
PERSON_PROPAGATION_TOKEN_BLOCKLIST = {
    "herr", "herrn", "frau", "fr", "dr", "prof", "professor", "doktor",
    "interviewer", "interviewerin", "teilnehmer", "teilnehmerin",
    "kunde", "kundin", "kollege", "kollegin", "mitarbeiter", "mitarbeiterin",
    "team", "projekt", "project", "software", "tool", "system",
    "hip", "pj", "di", "dm", "sgp",
}

# Wörter/Phrasen, die im Nahkontext für eine Person sprechen.
# Diese Liste ist bewusst breit, weil sie nicht mehr hart filtert,
# sondern nur erklärt, warum ein Treffer plausibel ist.
PERSON_CONTEXT_KEYWORDS = {
    # Anrede / Titel
    "herr", "frau", "dr", "prof", "doktor",

    # Rollen / Menschen
    "kollege", "kollegin", "kollegen", "mitarbeiter", "mitarbeiterin",
    "kunde", "kundin", "ansprechpartner", "ansprechpartnerin",
    "teilnehmer", "teilnehmerin", "interviewer", "interviewerin",
    "befragte", "befragter", "speaker", "sprecher", "sprecherin",
    "chef", "chefin", "leiter", "leiterin", "manager", "managerin",
    "teamleiter", "teamleiterin", "kontakt", "kontaktperson", "partner", "partnerin",


    # Kommunikation / Interview-Verben
    "sagt", "sagte", "meint", "meinte", "erzählt", "erzählte",
    "fragt", "fragte", "antwortet", "antwortete", "erklärt", "erklärte",
    "spricht", "sprach", "gesprochen", "telefoniert", "angerufen",
    "geschrieben", "mail", "e-mail", "meeting", "termin",

    # Namensfelder / Stammdaten
    "name", "vorname", "nachname", "namens",

    # generische Präpositions-Kontexte, z.B. "mit Anna", "von Birgit"
    "mit", "von", "an", "bei", "für", "laut", "durch",
}

# ============================================================
# WHITELIST: Begriffe, die NICHT anonymisiert werden sollen
# ============================================================
WHITELIST = {
    # Beispiele — bitte anpassen:
    "Speaker", "Speaker 1", "Speaker 2", "Speaker 3", "Speaker 4", "Speaker 5",
    "Sprecher", "Sprecher 1", "Sprecher 2", "Sprecher 3", "Sprecher 4", "Sprecher 5", "Mhm",
    "Robin", "SOM", "Som", "som", "Somm", "TID", "Tid", "TIT", "Tit",
    "Bosch", "Rexroth", "Bosch Rexroth", "Bosch Rexroth Schweiz AG", "Standort", "Organisation",
    "Bosch-Gruppe", "Firma Bosch Rexroth", "Rexrodt", "Firma", "Hengst", "DBE", "E-Shop", "eshop", "Unternehmen", "Sponsor", "Sponsorin", "Sponsoren", "Gruppenleitung", "Gruppenleiter", "Gruppenleiterin", "Werkleitung", "Werkleiter",
    "Kunde", "Kunden", "Einkäufer", "Name", "Namen", "Nachnamen", "Nachname", "Vornamen", "Vorname", "Kollege X", "Mitarbeiter", "Lieferant", "Lieferanten", "Mitarbeiterin", "Kollege", "Kollegen", "Kollegin", "namens", "Chef", "Chefin", "Mentor", "Mentoren", "Mentorin", "Ansprechpartner", "Endkunde", "Partner", "Endkunden", "Global Account Manager", "Einkaufsleiter", "Vertriebsleiter", "Vertriebsleitung", "Vertriebsmitarbeiter", "Vertriebsmann", "Vertriebler", "Außendienstmitarbeiter", "Mensch", "Menschen", "Herr Zylinder", "Verkäufer", "Vorgesetzten", "Vorgesetzter", "Vorgesetzte", "Person", "Frachtführer", "Teamleiter", "Abteilungsleiter", "Abteilungsleiterin", "Key User", "Key-User", "Keyuser", "Key Userin", "Kundenbetreuer", "Führungskraft", "Versuchsleiterin", "Versuchsleiter", "Leiterin des Versuchsteams", "Leiter des Versuchsteams", "Kundschaft", "Frau", "Herr", "Partnermanager", "Partnermanagerin", "Inder", "Inderin", "Teilnehmer", "Teilnehmerin", "Student", "Studentin", "Studenten", "Kleinkunde", "Kleinkundin", "Kleinkunden",
    "E-Mail", "E-Mails", "Mail", "e-mail", "mail", "e-Mail", "e-mail", "e-Mails", "mail", "mails", "e-mail", "e-mails", "E-Mail-Adresse", "E-Mail-Adressen", "emails", "email", "Mail-Account", "Mail-Accounts", "info@", "E-Mail-Verlauf", "E-Mail-Verkehr", "Mailverkehr", "E-Mail-Erfassung", "BER-E-Mails", "BER-E-Mail", "Mailbox Adressen", "Jira", "Jira ticket", "Telefon", "Telefonnummer", "Telefonnummern", "Mobilnummer", "Mobilnummern", "Telefongespräch", "Pilotkunden", "Pilotkunde", "Expert", "Experte", "Copilot", "Expat", "Kandidaten", "Kandidat", "Projektmanager", "Bediener", "Bedienerin", "Hotline", "Entwickler", "Entwicklerin", "Amerikaner", "Amerikanerin", "Besteller", "Bestellerin", "Händler", "Benutzername", "Benutzernamen", "Maschinist", "Praktiker",
    "Nachfolger", "Geschäftsführer", "Geschäftsführerin", "Sen", "Global Account Manager", "Trainer", "Trainerin", "Speditionschef", "Prüfer", "Prüferin", "Konstrukteur", "Kontaktperson", "Anforderer","Pilotkunden", "Servicetechniker", "technischen Leiter", "Betriebsrat", "Betriebsrätin", "Auftraggeber", "technische Leiterin", "Brust", "O. K.", "O.K.", "Namensschilder", "Outlook", "Soli", "K.I.", "K. I.", "A. I.", "A.I.", "C. R. M.", "Häkchen", "Haken", "Businesspartner", "Businesspartnerin", "Datenowner", "Werker", "A.D.M.", "ADM", "Kundenbestellung", "Kundenbestellungen", "Business Partnerin", "Business Partner",
    "DCP", "Dcp", "E-Achse", "Nummer",
    "Volkach", "Schweinfurt", "Lohr", "Ulm", "Elchingen",
    # "Hydraulik", "mobile Hydraulik", "Mobil", "Mobilhydraulik",
    "Moog", "Parker", "Hägglunds", "Sytronix", "A4VSO", "IndraDrive",
    # "Itasverbi", "Kunden", "Vertrieb", "Außendienst", "Regionalvertrieb",
    # "DACH-Region", "DCEM", "DCEN", "DCNA", "DCEE", "DC", "DCM", "Schweiz",
    # "Schweiz AG", "Industrie", "Branche", "Deutschland",
    # "Österreich", "Region", "Business Unit", "Gesellschaft für Fluidtechnik",
    # "VDMA",
    # "Firma", "Caterpillar", "John Deere", "Liebherr", "Wacker-Neusohn",
    # "China", "DACH",
    # "MAIL", "E-Mail",
    # Nach erstem Durchlauf: Entity-Liste prüfen und hier ergänzen
    "you", "your", "phone", "customer",
}
WHITELIST_LOWER = {w.lower() for w in WHITELIST}
# ============================================================
# Zusammenfassung
# ============================================================
print(f"Device:            {DEVICE}")
print(f"Modell (Quelle):   {MODEL_LOAD_SOURCE}")

print("\nAktive Label-Gruppen (in Reihenfolge):")
for i, (label, cfg) in enumerate(LABEL_CONFIG.items(), 1):
    pos_labels = [l.strip() for l in label.split(",")]
    neg_labels = cfg.get("negative_labels", [])
    neg_str = f"  negative: {neg_labels}" if neg_labels else ""
    print(f"  Durchlauf {i}: {pos_labels} → {cfg['tag']}  "
          f"(GLiNER-threshold={cfg['threshold']}){neg_str}")

print(f"\nAnzahl Durchläufe:  {len(LABEL_CONFIG)}")
print(f"Aktive Entities:    {ENTITY_TYPES_ACTIVE}")
print(f"Min Entity-Länge:   {MIN_ENTITY_LENGTH} Zeichen")
print(f"Pronomen-Blocklist: {len(PRONOUN_BLOCKLIST)} Einträge")
print(f"Whitelist:          {len(WHITELIST)} Einträge")
print(f"Chunk-Größe:        max {MAX_WORDS_PER_CHUNK} Wörter, "
      f"{OVERLAP_SENTENCES} Sätze Overlap")
print(f"Short-Segment Recall: {USE_SHORT_SEGMENT_RECALL_PASS} "
      f"für {sorted(SHORT_SEGMENT_RECALL_ENTITY_TYPES)} "
      f"(max {SHORT_SEGMENT_RECALL_MAX_CHARS} Zeichen/Segment)")

print(f"\nInput-Verzeichnis:  {LOCAL_INPUT_DIR}")
print(f"Input-Datei/Pfad:   {INPUT_FILE_PATH}")
print(f"Runtime-Modus:      {RUNTIME_MODE}")
print(f"Blob-Schreibzugriff: {'aktiv' if ALLOW_BLOB_WRITES else 'deaktiviert'}")
print(f"Output-Verzeichnis: {PRIMARY_OUTPUT_DIR}")
print(f"Lokales gliner_output: {LOCAL_OUTPUT_DIR}")
print(f"LLM-Adjudication:   {USE_LLM_ENTITY_ADJUDICATION} für {sorted(LLM_ENTITY_TYPES)}")
print(f"PHONE-Hybrid:       Regex-Recall={PHONE_ADD_REGEX_RECALL_PASS}, Syntax={PHONE_REQUIRE_VALID_NUMBER_SYNTAX}, LLM-ambig={PHONE_USE_LLM_FOR_AMBIGUOUS}, Policy={PHONE_AMBIGUOUS_POLICY}")
print(
    f"LLM-Exemption:      Speaker-Roster/Anrede/Speakerlabel; "
    f"Score >= {LLM_EXEMPT_PERSON_MULTI_TOKEN_MIN_SCORE} allein reicht nicht"
)
print(f"Speaker-Roster:     {USE_SPEAKER_ROSTER_EXEMPTION}")
print(f"Externe Whitelist:  {EXTERNAL_ENTITY_WHITELIST_PATH}")
print(f"Whitelist-Scope:    {sorted(EXTERNAL_WHITELIST_ENTITY_TYPES)}")
print(f"Whitelist-Kandidaten: {WHITELIST_CANDIDATES_PATH}")



Aktivierte Entity Types (geordnet): ['PER']
Aktive Inferenz-Durchläufe (LABEL_CONFIG Keys): ['person']
Device:            mps
Modell (Quelle):   urchade/gliner_multi_pii-v1

Aktive Label-Gruppen (in Reihenfolge):
  Durchlauf 1: ['person'] → [PERSON]  (GLiNER-threshold=0.5)

Anzahl Durchläufe:  1
Aktive Entities:    ['PER']
Min Entity-Länge:   3 Zeichen
Pronomen-Blocklist: 71 Einträge
Whitelist:          231 Einträge
Chunk-Größe:        max 250 Wörter, 2 Sätze Overlap
Short-Segment Recall: False für ['LOC', 'ORG', 'PER'] (max 320 Zeichen/Segment)

Input-Verzeichnis:  /Users/olivermaus/projekt/pii-pipeline/notebooks/gliner_input
Input-Datei/Pfad:   
Runtime-Modus:      local
Blob-Schreibzugriff: deaktiviert
Output-Verzeichnis: /Users/olivermaus/projekt/pii-pipeline/notebooks/gliner_output
Lokales gliner_output: /Users/olivermaus/projekt/pii-pipeline/notebooks/gliner_output
LLM-Adjudication:   True für ['PER']
PHONE-Hybrid:       Regex-Recall=True, Syntax=True, LLM-ambig=True, Policy=llm_

## Zelle 3: Text laden

In [6]:
# ============================================================
# Input bestimmen: v7-kompatibel neuester Blob oder optional expliziter Pfad
# ============================================================
# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# from azure.core.exceptions import ResourceNotFoundError


# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# def get_most_recent_blob(container_client):
#     """Wie v7: neuesten Blob im Container anhand last_modified bestimmen."""
#     blobs = list(container_client.list_blobs())
#     if not blobs:
#         raise ValueError(f"No blobs found in container '{container_client.container_name}'.")
#     sorted_blobs = sorted(blobs, key=lambda blob: blob.last_modified, reverse=True)
#     return sorted_blobs[0].name
#
#
# def download_blob(blob_service_client, container_name, blob_name, local_file_path):
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
#     os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
#     with open(local_file_path, "wb") as download_file:
#         download_bytes = blob_client.download_blob().readall()
#         download_file.write(download_bytes)
#     print(f"Downloaded '{blob_name}' to '{local_file_path}'.")
#
#
# def upload_file_to_blob_storage(blob_service_client, container_name, blob_name, local_file_path):
#     """Upload nur im Produktivmodus. Im lokalen Modus bewusst kein Blob-Schreibzugriff."""
#     if not ALLOW_BLOB_WRITES:
#         print(
#             f"Blob-Upload uebersprungen (runtime_mode={RUNTIME_MODE}): "
#             f"{container_name}/{blob_name}"
#         )
#         return False
#
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
#     with open(local_file_path, "rb") as data:
#         blob_client.upload_blob(data, overwrite=True)
#     print(f"Uploaded '{local_file_path}' to '{container_name}/{blob_name}'.")
#     return True
#
#
# def ensure_container_exists(blob_service_client, container_name):
#     """Container-Erstellung nur im Produktivmodus, nie im lokalen Testbetrieb."""
#     if not ALLOW_BLOB_WRITES:
#         print(
#             f"Container-Check fuer '{container_name}' uebersprungen "
#             f"(runtime_mode={RUNTIME_MODE}, keine Blob-Schreibzugriffe)."
#         )
#         return False
#
#     try:
#         blob_service_client.create_container(container_name)
#         print(f"Created container '{container_name}'.")
#         return True
#     except Exception:
#         # Container existiert wahrscheinlich bereits oder darf nicht erstellt werden.
#         return False


# --- COLAB: lokales Pendant zu get_most_recent_blob ---
def get_most_recent_local_file(input_dir):
    """Neueste Datei im lokalen Input-Verzeichnis anhand mtime bestimmen."""
    files = [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if os.path.isfile(os.path.join(input_dir, f))
    ]
    if not files:
        raise ValueError(
            f"Keine Dateien im Input-Verzeichnis '{input_dir}' gefunden. "
            f"Bitte Input-Datei dorthin hochladen oder input_file_path setzen."
        )
    return max(files, key=os.path.getmtime)


def resolve_explicit_input_path(input_file_path, input_dir):
    """
    Optionaler Debug-/Sonderfall:
    - absoluter Pfad: direkt verwenden
    - relativer Pfad / Dateiname: relativ zu LOCAL_INPUT_DIR
    """
    input_file_path = str(input_file_path).strip()

    if os.path.isabs(input_file_path):
        resolved_path = input_file_path
    else:
        resolved_path = os.path.join(input_dir, input_file_path)

    resolved_path = os.path.normpath(resolved_path)

    if not os.path.isfile(resolved_path):
        raise FileNotFoundError(
            f"Explizite Input-Datei nicht gefunden: {resolved_path}. "
            f"Bei leerem input_file_path wird die neueste Datei "
            f"aus dem Verzeichnis '{LOCAL_INPUT_DIR}' verarbeitet."
        )

    return resolved_path


input_file_path_value = str(globals().get("INPUT_FILE_PATH", "")).strip()
input_blob_name = None
INPUT_SOURCE_MODE = None

if input_file_path_value:
    # Optionaler v8-Debug-/Sonderfall: explizite lokale Datei.
    local_input_path = resolve_explicit_input_path(input_file_path_value, LOCAL_INPUT_DIR)
    input_filename = os.path.basename(local_input_path)
    input_blob_name = input_filename
    INPUT_SOURCE_MODE = "explicit_local_path"
    print(f"Input-Datei explizit gesetzt: {local_input_path}")
else:
    # --- COLAB: Default für Input = neueste Datei aus dem lokalen Input-Verzeichnis ---
    local_input_path = get_most_recent_local_file(LOCAL_INPUT_DIR)
    input_filename = os.path.basename(local_input_path)
    INPUT_SOURCE_MODE = "latest_local_file"
    print(f"Input-Datei (neueste in {LOCAL_INPUT_DIR}): {input_filename}")

    # --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
    # # v7-Default fuer Input: neuester Blob aus Container `data` lesen.
    # # Container-Erstellung/-Check ist ein Schreibzugriff und deshalb nur im Produktivmodus aktiv.
    # ensure_container_exists(blob_service_client, LOGS_CONTAINER_NAME)
    # container_client = blob_service_client.get_container_client(DATA_CONTAINER_NAME)
    # input_blob_name = get_most_recent_blob(container_client)
    #
    # print(f"Input Blob (neuester Upload): {input_blob_name}")
    #
    # local_input_path = os.path.join(LOCAL_WORK_DIR, input_blob_name)
    # input_filename = os.path.basename(input_blob_name)
    # INPUT_SOURCE_MODE = "latest_blob"
    #
    # download_blob(blob_service_client, DATA_CONTAINER_NAME, input_blob_name, local_input_path)

input_basename = os.path.splitext(os.path.basename(input_filename))[0]

# Output-Konvention:
#   runtime_mode=local -> PRIMARY_OUTPUT_DIR = LOCAL_OUTPUT_DIR
#   runtime_mode=prod  -> PRIMARY_OUTPUT_DIR = DBFS-Arbeitsverzeichnis, danach Blob-Upload nach data
anonymized_output_filename = f"{input_basename}_RexGuard.txt"
local_output_path = os.path.join(PRIMARY_OUTPUT_DIR, anonymized_output_filename)

# Text laden (UTF-8)
with open(local_input_path, "r", encoding="utf-8", errors="replace") as f:
    raw_text = f.read()

word_count = len(raw_text.split())
page_estimate = word_count / 250

# Colab-Portierung: INPUT_SOURCE_MODE "latest_blob" existiert nicht mehr,
# es wird immer der lokale Dateiname ausgegeben.
print(f"Datei geladen:     {input_filename}")
# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# if INPUT_SOURCE_MODE == "latest_blob":
#     print(f"Datei geladen:     {input_blob_name}")
# else:
#     print(f"Datei geladen:     {input_filename}")
print(f"Wörter:            {word_count:,}")
print(f"Seiten (≈250 W):   {page_estimate:.1f}")
print(f"Aktive Entities:   {ENTITY_TYPES_ACTIVE}")


ValueError: Keine Dateien im Input-Verzeichnis '/Users/olivermaus/projekt/pii-pipeline/notebooks/gliner_input' gefunden. Bitte Input-Datei dorthin hochladen oder input_file_path setzen.

## Zelle 4: Satzbasiertes Chunking
Zerlegt den Text in Chunks mit max. ~250 Wörtern und Satz-Overlap.

In [ ]:
import re

def split_into_sentences(text):
    """
    Einfacher satzbasierter Splitter für deutsche Texte.
    Doppelte Zeilenumbrüche = Absatzgrenze (immer splitten).
    """
    sentences = []
    paragraphs = re.split(r'\n\s*\n', text)

    for para in paragraphs:
        if not para.strip():
            continue
        parts = re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜ"\'])', para.strip())
        for part in parts:
            part = part.strip()
            if part:
                sentences.append(part)

    return sentences


def create_chunks(sentences, max_words=250, overlap=2):
    """
    Gruppiert Sätze zu Chunks mit max. max_words Wörtern.
    An Chunk-Grenzen werden 'overlap' Sätze wiederholt.
    """
    chunks = []
    i = 0

    while i < len(sentences):
        chunk_sentences = []
        word_count = 0
        j = i

        while j < len(sentences):
            sent_words = len(sentences[j].split())
            if word_count + sent_words > max_words and chunk_sentences:
                break
            chunk_sentences.append(sentences[j])
            word_count += sent_words
            j += 1

        chunk_text = " ".join(chunk_sentences)
        chunks.append({
            "text": chunk_text,
            "start_sent": i,
            "end_sent": j - 1,
            "word_count": word_count,
        })

        i = max(j - overlap, i + 1)

    return chunks


sentences = split_into_sentences(raw_text)
chunks = create_chunks(sentences, max_words=MAX_WORDS_PER_CHUNK, overlap=OVERLAP_SENTENCES)

print(f"Sätze gesamt:      {len(sentences)}")
print(f"Chunks erzeugt:    {len(chunks)}")
print(f"Wörter/Chunk:      min={min(c['word_count'] for c in chunks)}, "
      f"max={max(c['word_count'] for c in chunks)}, "
      f"avg={sum(c['word_count'] for c in chunks)/len(chunks):.0f}")
print(f"\nBeispiel Chunk 0 ({chunks[0]['word_count']} Wörter):")
print(chunks[0]["text"][:300] + "...")


## Zelle 5: Modell laden

In [ ]:
import os

# --- DATABRICKS-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# # --- Flash Attention Workaround (Databricks 16.4 ML) ---
# fa_dir = "/databricks/python/lib/python3.12/site-packages/flash_attn"
# fa_disabled = fa_dir + "_disabled"
# if os.path.exists(fa_dir) and not os.path.exists(fa_disabled):
#     try:
#         os.rename(fa_dir, fa_disabled)
#         print("flash_attn deaktiviert (Workaround für Databricks 16.4 ML)")
#     except Exception as e:
#         print(f"flash_attn Workaround fehlgeschlagen: {e}")
#
# dist_dir = "/databricks/python/lib/python3.12/site-packages/flash_attn-2.7.4.post1.dist-info"
# if os.path.exists(dist_dir):
#     try:
#         os.rename(dist_dir, dist_dir + "_disabled")
#     except Exception:
#         pass

from gliner import GLiNER
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Lade GLiNER Modell (local-first): {MODEL_LOAD_SOURCE}")
model = GLiNER.from_pretrained(MODEL_LOAD_SOURCE)
model = model.to(DEVICE)
model.eval()
print(f"Modell geladen auf {DEVICE}.")

# Optional: Wenn von HF geladen wurde und ein lokaler Pfad gewünscht ist,
# kann man hier (falls unterstützt) ein save_pretrained durchführen.
# (Nicht zwingend; hängt vom GLiNER-Package ab.)
if MODEL_LOAD_SOURCE == MODEL_HF_ID and os.path.isdir(os.path.dirname(MODEL_LOCAL_PATH)):
    try:
        if hasattr(model, "save_pretrained"):
            os.makedirs(MODEL_LOCAL_PATH, exist_ok=True)
            model.save_pretrained(MODEL_LOCAL_PATH)
            print(f"Modell lokal gespeichert: {MODEL_LOCAL_PATH}")
    except Exception as e:
        print(f"Hinweis: save_pretrained nicht durchgeführt ({e}).")


## Zelle 6: Inferenz mit Negativ-Labels

Jeder Eintrag in `LABEL_CONFIG` wird als eigener Inferenz-Durchlauf ausgeführt.

**Neu in v6:** Wenn `negative_labels` definiert sind, werden diese zusammen
mit den Positiv-Labels (aus dem Key) an GLiNER übergeben. Nach der Inferenz
werden **nur Treffer der Positiv-Labels behalten**. Die Negativ-Labels
entfalten ihre Wirkung *während* der Inferenz: Sie geben GLiNER die
Möglichkeit, Spans wie "Firma Schreiner" der ORG-Kategorie zuzuordnen,
statt sie mangels Alternative als PERSON zu klassifizieren.

Nach allen Durchläufen werden die Ergebnisse gemergt. Bei Span-Konflikten
(gleicher Text, verschiedene Tags) gewinnt der höchste Score.

In [ ]:
from tqdm import tqdm
from collections import Counter
import re


def run_inference_single_group(chunks, model, labels, threshold, batch_size=8,
                               chunk_id_offset=0, inference_source="chunk"):
    """
    Führt NER-Inferenz über alle Chunks/Segmente für EINE Label-Gruppe aus.

    Args:
        labels: ALLE Labels für diesen Durchlauf (positiv + negativ).
                Die Filterung auf Positiv-Labels erfolgt NACH der Inferenz.
        chunk_id_offset: Offset für Kontext-Lookups, wenn zusätzliche Recall-
                         Segmente hinter die normalen Chunks gehängt werden.
        inference_source: Diagnosekennzeichen, z.B. "chunk" oder
                          "short_segment_recall".
    """
    all_entities = []
    texts = [c["text"] for c in chunks]

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_preds = model.batch_predict_entities(
            batch, labels, flat_ner=True, threshold=threshold
        )
        for chunk_idx, preds in enumerate(batch_preds):
            global_chunk_id = chunk_id_offset + i + chunk_idx
            for ent in preds:
                all_entities.append({
                    "text": ent["text"],
                    "label": ent["label"],
                    "score": ent.get("score", 0),
                    "start": ent["start"],
                    "end": ent["end"],
                    "chunk_id": global_chunk_id,
                    "inference_source": inference_source,
                })
    return all_entities


def filter_positive_labels(entities, positive_labels):
    """
    Filtert Inferenz-Ergebnisse: behält nur Treffer, deren Label
    in den Positiv-Labels enthalten ist.

    Returns:
        kept:     Entities mit Positiv-Label
        rejected: Entities mit Negativ-Label (für Statistik)
    """
    pos_set = {l.lower().strip() for l in positive_labels}
    kept = []
    rejected = []
    for ent in entities:
        if ent["label"].lower().strip() in pos_set:
            kept.append(ent)
        else:
            rejected.append(ent)
    return kept, rejected


def deduplicate_entities(entities):
    """
    Entfernt Duplikate aus Chunk-Overlap.
    Behält pro (text, label)-Paar den höchsten Score.
    """
    best = {}
    for ent in entities:
        key = (ent["text"].strip(), ent["label"])
        if key not in best or ent["score"] > best[key]["score"]:
            best[key] = ent
    return list(best.values())


def merge_multi_pass_results(all_pass_results, label_config):
    """
    Mergt Ergebnisse aus mehreren Inferenz-Durchläufen.

    Bei Span-Konflikten (hier textbasiert, weil spätere Ersetzung ebenfalls
    textbasiert arbeitet) gewinnt der Treffer mit dem höchsten Score.
    """
    all_entities = []
    for config_key, entities in all_pass_results:
        tag = label_config[config_key]["tag"]
        for ent in entities:
            ent_copy = ent.copy()
            ent_copy["config_key"] = config_key
            ent_copy["tag"] = tag
            all_entities.append(ent_copy)

    # Konfliktauflösung: gleicher Text -> höchster Score gewinnt
    best_by_text = {}
    for ent in all_entities:
        text_key = ent["text"].strip()
        if text_key not in best_by_text or ent["score"] > best_by_text[text_key]["score"]:
            best_by_text[text_key] = ent

    return list(best_by_text.values())


def _config_key_to_entity_type(config_key):
    """Liefert PER/ORG/LOC/PHO/MAIL für einen LABEL_CONFIG-Key."""
    for entity_type, keys in ENTITY_TO_CONFIG_KEYS.items():
        if config_key in keys:
            return entity_type
    return None


def _get_recall_threshold(config_key, cfg):
    """
    Schwelle für den Short-Segment Recall.
    Standard: gleiche Schwelle wie der normale LABEL_CONFIG-Lauf.
    Optional: Override per Entity-Type ("PER") oder Config-Key ("person").
    """
    entity_type = _config_key_to_entity_type(config_key)
    overrides = globals().get("SHORT_SEGMENT_RECALL_THRESHOLD_OVERRIDES", {}) or {}
    if config_key in overrides:
        return overrides[config_key]
    if entity_type in overrides:
        return overrides[entity_type]
    return cfg["threshold"]


def _split_long_segment_for_recall(text, max_chars):
    """Splittet lange Segmente wortbasiert, ohne Wörter abzuschneiden."""
    text = re.sub(r"\s+", " ", str(text or "").strip())
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    out = []
    current = []
    current_len = 0
    for word in text.split():
        add_len = len(word) + (1 if current else 0)
        if current and current_len + add_len > max_chars:
            out.append(" ".join(current))
            current = [word]
            current_len = len(word)
        else:
            current.append(word)
            current_len += add_len
    if current:
        out.append(" ".join(current))
    return out


def create_short_segment_recall_chunks(raw_text, max_chars=320, min_chars=8):
    """
    Erzeugt kurze Recall-Segmente aus dem echten raw_text.

    Ziel: GLiNER bekommt zusätzlich lokale Satz-/Sprechersegmente statt nur
    lange Produktions-Chunks. Das hilft bei Fällen wie:
      "Rufe mal den August an ..."
    ohne Regex-Satzmuster für konkrete Namen oder Kontexte einzubauen.
    """
    segments = []
    seen = set()

    # Erst zeilen-/sprecherorientiert splitten, dann innerhalb langer Zeilen
    # nochmal an Satzgrenzen. So bleiben Interview-Speaker-Strukturen erhalten.
    line_parts = re.split(r"\n+", str(raw_text or ""))
    sentence_boundary = re.compile(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ\"'])")

    for line in line_parts:
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue

        sentence_parts = sentence_boundary.split(line)
        for part in sentence_parts:
            part = re.sub(r"\s+", " ", part).strip()
            if not part:
                continue

            for seg in _split_long_segment_for_recall(part, max_chars=max_chars):
                seg = seg.strip()
                if len(seg) < min_chars:
                    continue
                norm = seg.lower()
                if norm in seen:
                    continue
                seen.add(norm)
                segments.append({
                    "text": seg,
                    "start_sent": None,
                    "end_sent": None,
                    "word_count": len(seg.split()),
                    "recall_segment": True,
                })

    return segments


# ============================================================
# Separate Durchläufe ausführen
# ============================================================
print(f"Starte {len(LABEL_CONFIG)} separate Inferenz-Durchläufe:")
print(f"Chunks pro Durchlauf: {len(chunks)}")
print()

all_pass_results = []    # (config_key, deduplicated_entities)
normal_pass_results = [] # nur normale Chunk-Durchläufe, für Summary
total_neg_rejected = 0   # Zähler: durch Negativ-Labels umgelenkte Entities

for pass_num, (config_key, cfg) in enumerate(LABEL_CONFIG.items(), 1):
    # Positiv-Labels aus dem Key
    positive_labels = [l.strip() for l in config_key.split(",")]

    # Negativ-Labels (optional)
    negative_labels = cfg.get("negative_labels", [])

    # Alle Labels für diesen Durchlauf: positiv + negativ
    all_labels = positive_labels + negative_labels

    threshold = cfg["threshold"]
    tag = cfg["tag"]

    # Anzeige
    neg_info = ""
    if negative_labels:
        neg_info = f"  + negative: {negative_labels}"
    print(f"── Durchlauf {pass_num}/{len(LABEL_CONFIG)}: "
          f"{positive_labels} → {tag}  (threshold={threshold}){neg_info}")

    # Inferenz mit ALLEN Labels (positiv + negativ)
    raw_entities = run_inference_single_group(
        chunks, model, all_labels, threshold,
        inference_source="chunk",
    )
    print(f"   Rohe Entities (alle Labels): {len(raw_entities)}")

    # Negativ-Labels herausfiltern
    if negative_labels:
        kept_entities, rejected_entities = filter_positive_labels(
            raw_entities, positive_labels
        )
        neg_count = len(rejected_entities)
        total_neg_rejected += neg_count
        print(f"   Durch Negativ-Labels umgelenkt: {neg_count}")

        # Top-5 der umgelenkten Entities anzeigen (für Debugging)
        if rejected_entities:
            top_rejected = sorted(rejected_entities,
                                  key=lambda e: e["score"], reverse=True)[:5]
            for r in top_rejected:
                print(f"      ✗ '{r['text']}' → {r['label']} "
                      f"(score={r['score']:.3f})")
            if neg_count > 5:
                print(f"      ... und {neg_count - 5} weitere")
    else:
        kept_entities = raw_entities

    print(f"   Positiv-Label Entities: {len(kept_entities)}")

    # Deduplizierung (Chunk-Overlap)
    dedup_entities = deduplicate_entities(kept_entities)
    print(f"   Nach Dedup: {len(dedup_entities)}")

    all_pass_results.append((config_key, dedup_entities))
    normal_pass_results.append((config_key, dedup_entities))


# ============================================================
# Optionaler Short-Segment Recall-Pass
# ============================================================
short_segment_recall_chunks = []
short_segment_recall_stats = []
short_segment_recall_total_entities = 0
short_segment_recall_neg_rejected = 0
INFERENCE_CONTEXT_CHUNKS = chunks

if globals().get("USE_SHORT_SEGMENT_RECALL_PASS", False):
    active_recall_entity_types = set(globals().get("SHORT_SEGMENT_RECALL_ENTITY_TYPES", set())) & set(ENTITY_TYPES_ACTIVE)
    recall_config_items = [
        (ck, cfg) for ck, cfg in LABEL_CONFIG.items()
        if _config_key_to_entity_type(ck) in active_recall_entity_types
    ]

    if recall_config_items:
        short_segment_recall_chunks = create_short_segment_recall_chunks(
            raw_text,
            max_chars=SHORT_SEGMENT_RECALL_MAX_CHARS,
            min_chars=SHORT_SEGMENT_RECALL_MIN_CHARS,
        )
        INFERENCE_CONTEXT_CHUNKS = chunks + short_segment_recall_chunks
        recall_chunk_id_offset = len(chunks)

        print(f"\n{'=' * 50}")
        print("SHORT-SEGMENT RECALL-PASS")
        print(f"{'=' * 50}")
        print(
            f"Recall-Segmente: {len(short_segment_recall_chunks)} "
            f"(max {SHORT_SEGMENT_RECALL_MAX_CHARS} Zeichen, "
            f"min {SHORT_SEGMENT_RECALL_MIN_CHARS})"
        )

        for config_key, cfg in recall_config_items:
            entity_type = _config_key_to_entity_type(config_key)
            positive_labels = [l.strip() for l in config_key.split(",")]
            negative_labels = cfg.get("negative_labels", [])
            all_labels = positive_labels + negative_labels
            threshold = _get_recall_threshold(config_key, cfg)

            print(
                f"── Recall {entity_type}: {positive_labels} → {cfg['tag']} "
                f"(threshold={threshold})"
            )

            raw_recall_entities = run_inference_single_group(
                short_segment_recall_chunks,
                model,
                all_labels,
                threshold,
                batch_size=SHORT_SEGMENT_RECALL_BATCH_SIZE,
                chunk_id_offset=recall_chunk_id_offset,
                inference_source="short_segment_recall",
            )
            print(f"   Rohe Recall-Entities: {len(raw_recall_entities)}")

            if negative_labels:
                kept_recall_entities, rejected_recall_entities = filter_positive_labels(
                    raw_recall_entities,
                    positive_labels,
                )
                neg_count = len(rejected_recall_entities)
                short_segment_recall_neg_rejected += neg_count
                total_neg_rejected += neg_count
                print(f"   Durch Negativ-Labels umgelenkt: {neg_count}")
            else:
                kept_recall_entities = raw_recall_entities

            dedup_recall_entities = deduplicate_entities(kept_recall_entities)
            print(f"   Recall nach Dedup: {len(dedup_recall_entities)}")

            if dedup_recall_entities:
                all_pass_results.append((config_key, dedup_recall_entities))

            short_segment_recall_total_entities += len(dedup_recall_entities)
            short_segment_recall_stats.append({
                "config_key": config_key,
                "entity_type": entity_type,
                "tag": cfg.get("tag", ""),
                "threshold": threshold,
                "segments": len(short_segment_recall_chunks),
                "entities": len(dedup_recall_entities),
            })
    else:
        print("\nShort-Segment Recall: keine aktiven Entity-Typen im Scope.")
else:
    print("\nShort-Segment Recall deaktiviert.")


# ============================================================
# Ergebnisse mergen
# ============================================================
print(f"\n{'=' * 50}")
print("MERGE aller Durchläufe")
print(f"{'=' * 50}")

entities_merged = merge_multi_pass_results(all_pass_results, LABEL_CONFIG)

total_raw = sum(len(ents) for _, ents in all_pass_results)
print(f"Entities vor Merge:  {total_raw}")
print(f"Entities nach Merge: {len(entities_merged)}")
conflicts = total_raw - len(entities_merged)
if conflicts > 0:
    print(f"Span-/Text-Konflikte aufgelöst: {conflicts} "
          f"(höchster Score gewinnt)")
if total_neg_rejected > 0:
    print(f"\nDurch Negativ-Labels umgelenkt (gesamt): {total_neg_rejected}")
if short_segment_recall_stats:
    print("\nShort-Segment Recall Summary:")
    for s in short_segment_recall_stats:
        print(
            f"  {s['entity_type']} {s['tag']}: {s['entities']} Entities "
            f"aus {s['segments']} Segmenten (threshold={s['threshold']})"
        )


## Zelle 7: Post-Processing
Vierstufiger Filter:
1. **Per-Label Threshold** — jede Entitätsklasse hat eigenen Schwellenwert
2. **Mindestlänge** — Entities unter MIN_ENTITY_LENGTH Zeichen werden verworfen
3. **Pronomen-Blocklist** — deutsche Pronomen werden entfernt (exakt UND wortbasiert in zusammengesetzten Entities)
4. **Whitelist** — technische Begriffe werden entfernt (exakt UND wortbasiert in zusammengesetzten Entities)

In [ ]:
# import pandas as pd

# def apply_post_processing(entities, label_config, min_length,
#                           pronoun_blocklist_lower, whitelist_lower):
#     """
#     Wendet alle Post-Processing Filter sequenziell an.

#     Returns:
#         filtered:  Liste der finalen Entities
#         removed:   Dict mit Grund → Liste entfernter Entities
#     """
#     filtered = []
#     removed = {
#         "threshold": [],
#         "min_length": [],
#         "pronoun": [],
#         "whitelist": [],
#     }

#     for ent in entities:
#         text_stripped = ent["text"].strip()
#         text_lower = text_stripped.lower()
#         config_key = ent.get("config_key", "")
#         score = ent["score"]

#         # 1. Per-Label Threshold
#         label_threshold = label_config.get(config_key, {}).get("threshold", 0.5)
#         if score < label_threshold:
#             removed["threshold"].append(ent)
#             continue

#         # 2. Mindestlänge
#         if len(text_stripped) < min_length:
#             removed["min_length"].append(ent)
#             continue

#         # 3. Pronomen-Blocklist (exakt ODER wortbasiert)
#         #    Exakt:       Entity "mein"       → Match auf "mein"
#         #    Wortbasiert: Entity "mein Chef"  → Wort "mein" matcht
#         entity_words_lower = set(text_lower.split())
#         pronoun_hit = entity_words_lower & pronoun_blocklist_lower
#         if pronoun_hit:
#             ent["filter_detail"] = f"Pronomen-Wort: {pronoun_hit}"
#             removed["pronoun"].append(ent)
#             continue

#         # 4. Whitelist (exakt ODER wortbasiert)
#         #    Exakt:       Entity "Mitarbeiter"   → Match auf "Mitarbeiter"
#         #    Wortbasiert: Entity "2 Mitarbeiter" → Wort "mitarbeiter" matcht
#         whitelist_hit = entity_words_lower & whitelist_lower
#         if whitelist_hit:
#             ent["filter_detail"] = f"Whitelist-Wort: {whitelist_hit}"
#             removed["whitelist"].append(ent)
#             continue

#         filtered.append(ent)

#     return filtered, removed


# # Post-Processing anwenden
# entities_filtered, entities_removed = apply_post_processing(
#     entities_merged,
#     LABEL_CONFIG,
#     MIN_ENTITY_LENGTH,
#     PRONOUN_BLOCKLIST_LOWER,
#     WHITELIST_LOWER,
# )

# # Zusammenfassung
# print("=" * 60)
# print("POST-PROCESSING ZUSAMMENFASSUNG")
# print("=" * 60)
# print(f"Entities nach Merge:       {len(entities_merged)}")
# print(f"  Entfernt durch Threshold:    {len(entities_removed['threshold'])}")
# print(f"  Entfernt durch Mindestlänge: {len(entities_removed['min_length'])}")
# print(f"  Entfernt durch Pronomen:     {len(entities_removed['pronoun'])}")
# print(f"  Entfernt durch Whitelist:    {len(entities_removed['whitelist'])}")
# print(f"Entities final:            {len(entities_filtered)}")

# # Details zu entfernten Entities
# for reason, ents in entities_removed.items():
#     if ents:
#         print(f"\n--- Entfernt ({reason}) ---")
#         for e in sorted(ents, key=lambda x: x["score"], reverse=True)[:20]:
#             tag = e.get("tag", "?")
#             detail = e.get('filter_detail', '')
#             detail_str = f'  [{detail}]' if detail else ''
#             print(f"  '{e['text']}' → {tag}  "
#                   f"(score={e['score']:.3f}){detail_str}")
#         if len(ents) > 20:
#             print(f"  ... und {len(ents) - 20} weitere")


In [ ]:
import os
import pandas as pd
import re
import json
import csv
from collections import Counter, OrderedDict
from datetime import datetime


def _normalize_entity_text(s):
    """
    Normalisiert Entity-/Whitelist-Text für robuste Exact-Matches.
    Beispiele:
      "O. K." -> "o.k."
      "O.K."  -> "o.k."
      " O.   K. " -> "o.k."
    """
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)                    # Mehrfachspaces reduzieren
    s = re.sub(r"(?<=\w)\.\s+(?=\w\.)", ".", s)   # "o. k." -> "o.k."
    return s


def _tokenize_entity(text):
    """Tokenisierung für einfache PERSON-Heuristiken."""
    return re.findall(r"[A-Za-zÄÖÜäöüßÀ-ÿ]+(?:[-'][A-Za-zÄÖÜäöüßÀ-ÿ]+)?|\d+", text)


def _looks_like_hard_non_person(text):
    """
    Sehr enge harte Ausschlussregeln für PERSON vor dem LLM.

    Produktive Grundregel:
    - Hart entfernt wird nur noch, was eindeutig technisch/strukturell kein
      Personenname sein kann.
    - Sprachliche, ambige oder namensähnliche Treffer werden NICHT hart entfernt.
      Sie bleiben anonymisiert oder gehen später in die LLM-Adjudication.

    Wichtig: Diese Funktion soll keine generischen deutschen Begriffe wie
    "Kundengespräch", "Automobilisten", "Instandhalter" etc. droppen.
    Solche Fälle gehören ins LLM, nicht in harte Vorfilter.
    """
    stripped = str(text or "").strip()
    if not stripped:
        return True

    lower = stripped.lower()
    compact = re.sub(r"\s+", "", lower)

    # Sehr klare Mail-/URL-/Pfad-Artefakte.
    if re.fullmatch(r"e-?mails?|emails?|mails?", compact):
        return True
    if any(marker in lower for marker in ["http://", "https://", "www."]):
        return True
    if "@" in stripped or "\\" in stripped:
        return True

    def _is_name_like_token(token):
        """Bewusst großzügiges Schutzsignal gegen harte PERSON-Entfernung."""
        token = str(token).strip(" \t\r\n.,;:()[]{}")
        if len(token) < 2:
            return False
        if any(ch.isdigit() for ch in token):
            return False
        if any(sep in token for sep in ["/", "_", "."]):
            return False
        letters_only = re.sub(r"[-'’]", "", token)
        if len(letters_only) >= 2 and letters_only.isupper():
            return False
        # Mindestens ein Kleinbuchstabe schützt u.a. Nachnamen, Vornamen und
        # deutsch großgeschriebene ambige Wörter vor hartem Dropping.
        if not any(ch.islower() for ch in token):
            return False
        return bool(re.fullmatch(r"[A-ZÄÖÜÀ-Ÿ][A-Za-zÄÖÜäöüßÀ-ÿ'’\-]{1,}", token))

    def _starts_with_plausible_name_prefix(value):
        # Schützt Mehrwortnamen auch dann, wenn GLiNER rechts noch technische
        # Zusätze erwischt hat. Beispiel: "Mustermann Max (AbC/DEF3.2)".
        name_token = r"[A-ZÄÖÜÀ-Ÿ][A-Za-zÄÖÜäöüßÀ-ÿ'’\-]{1,}"
        m = re.match(rf"^({name_token})\s+({name_token})\b", value.strip())
        if not m:
            return False
        return _is_name_like_token(m.group(1)) and _is_name_like_token(m.group(2))

    tokens = _tokenize_entity(stripped)

    # Plausibler Mehrwortname am Anfang: niemals hart entfernen.
    if _starts_with_plausible_name_prefix(stripped):
        return False

    # Slash/Underscore/Dot-Codes. Namen wie "Müller/Schmidt" sollen nicht hart
    # entfernt werden, technische Muster wie "AbC/DEF3.2" oder "ABC_DEF" schon.
    if any(sep in stripped for sep in ["/", "_"]):
        segments = [s for s in re.split(r"[/_]+", stripped) if s]
        clean_segments = [s.strip(" .,:;()[]{}") for s in segments]
        if clean_segments and all(_is_name_like_token(s) for s in clean_segments):
            return False
        if re.fullmatch(r"[A-Za-zÄÖÜäöüß0-9.]{1,12}([/_][A-Za-zÄÖÜäöüß0-9.]{1,12})+", stripped):
            return True

    # Bindestrich-Codes nur dann hart entfernen, wenn sie klar codeartig sind.
    # Dadurch bleibt z.B. "Jean-Luc" geschützt, "PJ-DI" aber entfernbar.
    if "-" in stripped:
        hyphen_parts = [p for p in stripped.split("-") if p]
        if hyphen_parts and all(re.fullmatch(r"[A-ZÄÖÜ0-9]{1,5}", p) for p in hyphen_parts):
            return True

    # Reine ALLCAP-Abkürzungen / Org-Codes.
    raw_tokens = stripped.split()
    if len(raw_tokens) >= 2 and all(re.fullmatch(r"[A-ZÄÖÜ]{2,}", t) for t in raw_tokens):
        return True
    if len(raw_tokens) == 1 and re.fullmatch(r"[A-ZÄÖÜ]{3,}", raw_tokens[0]):
        return True

    # Zahlen nur dann hart entfernen, wenn kein namensähnliches Schutzsignal
    # vorhanden ist. Im Zweifel lieber anonymisieren/LLM statt offenlassen.
    if any(ch.isdigit() for ch in stripped):
        if any(_is_name_like_token(token) for token in tokens):
            return False
        return True

    # Sehr symbolische technische Tokens ohne Namen-Signal.
    letters = sum(ch.isalpha() for ch in stripped)
    if letters / max(len(stripped), 1) < 0.45 and not any(_is_name_like_token(token) for token in tokens):
        return True

    return False


# Backward-compatible alias, falls ältere Debug-Zellen den alten Namen nutzen.
def _looks_like_code_or_abbreviation(text):
    return _looks_like_hard_non_person(text)


def _get_chunk_text(ent, chunks):
    """Liefert den Chunk-Text für eine Entity."""
    if not chunks:
        return ""
    chunk_id = ent.get("chunk_id")
    if chunk_id is None or chunk_id < 0 or chunk_id >= len(chunks):
        return ""
    return chunks[chunk_id].get("text", "")


def _get_entity_context(ent, chunks, context_chars):
    """Liefert lokalen Zeichen-Kontext um eine Entity aus dem jeweiligen Chunk."""
    chunk_text = _get_chunk_text(ent, chunks)
    if not chunk_text:
        return ""

    start = max(0, int(ent.get("start", 0)) - context_chars)
    end = min(len(chunk_text), int(ent.get("end", 0)) + context_chars)
    return chunk_text[start:end]


def _get_line_context(ent, chunks, line_radius=1):
    """
    Liefert die aktuelle Zeile plus Nachbarzeilen.
    Für Interviews ist das oft aussagekräftiger als nur +/- Zeichen,
    weil Sprecher-/Antwort-Strukturen zeilenweise organisiert sind.
    """
    chunk_text = _get_chunk_text(ent, chunks)
    if not chunk_text:
        return ""

    start = int(ent.get("start", 0))
    lines = chunk_text.splitlines(keepends=True)
    pos = 0
    current_idx = None
    for idx, line in enumerate(lines):
        next_pos = pos + len(line)
        if pos <= start <= next_pos:
            current_idx = idx
            break
        pos = next_pos

    if current_idx is None:
        return ""

    from_idx = max(0, current_idx - line_radius)
    to_idx = min(len(lines), current_idx + line_radius + 1)
    return "".join(lines[from_idx:to_idx])


def _get_sentence_context(ent, chunks):
    """
    Liefert grob den Satz um die Entity. Das hilft bei Fließtext wie
    "das kam von Anna" oder "ich habe mit Birgit gesprochen".
    """
    chunk_text = _get_chunk_text(ent, chunks)
    if not chunk_text:
        return ""

    start = int(ent.get("start", 0))
    end = int(ent.get("end", start))
    left_candidates = [chunk_text.rfind(sep, 0, start) for sep in [".", "!", "?", "\n"]]
    right_candidates = [chunk_text.find(sep, end) for sep in [".", "!", "?", "\n"]]

    left = max(left_candidates)
    right_existing = [p for p in right_candidates if p != -1]
    right = min(right_existing) if right_existing else len(chunk_text)

    return chunk_text[left + 1:right].strip()


def _has_broad_person_context(ent, chunks):
    """
    Breite, interview-taugliche Kontextdiagnose.

    Wichtig: Diese Funktion entscheidet nicht mehr hart über Behalten/Entfernen.
    Sie liefert nur eine Begründung, wenn ein einwortiger PERSON-Treffer im
    Kontext plausibel ist. Der Recall bleibt dadurch hoch.
    """
    entity = ent["text"].strip()
    if not entity:
        return False

    contexts = [
        _get_entity_context(ent, chunks, PERSON_CONTEXT_CHARS),
        _get_line_context(ent, chunks, line_radius=1),
        _get_sentence_context(ent, chunks),
    ]
    context = "\n".join(c for c in contexts if c).lower()
    if not context:
        return False

    escaped = re.escape(entity)

    # Keyword-Match im erweiterten Kontext.
    if any(re.search(rf"\b{re.escape(keyword)}\b", context) for keyword in PERSON_CONTEXT_KEYWORDS):
        return True

    # Sprecher-/Transkriptmuster: "Anna:", "Anna -", "Anna –".
    if re.search(rf"\b{escaped}\s*[:\-–—]", context, flags=re.IGNORECASE):
        return True

    # Name nach Anrede/Titel.
    if re.search(rf"\b(herr|frau|dr\.?|prof\.?)\s+{escaped}\b", context, flags=re.IGNORECASE):
        return True

    # Typische Präpositionsmuster: "mit Anna", "von Birgit", "an Anne".
    if re.search(rf"\b(mit|von|an|bei|für|laut|durch)\s+{escaped}\b", context, flags=re.IGNORECASE):
        return True

    # Kommunikationsmuster direkt nach dem Namen.
    if re.search(
        rf"\b{escaped}\s+(sagt|sagte|meint|meinte|fragt|fragte|antwortet|antwortete|erzählt|erzählte|erklärt|erklärte)\b",
        context,
        flags=re.IGNORECASE,
    ):
        return True

    return False


# Backward-compatible alias, falls ältere Debug-Zellen den alten Namen nutzen.
def _has_person_context(ent, chunks):
    return _has_broad_person_context(ent, chunks)


def _looks_like_person_token(text):
    """
    Sehr großzügige Formprüfung für ein einzelnes Namens-Token.
    Wird nur für Logging/Begründung verwendet, nicht als harte Pflicht.
    """
    stripped = text.strip()
    if not stripped or any(ch.isdigit() for ch in stripped):
        return False
    if re.fullmatch(r"[A-ZÄÖÜ]{3,}", stripped):
        return False
    return bool(re.fullmatch(r"[A-ZÄÖÜÀ-Ÿ][A-Za-zÄÖÜäöüßÀ-ÿ'-]{2,}", stripped))


def _looks_like_person_name_prefix_token(token):
    """
    Prüft Tokens im Namenspräfix einer GLiNER-PERSON-Spanne.

    Zweck: Wenn GLiNER z.B. "Mustermann Max (AbC/DEF3.2)" als PERSON
    erkennt, darf der Klammer-/Org-Zusatz nicht dazu führen, dass der echte
    Name wegen Ziffern/Sonderzeichen als hartes Non-Person-Muster entfernt
    wird. Der Präfix muss aber klar wie ein Personenname aussehen.
    """
    token = str(token).strip(" \t\r\n.,;:()[]{}")
    if len(token) < 2:
        return False
    if any(ch.isdigit() for ch in token):
        return False
    if re.fullmatch(r"[A-ZÄÖÜ]{2,}", token):
        return False
    if token.lower() in PERSON_PROPAGATION_TOKEN_BLOCKLIST:
        return False
    return bool(re.fullmatch(r"[A-ZÄÖÜÀ-Ÿ][A-Za-zÄÖÜäöüßÀ-ÿ'’-]{1,}", token))


def _trim_person_entity_to_name_prefix(ent):
    """
    Kürzt PERSON-Spannen mit nachgestelltem Klammer-/Org-Zusatz auf den
    eigentlichen Namensanteil.

    Beispiel:
      "Mustermann Max (AbC/DEF3.2)" -> "Mustermann Max"

    Wichtig:
    - Der Name bleibt anonymisiert.
    - Der Org-/Klammerzusatz wird nicht als Namensbestandteil behandelt.
    - Ohne klaren mehrteiligen Namenspräfix wird nichts geändert.
    """
    original = str(ent.get("text", ""))
    stripped = original.strip()
    if not stripped:
        return False

    # Nur bracketartige Zusätze am Ende der PERSON-Spanne entfernen.
    # Das vermeidet Eingriffe in normale Namen und fokussiert den beobachteten
    # Fehlerfall: Name + organisatorischer Klammerzusatz mit Slash/Ziffern/Punkt.
    name_token = r"[A-ZÄÖÜÀ-Ÿ][A-Za-zÄÖÜäöüßÀ-ÿ'’-]{1,}"
    match = re.match(
        rf"^(?P<name>{name_token}(?:\s+{name_token}){{1,}})\s*[\(\[\{{].*$",
        stripped,
    )
    if not match:
        return False

    name_prefix = re.sub(r"\s+", " ", match.group("name").strip())
    tokens = _tokenize_entity(name_prefix)
    if len(tokens) < 2:
        return False
    if not all(_looks_like_person_name_prefix_token(token) for token in tokens):
        return False

    leading_chars = len(original) - len(original.lstrip())
    try:
        original_start = int(ent.get("start", 0))
        ent["start"] = original_start + leading_chars
        ent["end"] = ent["start"] + len(name_prefix)
    except Exception:
        # Start/End sind Diagnose-/Kontextinformationen. Für die spätere
        # Regex-basierte Anonymisierung ist vor allem ent["text"] relevant.
        pass

    ent["text"] = name_prefix
    ent["person_span_trimmed"] = True
    return True


def _passes_person_specific_filter(ent, chunks):
    """
    Recall-orientierter PERSON-Filter nach GLiNER.

    Regeln:
    - Harte Non-Person-Muster raus: Codes, IDs, reine Abkürzungen,
      Mail-/URL-Fragmente, sehr technische Tokens.
    - Mehrwortige PERSON-Treffer ab niedrigem Mindestscore behalten.
    - Einwortige PERSON-Treffer ab niedrigem Mindestscore ebenfalls behalten.
      Kontext wird nur als Diagnose verwendet, nicht mehr als Pflichtbedingung.
    """
    _trim_person_entity_to_name_prefix(ent)

    text = ent["text"].strip()
    score = float(ent.get("score", 0))
    tokens = _tokenize_entity(text)

    if _looks_like_hard_non_person(text):
        ent["filter_detail"] = "PERSON entfernt: hartes Non-Person-Muster"
        return False

    if not tokens:
        ent["filter_detail"] = "PERSON entfernt: keine Namens-Tokens"
        return False

    # Mehrwortige Namen / Namensspannen.
    if len(tokens) >= 2:
        if score >= PERSON_MULTI_TOKEN_MIN_SCORE:
            if ent.get("person_span_trimmed"):
                ent["filter_detail"] = (
                    "PERSON-Mehrwort behalten; "
                    "Klammer-/Org-Zusatz aus PERSON-Spanne entfernt"
                )
            else:
                ent["filter_detail"] = "PERSON-Mehrwort recall-orientiert behalten"
            return True
        ent["filter_detail"] = (
            f"PERSON-Mehrwort unter Score {PERSON_MULTI_TOKEN_MIN_SCORE}"
        )
        return False

    # Einwortige Treffer: bei Anonymisierung im Zweifel behalten.
    token_lower = tokens[0].lower()
    allowlist_lower = {x.lower() for x in PERSON_SINGLE_TOKEN_ALLOWLIST}

    if token_lower in allowlist_lower and score >= PERSON_SINGLE_TOKEN_CONTEXT_MIN_SCORE:
        ent["filter_detail"] = "PERSON-Einwort per Allowlist behalten"
        return True

    if score < PERSON_SINGLE_TOKEN_MIN_SCORE:
        ent["filter_detail"] = (
            f"PERSON-Einwort unter Recall-Minimum "
            f"({score:.3f} < {PERSON_SINGLE_TOKEN_MIN_SCORE})"
        )
        return False

    if _has_broad_person_context(ent, chunks):
        ent["filter_detail"] = "PERSON-Einwort behalten: breiter Personenkontext"
    elif _looks_like_person_token(text):
        ent["filter_detail"] = "PERSON-Einwort behalten: Namensform/Recall"
    else:
        ent["filter_detail"] = "PERSON-Einwort behalten: Recall-Fallback"

    return True


def _strip_non_name_suffixes_for_propagation(text):
    """
    Entfernt Klammer-/Rollen-/Org-Zusätze vor der Namensbestandteil-Ableitung.
    Beispiel: "Schlenkrich Susanne (HiP/PJ -DI)" -> "Schlenkrich Susanne".
    """
    cleaned = re.sub(r"\([^)]*\)|\[[^\]]*\]|\{[^}]*\}", " ", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def _is_propagatable_person_name_token(token, whitelist_lower=None, pronoun_blocklist_lower=None):
    """
    Prüft, ob ein Token aus einem sicheren Mehrwort-PERSON-Treffer als
    einzelner Namensbestandteil in den Text propagiert werden darf.
    Die Regeln sind bewusst recall-orientiert, blocken aber Codes/Titel.
    """
    token = token.strip()
    if len(token) < PERSON_PROPAGATION_MIN_TOKEN_LENGTH:
        return False

    if any(ch.isdigit() for ch in token):
        return False
    if _looks_like_hard_non_person(token):
        return False

    # Name-artige Groß-/Kleinschreibung: Tom, Max, Müller, O'Connor, Anne-Marie.
    if not re.fullmatch(r"[A-ZÄÖÜÀ-Ÿ][A-Za-zÄÖÜäöüßÀ-ÿ'’-]*", token):
        return False

    # Reine Großbuchstaben sind meist Kürzel. "TIM" würde dadurch nicht propagiert.
    if re.fullmatch(r"[A-ZÄÖÜ]{2,}", token):
        return False

    token_lower = token.lower()
    if token_lower in PERSON_PROPAGATION_TOKEN_BLOCKLIST:
        return False
    if whitelist_lower and token_lower in whitelist_lower:
        return False
    if pronoun_blocklist_lower and token_lower in pronoun_blocklist_lower:
        return False

    return True


def _compile_name_token_pattern(token):
    """
    Boundary-sicheres Pattern für einzelne Namensbestandteile.
    Verhindert Ersetzungen innerhalb längerer Wörter, z.B. "Tom" in "Timing".
    """
    letter_class = r"A-Za-zÄÖÜäöüßÀ-ÿ"
    return re.compile(rf"(?<![{letter_class}]){re.escape(token)}(?![{letter_class}])")


def _find_first_token_occurrence(raw_text, token):
    pattern = _compile_name_token_pattern(token)
    match = pattern.search(raw_text)
    if not match:
        return None, None
    return match.start(), match.end()


def propagate_person_name_parts_from_full_names(
    entities,
    raw_text,
    whitelist_lower=None,
    pronoun_blocklist_lower=None,
):
    """
    Ergänzt einzelne Namensbestandteile aus sicheren Mehrwort-PERSON-Treffern.

    Beispiel:
      Erkannter Treffer: "Tom Richter" [PERSON]
      Ergänzt:           "Tom" [PERSON], "Richter" [PERSON]

    Wichtig für die LLM-Adjudication:
    Wenn ein Namensbestandteil bereits direkt von GLiNER erkannt wurde,
    wird diese bestehende Entity NICHT übersprungen, sondern als
    `propagated_name_part=True` markiert. Diese Markierung dient nur noch
    der Diagnose. Propagierte Namensbestandteile werden nicht mehr
    automatisch vom LLM ausgenommen.
    """
    if not PROPAGATE_PERSON_NAME_PARTS:
        return entities, []

    whitelist_lower = whitelist_lower or set()
    pronoun_blocklist_lower = pronoun_blocklist_lower or set()

    result = list(entities)
    propagated = []

    # Nur bestehende PERSON-Entities blockieren eine neue PERSON-Ergänzung.
    # Ein gleichlautender ORG/LOC/etc.-Treffer soll die PERSON-Propagation nicht verhindern.
    existing_person_norm = {
        _normalize_entity_text(e.get("text", ""))
        for e in result
        if e.get("tag") == "[PERSON]"
    }
    propagated_reported = set()

    def _mark_existing_person_name_parts(norm, full_name_text):
        """Markiert bereits vorhandene PERSON-Einzeltreffer als aus Vollnamen bestätigt."""
        marked = []
        for existing in result:
            if existing.get("tag") != "[PERSON]":
                continue
            if _normalize_entity_text(existing.get("text", "")) != norm:
                continue

            existing["propagated_name_part"] = True

            previous_source = str(existing.get("propagated_from_full_name", "")).strip()
            if previous_source:
                sources = [s.strip() for s in previous_source.split(" | ") if s.strip()]
                if full_name_text and full_name_text not in sources:
                    sources.append(full_name_text)
                existing["propagated_from_full_name"] = " | ".join(sources)
            else:
                existing["propagated_from_full_name"] = full_name_text

            existing["filter_detail"] = (
                f"PERSON-Namensbestandteil bestätigt aus: "
                f"{existing.get('propagated_from_full_name', '')}"
            )
            marked.append(existing)

        return marked

    for ent in entities:
        if ent.get("tag") != "[PERSON]":
            continue

        score = float(ent.get("score", 0.0))
        if score < PERSON_PROPAGATION_MIN_FULL_NAME_SCORE:
            continue

        base_text = _strip_non_name_suffixes_for_propagation(ent.get("text", ""))
        tokens = _tokenize_entity(base_text)
        name_tokens = [
            t for t in tokens
            if _is_propagatable_person_name_token(
                t,
                whitelist_lower=whitelist_lower,
                pronoun_blocklist_lower=pronoun_blocklist_lower,
            )
        ]

        # Nur aus echten Mehrwort-Namensspannen ableiten.
        if len(name_tokens) < 2:
            continue

        full_name_text = ent.get("text", "")

        for token in name_tokens:
            norm = _normalize_entity_text(token)

            # Fix: Wenn GLiNER den Einzelnamen bereits direkt erkannt hat,
            # wird er als propagiert/bestätigt markiert statt übersprungen.
            if norm in existing_person_norm:
                marked_existing = _mark_existing_person_name_parts(norm, full_name_text)
                for existing in marked_existing:
                    marker = (
                        norm,
                        existing.get("start"),
                        existing.get("end"),
                        existing.get("chunk_id"),
                    )
                    if marker not in propagated_reported:
                        propagated.append(existing)
                        propagated_reported.add(marker)
                continue

            start, end = _find_first_token_occurrence(raw_text, token)
            if start is None:
                continue

            derived = {
                "text": token,
                "label": "person_name_part",
                "score": score,
                "start": start,
                "end": end,
                "chunk_id": -1,
                "config_key": ent.get("config_key", "person"),
                "tag": "[PERSON]",
                "filter_detail": f"PERSON-Namensbestandteil propagiert aus: {full_name_text}",
                "propagated_name_part": True,
                "propagated_from_full_name": full_name_text,
            }
            result.append(derived)
            propagated.append(derived)
            existing_person_norm.add(norm)

    return result, propagated

# ============================================================
# EXTERNE ENTITY-WHITELIST + LLM-ADJUDICATION NUR FÜR PERSON
# ============================================================
# Hinweis: `entity_types`, ENTITY_MAPPING, ENTITY_TYPES_ACTIVE, LABEL_CONFIG
# und die ORG/LOC/PHONE/MAIL-Logik bleiben unverändert. Die neuen Schritte
# greifen nach Post-Processing und Name-Propagation ein.

TAG_TO_ENTITY_TYPE = {
    "[PERSON]": "PERSON",
    "[ORG]": "ORG",
    "[LOC]": "LOC",
    "[PHONE]": "PHONE",
    "[MAIL]": "MAIL",
}

TAG_TO_ENTITY_CODE = {
    "[PERSON]": "PER",
    "[ORG]": "ORG",
    "[LOC]": "LOC",
    "[PHONE]": "PHO",
    "[MAIL]": "MAIL",
}

ENTITY_TYPE_ALIASES = {
    "PER": "PERSON",
    "PERSON": "PERSON",
    "[PERSON]": "PERSON",
    "ORG": "ORG",
    "ORGANIZATION": "ORG",
    "ORGANISATION": "ORG",
    "[ORG]": "ORG",
    "LOC": "LOC",
    "LOCATION": "LOC",
    "[LOC]": "LOC",
    "PHO": "PHONE",
    "PHONE": "PHONE",
    "PHONE NUMBER": "PHONE",
    "TELEFON": "PHONE",
    "[PHONE]": "PHONE",
    "MAIL": "MAIL",
    "EMAIL": "MAIL",
    "E-MAIL": "MAIL",
    "[MAIL]": "MAIL",
}

LLM_DIAGNOSTIC_FIELDS = [
    "llm_checked",
    "llm_decision",
    "llm_confidence",
    "llm_reason",
    "llm_exempt_reason",
    "external_whitelist_hit",
]


def normalize_entity_text(text):
    """Normalisiert Entity-Text für externe Whitelist und LLM-Dedupe."""
    return re.sub(r"\s+", " ", str(text).strip().lower())


def normalize_entity_type(entity_type):
    """Normalisiert Entity-Type-Angaben aus CSV/Tags auf PERSON/ORG/LOC/PHONE/MAIL."""
    raw = str(entity_type).strip()
    if not raw:
        return ""
    upper = re.sub(r"\s+", " ", raw.upper())
    return ENTITY_TYPE_ALIASES.get(upper, upper)


def tag_to_entity_type(tag):
    """Wandelt Ausgabe-Tags wie [PERSON] in externe Entity-Typen wie PERSON."""
    return TAG_TO_ENTITY_TYPE.get(str(tag).strip(), normalize_entity_type(tag))


def tag_to_entity_code(tag):
    """Wandelt Ausgabe-Tags in die bestehenden internen Entity-Codes PER/ORG/..."""
    return TAG_TO_ENTITY_CODE.get(str(tag).strip(), "")


def _parse_active_flag(value):
    """Robuste Interpretation der optionalen active-Spalte."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return True
    text = str(value).strip().lower()
    if text == "":
        return True
    return text not in {"false", "0", "no", "nein", "n", "inactive", "inaktiv"}



def _entity_type_to_code_for_whitelist(entity_type):
    """Maps external entity type names to internal codes used by EXTERNAL_WHITELIST_ENTITY_TYPES."""
    normalized = normalize_entity_type(entity_type)
    return {
        "PERSON": "PER",
        "ORG": "ORG",
        "LOC": "LOC",
        "PHONE": "PHO",
        "MAIL": "MAIL",
    }.get(normalized, normalized)


def _add_whitelist_name(names, raw_value):
    """Adds a non-empty entity name to an OrderedDict keyed by normalized text."""
    text = str(raw_value).strip()
    normalized = normalize_entity_text(text)
    if normalized:
        names.setdefault(normalized, text)


def _extract_entity_names_from_whitelist_df(df, respect_active=True, source_path=""):
    """
    Extracts entity names from entity_whitelist.csv.

    New DSGVO-minimal format:
        entity_text
        Automobilisten
        Kundengespraech

    Backward compatibility:
    - old multi-column files with entity_text / normalized_text are still readable
    - one-column files without header are accepted as best effort
    - if entity_type is present, only scoped types are imported
    """
    names = OrderedDict()
    if df is None:
        return names

    columns = list(df.columns)
    if not columns:
        return names

    col_by_lower = {str(c).strip().lower(): c for c in columns}
    text_col = (
        col_by_lower.get("entity_text")
        or col_by_lower.get("entity_name")
        or col_by_lower.get("entity")
        or col_by_lower.get("name")
        or col_by_lower.get("normalized_text")
    )
    active_col = col_by_lower.get("active")
    entity_type_col = col_by_lower.get("entity_type")
    allowed_codes = set(globals().get("EXTERNAL_WHITELIST_ENTITY_TYPES", {"PER"}))

    # Support one-column files without a header. pandas treats the first line as column name.
    if text_col is None and len(columns) == 1:
        only_col = columns[0]
        header_candidate = str(only_col).strip()
        known_headers = {"entity_text", "entity_name", "entity", "name", "normalized_text"}
        if header_candidate and header_candidate.lower() not in known_headers:
            _add_whitelist_name(names, header_candidate)
        text_col = only_col

    if text_col is None:
        raise ValueError(
            "entity_whitelist.csv has no usable entity-name column. "
            "Expected one column named entity_text, or a legacy column entity_text/normalized_text."
        )

    for _, row in df.iterrows():
        if respect_active and active_col is not None and not _parse_active_flag(row.get(active_col, "")):
            continue
        if entity_type_col is not None:
            code = _entity_type_to_code_for_whitelist(row.get(entity_type_col, ""))
            if code and code not in allowed_codes:
                continue
        _add_whitelist_name(names, row.get(text_col, ""))

    return names


def load_external_entity_whitelist(path):
    """
    Loads the active external entity whitelist as a set of normalized entity names.
    Missing or malformed whitelist files only produce a warning and do not stop the notebook.
    """
    if not path or not os.path.exists(path):
        print(f"Warnung: Externe Entity-Whitelist nicht gefunden: {path}. Lauf wird ohne externe Whitelist fortgesetzt.")
        return set()

    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
        names = _extract_entity_names_from_whitelist_df(df, respect_active=True, source_path=path)
        print(f"Externe Entity-Whitelist geladen: {len(names)} aktive Entitynamen.")
        return set(names.keys())
    except Exception as exc:
        print(f"Warnung: Externe Entity-Whitelist konnte nicht gelesen werden ({path}): {exc}. Lauf wird ohne externe Whitelist fortgesetzt.")
        return set()


def apply_external_entity_whitelist(entities, external_whitelist_keys):
    """
    Removes scoped entities from entities_filtered using the external active whitelist.

    DSGVO-minimal whitelist format is name-only; therefore matching is now done
    by normalized entity text. Scope protection remains limited to PER via
    EXTERNAL_WHITELIST_ENTITY_TYPES so ORG/LOC/PHONE/MAIL stay unchanged.
    """
    kept = []
    removed = []
    allowed_entity_codes = set(globals().get("EXTERNAL_WHITELIST_ENTITY_TYPES", {"PER"}))

    for ent in entities:
        ent.setdefault("external_whitelist_hit", False)
        entity_code = tag_to_entity_code(ent.get("tag", ""))

        if entity_code not in allowed_entity_codes:
            kept.append(ent)
            continue

        normalized = normalize_entity_text(ent.get("text", ""))

        if normalized in external_whitelist_keys:
            ent["external_whitelist_hit"] = True
            ent["filter_detail"] = "Entfernt durch externe Entity-Whitelist"
            ent["removed_reason"] = "external_whitelist"
            removed.append(ent)
        else:
            kept.append(ent)

    return kept, removed

def initialize_llm_diagnostics(entities):
    """Sorgt dafür, dass finale und entfernte Entities einheitliche Diagnosefelder haben."""
    for ent in entities:
        ent.setdefault("llm_checked", False)
        ent.setdefault("llm_decision", "")
        ent.setdefault("llm_confidence", "")
        ent.setdefault("llm_reason", "")
        ent.setdefault("llm_exempt_reason", "")
        ent.setdefault("external_whitelist_hit", False)


def make_entity_key(ent):
    """Dedupe-/Decision-Key: entity_type + normalized_text."""
    return (tag_to_entity_type(ent.get("tag", "")), normalize_entity_text(ent.get("text", "")))


def _compile_entity_text_pattern(entity_text, ignore_case=False):
    """Boundary-sicheres Pattern für Entity-Kontexte im Originaltext."""
    letter_class = r"A-Za-zÄÖÜäöüßÀ-ÿ"
    flags = re.IGNORECASE if ignore_case else 0
    return re.compile(rf"(?<![{letter_class}]){re.escape(entity_text)}(?![{letter_class}])", flags=flags)


def _find_entity_matches_in_raw_text(raw_text, entity_text):
    """Findet Entity-Vorkommen im Originaltext; erst case-sensitiv, dann fallback case-insensitiv."""
    entity_text = str(entity_text).strip()
    if not raw_text or not entity_text:
        return []

    pattern = _compile_entity_text_pattern(entity_text, ignore_case=False)
    matches = list(pattern.finditer(raw_text))
    if matches:
        return matches

    pattern = _compile_entity_text_pattern(entity_text, ignore_case=True)
    return list(pattern.finditer(raw_text))


def _count_entity_occurrences(raw_text, entity_text):
    return len(_find_entity_matches_in_raw_text(raw_text, entity_text))


def _collect_llm_contexts(entity_text, raw_text, entities=None,
                          left_chars=None, right_chars=None, max_contexts=None):
    """Erzeugt 240/240-Kontexte für das LLM, ohne den vollständigen Interviewtext zu senden."""
    left_chars = LLM_CONTEXT_CHARS_LEFT if left_chars is None else left_chars
    right_chars = LLM_CONTEXT_CHARS_RIGHT if right_chars is None else right_chars
    max_contexts = LLM_MAX_CONTEXTS_PER_CANDIDATE if max_contexts is None else max_contexts

    contexts = []
    seen = set()

    for match in _find_entity_matches_in_raw_text(raw_text, entity_text):
        left = raw_text[max(0, match.start() - left_chars):match.start()]
        matched = raw_text[match.start():match.end()]
        right = raw_text[match.end():min(len(raw_text), match.end() + right_chars)]
        signature = (left[-80:], matched, right[:80])
        if signature in seen:
            continue
        seen.add(signature)
        contexts.append({
            "left_context": left,
            "matched_text": matched,
            "right_context": right,
        })
        if len(contexts) >= max_contexts:
            return contexts

    # Fallback auf Chunk-Kontext, falls der exakte Text im raw_text nicht gefunden wird.
    for ent in entities or []:
        chunk_text = _get_chunk_text(ent, globals().get("chunks", []))
        if not chunk_text:
            continue
        start = int(ent.get("start", 0))
        end = int(ent.get("end", start))
        left = chunk_text[max(0, start - left_chars):start]
        matched = chunk_text[start:end] or str(entity_text)
        right = chunk_text[end:min(len(chunk_text), end + right_chars)]
        signature = (left[-80:], matched, right[:80])
        if signature in seen:
            continue
        seen.add(signature)
        contexts.append({
            "left_context": left,
            "matched_text": matched,
            "right_context": right,
        })
        if len(contexts) >= max_contexts:
            break

    return contexts


def _has_title_context_before_entity(ent, raw_text):
    """Exempt, wenn eine Anrede/ein Titel direkt vor dem Treffer steht."""
    entity_text = ent.get("text", "").strip()
    if not entity_text:
        return False

    title_pattern = re.compile(
        r"(?:^|[\s\n\(\[])(herr|herrn|frau|fr\.?|dr\.?|prof\.?|professor|doktor)\s+$",
        flags=re.IGNORECASE,
    )
    for match in _find_entity_matches_in_raw_text(raw_text, entity_text):
        left = raw_text[max(0, match.start() - 80):match.start()]
        if title_pattern.search(left):
            return True
    return False


def _has_speaker_label_context(ent, raw_text):
    """Exempt, wenn der Treffer als Sprecherlabel genutzt wird, z.B. 'Anna:' oder 'Tom Richert:'."""
    entity_text = ent.get("text", "").strip()
    if not entity_text:
        return False

    line_start_pattern = re.compile(
        rf"(?m)^\s*{re.escape(entity_text)}\s*[:\-–—]",
        flags=re.IGNORECASE,
    )
    if line_start_pattern.search(raw_text):
        return True

    for match in _find_entity_matches_in_raw_text(raw_text, entity_text):
        left = raw_text[max(0, match.start() - 120):match.start()]
        right = raw_text[match.end():match.end() + 10]
        if re.search(r"(^|\n)\s*$", left) and re.match(r"\s*[:\-–—]", right):
            return True
    return False




def _speaker_roster_clean_label(label):
    """
    Bereinigt Sprecherlabels auf den eigentlichen Namensanteil.

    Beispiel:
      "Mustermann Max (AbC/DEF3.2)" -> "Mustermann Max"
    """
    label = str(label or "").strip()
    if not label:
        return ""

    # Zeitstempel am Ende entfernen.
    label = re.sub(r"\s+\d{1,2}:\d{2}(?::\d{2})?\s*$", "", label).strip()

    # Klammer-/Org-Zusätze entfernen.
    label = _strip_non_name_suffixes_for_propagation(label)

    # Sprecher-/Diarization-Artefakte nicht als Namen aufnehmen.
    if re.fullmatch(r"(?i)(sprecher|speaker|interviewer|interviewerin|teilnehmer|teilnehmerin)(\s+\d+)?", label):
        return ""

    label = re.sub(r"\s+", " ", label).strip(" :-–—\t\r\n")
    return label


def _speaker_roster_name_tokens(label):
    """Validiert Sprecherlabel-Tokens konservativ als mehrteiligen Namen."""
    cleaned = _speaker_roster_clean_label(label)
    if not cleaned:
        return []

    tokens = _tokenize_entity(cleaned)
    min_tokens = int(globals().get("SPEAKER_ROSTER_MIN_TOKENS", 2))
    max_tokens = int(globals().get("SPEAKER_ROSTER_MAX_TOKENS", 4))
    if len(tokens) < min_tokens or len(tokens) > max_tokens:
        return []

    if not all(_looks_like_person_name_prefix_token(token) for token in tokens):
        return []

    return tokens


def build_speaker_roster_from_transcript(raw_text):
    """
    Baut eine temporäre Namensliste aus klaren Sprecherzeilen des aktuellen Transkripts.

    Zweck:
    - echter Strukturbeweis für Interview-Sprecher
    - keine persistente Whitelist
    - keine allgemeine Namensliste
    - funktioniert nur, wenn das Dokument Sprecherlabels enthält
    """
    roster_full_names = OrderedDict()
    roster_name_parts = OrderedDict()

    if not globals().get("USE_SPEAKER_ROSTER_EXEMPTION", True):
        return {"full_names": roster_full_names, "name_parts": roster_name_parts}

    for line in str(raw_text or "").splitlines():
        raw_line = line.strip()
        if not raw_line:
            continue

        label_candidates = []

        # Hauptformat aus den Interviewtranskripten:
        # "Mustermann Max (AbC/DEF3.2)"
        timestamp_match = re.match(
            r"^\s*(?P<label>.+?)\s+\d{1,2}:\d{2}(?::\d{2})?\s*$",
            raw_line,
        )
        if timestamp_match:
            label_candidates.append(timestamp_match.group("label"))

        # Alternatives Sprecherformat: "Tom Richert:" oder "Tom Richert -".
        # Nur als Label-Kandidat, wenn der linke Teil nach mehrteiligem Namen aussieht.
        label_match = re.match(r"^\s*(?P<label>[^:\-–—]{2,80})\s*[:\-–—]\s*(?:$|\S)", raw_line)
        if label_match:
            label_candidates.append(label_match.group("label"))

        for candidate in label_candidates:
            tokens = _speaker_roster_name_tokens(candidate)
            if not tokens:
                continue

            full_name = " ".join(tokens)
            full_norm = normalize_entity_text(full_name)
            if not full_norm:
                continue
            roster_full_names.setdefault(full_norm, full_name)

            for token in tokens:
                part_norm = normalize_entity_text(token)
                if part_norm:
                    roster_name_parts.setdefault(part_norm, token)

    return {"full_names": roster_full_names, "name_parts": roster_name_parts}


def _entity_matches_speaker_roster(ent, speaker_roster=None):
    """
    Prüft, ob eine PERSON-Entity durch den temporären Speaker-Roster abgesichert ist.
    Rückgabe: (bool, reason)
    """
    if not globals().get("USE_SPEAKER_ROSTER_EXEMPTION", True):
        return False, ""

    speaker_roster = speaker_roster or globals().get("speaker_roster", {}) or {}
    full_names = speaker_roster.get("full_names", {}) or {}
    name_parts = speaker_roster.get("name_parts", {}) or {}

    raw_text_value = str(ent.get("text", "")).strip()
    if not raw_text_value:
        return False, ""

    # Klammer-/Org-Zusatz ggf. abstreifen, damit "Name (Org)" den Roster trifft.
    comparable = _strip_non_name_suffixes_for_propagation(raw_text_value)
    norm = normalize_entity_text(comparable)
    tokens = _tokenize_entity(comparable)

    if len(tokens) >= 2 and norm in full_names:
        ent["speaker_roster_hit"] = True
        ent["speaker_roster_match"] = full_names[norm]
        return True, "speaker_roster_full_name"

    if len(tokens) == 1 and norm in name_parts:
        ent["speaker_roster_hit"] = True
        ent["speaker_roster_match"] = name_parts[norm]
        return True, "speaker_roster_name_part"

    return False, ""

def should_exempt_person_from_llm(ent, raw_text, speaker_roster=None):
    """
    True bedeutet:
    - nicht ans LLM senden
    - in entities_filtered behalten
    - anonymisieren

    v8.11-Regel:
    Ein hoher Mehrwort-Score reicht nicht mehr allein für Exemption.
    Exempt wird nur bei hartem Strukturbeweis: Speaker-Roster, Anrede/Titel
    oder direktes Sprecherlabel.
    """
    if ent.get("tag") != "[PERSON]":
        return True, "not_person_scope"

    # Propagierte Namensbestandteile bleiben nur Diagnose-Metadaten.
    # Sie werden erst dann LLM-exempt, wenn sie zusätzlich im Speaker-Roster stehen.
    roster_hit, roster_reason = _entity_matches_speaker_roster(ent, speaker_roster=speaker_roster)
    if roster_hit:
        return True, roster_reason

    if _has_title_context_before_entity(ent, raw_text):
        return True, "title_or_salutation_context"

    if _has_speaker_label_context(ent, raw_text):
        return True, "speaker_label_context"

    # Bewusst KEINE Exemption mehr allein wegen:
    # len(tokens) >= 2 and score >= LLM_EXEMPT_PERSON_MULTI_TOKEN_MIN_SCORE
    # Beispiele: "Business Unit Leiter", "Business Unit Leitung", "Data Scientist".

    return False, ""


def build_llm_entity_candidates(entities, raw_text, speaker_roster=None):
    """Bildet deduplizierte PERSON-Kandidaten für die LLM-Adjudication."""
    grouped = OrderedDict()
    exempt_counts = Counter()

    initialize_llm_diagnostics(entities)

    for ent in entities:
        if ent.get("tag") != "[PERSON]":
            ent["llm_checked"] = False
            continue

        if tag_to_entity_code(ent.get("tag")) not in LLM_ENTITY_TYPES:
            ent["llm_checked"] = False
            ent["llm_exempt_reason"] = "entity_type_not_in_llm_scope"
            continue

        exempt, reason = should_exempt_person_from_llm(ent, raw_text, speaker_roster=speaker_roster)
        if exempt:
            ent["llm_checked"] = False
            ent["llm_exempt_reason"] = reason
            exempt_counts[reason] += 1
            continue

        key = make_entity_key(ent)
        grouped.setdefault(key, []).append(ent)

    candidates = []
    for idx, key in enumerate(sorted(grouped.keys(), key=lambda k: (k[0], k[1])), 1):
        ents = grouped[key]
        entity_type, normalized_text = key
        best_ent = max(ents, key=lambda e: float(e.get("score", 0.0)))
        entity_text = best_ent.get("text", "").strip()
        occurrence_count = max(_count_entity_occurrences(raw_text, entity_text), len(ents))
        contexts = _collect_llm_contexts(
            entity_text,
            raw_text,
            entities=ents,
            left_chars=LLM_CONTEXT_CHARS_LEFT,
            right_chars=LLM_CONTEXT_CHARS_RIGHT,
            max_contexts=LLM_MAX_CONTEXTS_PER_CANDIDATE,
        )

        candidates.append({
            "candidate_id": f"PER_{idx:06d}",
            "entity_text": entity_text,
            "normalized_text": normalized_text,
            "entity_type": entity_type,
            "gliner_label": best_ent.get("label", ""),
            "max_score": round(float(best_ent.get("score", 0.0)), 6),
            "occurrence_count": int(occurrence_count),
            "contexts": contexts,
            "_key": key,
        })

    return candidates, exempt_counts


LLM_ENTITY_ADJUDICATION_SYSTEM_PROMPT = """Du bist ein Prüfer für Entity-Erkennung in deutschen Interviewtranskripten.

Aufgabe:
Entscheide für jeden Kandidaten, ob die von GLiNER erkannte Entity wirklich eine schützenswerte personenbezogene Entity des angegebenen Typs ist.

Aktueller Scope:
Nur entity_type = PERSON.

Definitionen:
- ANON: Der Kandidat ist wahrscheinlich ein echter Personenname oder ein personenbezogener Eigenname. Der Treffer soll im Zieldokument anonymisiert werden.
- WHITEL: Der Kandidat ist eindeutig kein Personenname, sondern z. B. ein generischer Begriff, eine Rolle, eine Gruppe, ein Prozess, ein Produkt, ein Tool, eine Tätigkeit, ein normales deutsches Substantiv oder eine Abkürzung. Der Treffer soll nicht anonymisiert werden und kommt in die aktive Entity-Whitelist.
- REVIEW: Der Kandidat ist unklar oder ambig. Wenn es auch nur plausibel ein Personenname sein kann, wähle REVIEW. REVIEW wird im System anonymisiert und als Prüfkandidat dokumentiert.

Wichtige Regeln:
1. Bei Unsicherheit immer REVIEW, niemals WHITEL.
2. Deutsche Nachnamen können wie normale Wörter aussehen, z. B. Schatz, Stadelbauer, Wagner. Solche Fälle nicht leichtfertig als WHITEL markieren.
3. Generische deutsche Substantive wie Kundengespräch, Kundenkontakt, Automobilisten, Instandhalter sollen WHITEL sein, sofern der Kontext nicht klar für einen Personennamen spricht.
4. Du darfst nur auf Basis der übergebenen JSON-Daten entscheiden.
5. Gib ausschließlich valides JSON im vorgegebenen Format zurück.

Antwortformat:
{
  "decisions": [
    {
      "candidate_id": "...",
      "decision": "ANON|WHITEL|REVIEW",
      "confidence": 0.0,
      "reason": "kurze Begründung"
    }
  ]
}
"""


def _candidate_for_llm_payload(candidate):
    """Entfernt interne Felder, bevor Kandidaten an das LLM gesendet werden."""
    return {k: v for k, v in candidate.items() if not k.startswith("_")}


def _validate_llm_decisions_response(parsed, batch_candidates):
    """Validiert LLM-JSON streng. Ungültige Antworten brechen den Notebook-Lauf ab."""
    if not isinstance(parsed, dict):
        raise ValueError("Ungültige LLM-Antwort: Top-Level ist kein JSON-Objekt.")
    if "decisions" not in parsed or not isinstance(parsed["decisions"], list):
        raise ValueError("Ungültige LLM-Antwort: Feld 'decisions' fehlt oder ist keine Liste.")

    candidate_by_id = {c["candidate_id"]: c for c in batch_candidates}
    allowed_decisions = {"ANON", "WHITEL", "REVIEW"}
    decisions_by_id = {}

    for item in parsed["decisions"]:
        if not isinstance(item, dict):
            raise ValueError("Ungültige LLM-Antwort: Decision-Eintrag ist kein Objekt.")
        missing = {"candidate_id", "decision", "confidence", "reason"} - set(item.keys())
        if missing:
            raise ValueError(f"Ungültige LLM-Antwort: Pflichtfelder fehlen: {sorted(missing)}")

        candidate_id = str(item["candidate_id"]).strip()
        if candidate_id not in candidate_by_id:
            raise ValueError(f"Ungültige LLM-Antwort: unbekannte candidate_id '{candidate_id}'.")
        if candidate_id in decisions_by_id:
            raise ValueError(f"Ungültige LLM-Antwort: doppelte candidate_id '{candidate_id}'.")

        decision = str(item["decision"]).strip().upper()
        if decision not in allowed_decisions:
            raise ValueError(f"Ungültige LLM-Antwort: unerwartete Decision '{item['decision']}'.")

        try:
            confidence = float(item["confidence"])
        except Exception as exc:
            raise ValueError(f"Ungültige LLM-Antwort: confidence für {candidate_id} ist nicht numerisch.") from exc
        if not 0.0 <= confidence <= 1.0:
            raise ValueError(f"Ungültige LLM-Antwort: confidence für {candidate_id} liegt außerhalb 0..1.")

        reason = str(item["reason"]).strip()
        if not reason:
            raise ValueError(f"Ungültige LLM-Antwort: reason für {candidate_id} ist leer.")

        candidate = candidate_by_id[candidate_id]
        decisions_by_id[candidate_id] = {
            "candidate_id": candidate_id,
            "candidate_key": candidate["_key"],
            "entity_text": candidate["entity_text"],
            "normalized_text": candidate["normalized_text"],
            "entity_type": candidate["entity_type"],
            "decision": decision,
            "confidence": confidence,
            "reason": reason,
        }

    missing_ids = sorted(set(candidate_by_id) - set(decisions_by_id))
    if missing_ids:
        raise ValueError(f"Ungültige LLM-Antwort: Decisions fehlen für Kandidaten: {missing_ids}")

    return list(decisions_by_id.values())


def call_llm_entity_adjudication(candidates):
    """Sendet deduplizierte PERSON-Kandidaten batchweise an das LLM und gibt Decisions nach Entity-Key zurück."""
    if not candidates:
        return {}

    all_decisions = []
    total_batches = (len(candidates) + LLM_BATCH_SIZE - 1) // LLM_BATCH_SIZE

    for batch_idx in range(0, len(candidates), LLM_BATCH_SIZE):
        batch = candidates[batch_idx:batch_idx + LLM_BATCH_SIZE]
        batch_num = batch_idx // LLM_BATCH_SIZE + 1
        request_payload = {
            "task": "entity_adjudication",
            "entity_type_scope": ["PERSON"],
            "decision_labels": ["ANON", "WHITEL", "REVIEW"],
            "candidates": [_candidate_for_llm_payload(c) for c in batch],
        }

        print(f"LLM-Adjudication Batch {batch_num}/{total_batches}: {len(batch)} Kandidaten")
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": LLM_ENTITY_ADJUDICATION_SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(request_payload, ensure_ascii=False, indent=2)},
            ],
            temperature=0.0,
            max_tokens=4000,
            response_format={"type": "json_object"},
        )

        content = response.choices[0].message.content
        try:
            parsed = json.loads(content)
        except Exception as exc:
            raise ValueError(f"Ungültiges LLM-JSON in Batch {batch_num}: {content[:500]}") from exc

        all_decisions.extend(_validate_llm_decisions_response(parsed, batch))

    decision_by_key = {}
    for decision in all_decisions:
        decision_by_key[decision["candidate_key"]] = decision

    return decision_by_key


def apply_llm_entity_decisions(entities, llm_decision_by_key):
    """WHITEL entfernt Entities; ANON/REVIEW bleiben in entities_filtered und werden anonymisiert."""
    kept = []
    removed = []

    for ent in entities:
        key = make_entity_key(ent)
        decision = llm_decision_by_key.get(key)

        if decision is None:
            ent.setdefault("llm_checked", False)
            kept.append(ent)
            continue

        ent["llm_checked"] = True
        ent["llm_decision"] = decision["decision"]
        ent["llm_confidence"] = decision["confidence"]
        ent["llm_reason"] = decision["reason"]
        ent["llm_exempt_reason"] = ""

        if decision["decision"] == "WHITEL":
            ent["filter_detail"] = "Entfernt durch LLM-WHITEL"
            ent["removed_reason"] = "llm_whitel"
            removed.append(ent)
        else:
            kept.append(ent)

    return kept, removed



ACTIVE_ENTITY_WHITELIST_COLUMNS = ["entity_text"]

WHITELIST_CANDIDATE_COLUMNS = [
    "created_at",
    "source_file",
    "entity_text",
    "normalized_text",
    "entity_type",
    "llm_decision",
    "llm_confidence",
    "llm_reason",
    "occurrence_count",
    "max_score",
    "example_context",
]


def _ensure_csv_header(path, fieldnames):
    """Creates a CSV with header if it is missing or empty."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
        return True
    return False


def _read_existing_csv_for_append(path):
    """Reads an existing CSV for safe append/rewrite operations."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def _column_lookup(df):
    return {str(c).strip().lower(): c for c in df.columns}


def _existing_entity_names_from_df(df):
    """Extracts existing active whitelist names from old or new entity_whitelist.csv formats."""
    return _extract_entity_names_from_whitelist_df(df, respect_active=True)


def _write_active_whitelist_name_list(path, names_by_norm):
    """
    Writes the active entity_whitelist.csv in DSGVO-minimal format:
    exactly one column, entity_text, and no diagnostic metadata.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    values = list(names_by_norm.values())
    pd.DataFrame({"entity_text": values}).to_csv(path, index=False, encoding="utf-8")


def _write_rows_with_schema(path, existing_df, new_rows, fieldnames):
    """Writes existing + new rows with a stable schema."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    new_df = pd.DataFrame(new_rows)

    if existing_df is None or existing_df.empty:
        existing_df = pd.DataFrame(columns=fieldnames)

    columns = list(fieldnames)
    for col in list(existing_df.columns) + list(new_df.columns):
        if col not in columns:
            columns.append(col)

    for col in columns:
        if col not in existing_df.columns:
            existing_df[col] = ""
        if col not in new_df.columns:
            new_df[col] = ""

    frames = []
    if not existing_df.empty:
        frames.append(existing_df[columns])
    if not new_df.empty:
        frames.append(new_df[columns])

    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
    combined.to_csv(path, index=False, encoding="utf-8")


def _build_llm_decision_export_rows(entities, source_file, default_decision=""):
    """Baut deduplizierte Diagnosezeilen aus LLM-annotierten Entities."""
    grouped = OrderedDict()
    for ent in entities or []:
        grouped.setdefault(make_entity_key(ent), []).append(ent)

    created_at = datetime.now().isoformat(timespec="seconds")
    rows = []
    for key, ents in grouped.items():
        entity_type, normalized_text = key
        best_ent = max(ents, key=lambda e: float(e.get("score", 0.0)))
        entity_text = best_ent.get("text", "").strip()
        contexts = _collect_llm_contexts(entity_text, raw_text, entities=ents, max_contexts=1)
        if contexts:
            ctx = contexts[0]
            example_context = f"{ctx.get('left_context', '')}{ctx.get('matched_text', '')}{ctx.get('right_context', '')}"
            example_context = re.sub(r"\s+", " ", example_context).strip()
        else:
            example_context = ""

        rows.append({
            "created_at": created_at,
            "source_file": source_file,
            "entity_text": entity_text,
            "normalized_text": normalized_text,
            "entity_type": entity_type,
            "llm_decision": best_ent.get("llm_decision", default_decision),
            "llm_confidence": best_ent.get("llm_confidence", ""),
            "llm_reason": best_ent.get("llm_reason", ""),
            "occurrence_count": max(_count_entity_occurrences(raw_text, entity_text), len(ents)),
            "max_score": max(float(e.get("score", 0.0)) for e in ents),
            "example_context": example_context,
        })
    return rows



def append_llm_whitels_to_active_whitelist(path, whitel_entities, source_file):
    """
    Writes LLM-WHITELs directly into entity_whitelist.csv.

    DSGVO-minimal policy: entity_whitelist.csv contains only entity names.
    No source file, reason, confidence, timestamp, score, or context is persisted.
    Existing legacy multi-column files are rewritten to the same one-column format.
    """
    whitel_entities = [
        ent for ent in (whitel_entities or [])
        if str(ent.get("llm_decision", "")).upper() == "WHITEL"
    ]

    existing_df = _read_existing_csv_for_append(path)
    existing_names = _existing_entity_names_from_df(existing_df)

    new_names = OrderedDict()
    for ent in whitel_entities:
        text = str(ent.get("text", "")).strip()
        norm = normalize_entity_text(text)
        if norm:
            new_names.setdefault(norm, text)

    added_count = 0
    duplicate_count = 0
    for norm, text in new_names.items():
        if norm in existing_names:
            duplicate_count += 1
            continue
        existing_names[norm] = text
        added_count += 1

    # Always rewrite to enforce the one-column DSGVO-minimal schema.
    _write_active_whitelist_name_list(path, existing_names)

    if not whitel_entities:
        print(f"Keine LLM-WHITELs. Aktive Entity-Whitelist im Ein-Spalten-Format sichergestellt: {path}")
    else:
        print(
            f"Aktive Entity-Whitelist aktualisiert: {path} "
            f"(+{added_count} LLM-WHITELs, {duplicate_count} bereits vorhanden; nur entity_text wird gespeichert)"
        )
    return added_count


def append_whitelist_candidates(path, review_entities, source_file):
    """
    Hängt LLM-REVIEW-Kandidaten an whitelist_candidates.csv an.
    REVIEW bleibt anonymisiert, wird aber für spätere manuelle Prüfung sichtbar.
    """
    review_entities = [
        ent for ent in (review_entities or [])
        if str(ent.get("llm_decision", "")).upper() == "REVIEW"
    ]

    if not review_entities:
        created = _ensure_csv_header(path, WHITELIST_CANDIDATE_COLUMNS)
        suffix = " Header neu angelegt." if created else ""
        print(f"Keine LLM-REVIEWs → whitelist_candidates.csv bleibt unverändert.{suffix}")
        return 0

    existing_df = _read_existing_csv_for_append(path)
    rows = _build_llm_decision_export_rows(review_entities, source_file, default_decision="REVIEW")
    _write_rows_with_schema(path, existing_df, rows, WHITELIST_CANDIDATE_COLUMNS)

    print(f"Whitelist-Kandidaten erweitert: {path} (+{len(rows)} LLM-REVIEW-Kandidaten)")
    return len(rows)




# ============================================================
# PHONE-spezifisches Post-Processing + Regex-Recall
# ============================================================
# Ziel:
# - generische Begriffe wie "Telefon", "Telefonnummer", "Mobilnummer" entfernen
# - klare Telefonnummern deterministisch behalten
# - offensichtliche Nicht-Telefonnummern wie Kreditkarten-/Teile-/Seriennummern entfernen
# - nur ambige Zahlenkandidaten optional per LLM adjudizieren

PHONE_NUMBER_CANDIDATE_RE = re.compile(
    r"""(?ix)
    (?<![\w+])
    (?P<number>
        (?:\+\s*\d{1,3}(?:[\s./()\-]*\d){6,18})
        |
        (?:00\s*\d{1,3}(?:[\s./()\-]*\d){6,18})
        |
        (?:0\d{1,5}(?:[\s./()\-]*\d){5,17})
        |
        # Mehrteilige Zahlenfolgen ohne führende 0/+.
        # Wichtig für Transkripte wie "Rufnummer 4443 5567 2334 9985".
        # Ob es wirklich PHONE ist, entscheidet danach der Kontextfilter.
        (?:\d{2,6}(?:[\s./()\-]+\d{2,6}){1,5})
        |
        # Kurze Durchwahlen / GLiNER-Zahlenkandidaten.
        (?:\d{3,6})
    )
    (?![\w])
    """
)

PHONE_POSITIVE_CONTEXT_RE = re.compile(
    r"""(?ix)
    \b(
        tel\.?|telefon(?:nummern?)?|mobil(?:nummern?)?|handy(?:nummern?)?|
        rufnummern?|durchwahl|dw|extension|phone|mobile|cell|
        erreichbar|anrufen|zurückrufen|rückruf|kontakt(?:nummer)?
    )\b
    """
)

PHONE_EXTENSION_CONTEXT_RE = re.compile(
    r"""(?ix)
    \b(durchwahl|dw|extension|ext\.?|nebenstelle)\b
    """
)

PHONE_NEGATIVE_CONTEXT_RE = re.compile(
    r"""(?ix)
    \b(
        kreditkarten?(?:nummer)?|kartennummer|iban|bic|konto(?:nummer)?|
        material(?:nummer)?|materialnr\.?|mat\.?nr\.?|teilenummer|teile(?:nummer)?|
        artikel(?:nummer)?|art\.?nr\.?|sachnummer|seriennummer|serial(?:\s+number)?|
        bestell(?:nummer)?|auftrags(?:nummer)?|kundennummer|lieferanten(?:nummer)?|
        ticket(?:nummer)?|case(?:\s+id)?|vorgangsnummer|fehlernummer|incident|
        sap|id|identnummer|personalnummer|kostenstelle|projektnummer|plz|postleitzahl
    )\b
    """
)

PHONE_TRIM_CHARS = " \t\r\n.,;:!?)]}>'\""


def _phone_digits(text):
    return re.sub(r"\D", "", str(text or ""))


def _luhn_checksum_is_valid(digits):
    """Erkennt typische Kreditkartennummern als Negativsignal."""
    digits = str(digits or "")
    if not digits.isdigit() or len(digits) < 13:
        return False
    total = 0
    parity = len(digits) % 2
    for idx, ch in enumerate(digits):
        value = int(ch)
        if idx % 2 == parity:
            value *= 2
            if value > 9:
                value -= 9
        total += value
    return total % 10 == 0


def _phone_context_window(raw_text, start, end, chars=None):
    chars = PHONE_CONTEXT_CHARS if chars is None else chars
    raw_text = raw_text or ""
    start = max(0, int(start or 0))
    end = max(start, int(end or start))
    left = raw_text[max(0, start - chars):start]
    right = raw_text[end:min(len(raw_text), end + chars)]
    return left, right, left + right


def _nearest_context_distance(pattern, left, right):
    distances = []
    for match in pattern.finditer(left or ""):
        distances.append(len(left) - match.end())
    for match in pattern.finditer(right or ""):
        distances.append(match.start())
    return min(distances) if distances else None


def _phone_context_signals(raw_text, start, end):
    left, right, window = _phone_context_window(raw_text, start, end)
    pos_dist = _nearest_context_distance(PHONE_POSITIVE_CONTEXT_RE, left, right)
    neg_dist = _nearest_context_distance(PHONE_NEGATIVE_CONTEXT_RE, left, right)
    pos_left_dist = _nearest_context_distance(PHONE_POSITIVE_CONTEXT_RE, left, "")
    neg_left_dist = _nearest_context_distance(PHONE_NEGATIVE_CONTEXT_RE, left, "")
    near_left = left[-60:]
    near_right = right[:60]
    strong_positive = bool(PHONE_POSITIVE_CONTEXT_RE.search(near_left) or PHONE_POSITIVE_CONTEXT_RE.search(near_right))
    strong_negative = bool(PHONE_NEGATIVE_CONTEXT_RE.search(near_left) or PHONE_NEGATIVE_CONTEXT_RE.search(near_right))
    dominant = "none"

    # Ein direkt vorangestelltes Label wie "Rufnummer 123" oder
    # "Kreditkartennummer 123" ist stärker als ein späterer Kontext im
    # rechten Fenster. Das ist entscheidend bei zwei identischen Zahlenfolgen
    # in aufeinanderfolgenden Sätzen.
    if pos_left_dist is not None and pos_left_dist <= 25 and (neg_left_dist is None or pos_left_dist < neg_left_dist):
        dominant = "positive"
    elif neg_left_dist is not None and neg_left_dist <= 25 and (pos_left_dist is None or neg_left_dist < pos_left_dist):
        dominant = "negative"
    elif pos_dist is not None and neg_dist is None:
        dominant = "positive"
    elif neg_dist is not None and pos_dist is None:
        dominant = "negative"
    elif pos_dist is not None and neg_dist is not None:
        if pos_dist + 12 < neg_dist:
            dominant = "positive"
        elif neg_dist + 12 < pos_dist:
            dominant = "negative"
        else:
            dominant = "conflict"
    return {
        "positive": pos_dist is not None,
        "negative": neg_dist is not None,
        "positive_distance": pos_dist,
        "negative_distance": neg_dist,
        "strong_positive": strong_positive,
        "strong_negative": strong_negative,
        "dominant": dominant,
    }


def _has_phone_positive_context(raw_text, start, end):
    return _phone_context_signals(raw_text, start, end)["positive"]


def _has_phone_extension_context(raw_text, start, end):
    left, right, window = _phone_context_window(raw_text, start, end)
    return bool(PHONE_EXTENSION_CONTEXT_RE.search(left[-80:] + right[:80]))


def _has_phone_negative_context(raw_text, start, end):
    return _phone_context_signals(raw_text, start, end)["negative"]


def _looks_like_international_phone(text):
    return bool(re.match(r"^(?:\+\s*\d{1,3}|00\s*\d{1,3})\b", str(text or "").strip()))


def _looks_like_german_mobile(digits):
    return bool(re.match(r"^01[5-7]\d{7,12}$", str(digits or "")))


def _has_phone_separators(text):
    return bool(re.search(r"[\s/().-]", str(text or "").strip()))


def _classify_phone_number_text(number_text, raw_text=None, start=None, end=None, source=""):
    raw_text = raw_text or ""
    number_text = str(number_text or "").strip(PHONE_TRIM_CHARS)
    digits = _phone_digits(number_text)
    digit_count = len(digits)
    if not digits:
        return "DROP", "PHONE entfernt: keine Ziffern"
    start = 0 if start is None else int(start)
    end = start + len(number_text) if end is None else int(end)
    signals = _phone_context_signals(raw_text, start, end) if raw_text else {"positive": False, "negative": False, "dominant": "none"}
    has_positive = bool(signals.get("positive"))
    has_negative = bool(signals.get("negative"))
    dominant_context = signals.get("dominant", "none")
    has_extension = _has_phone_extension_context(raw_text, start, end) if raw_text else False
    is_international = _looks_like_international_phone(number_text)
    has_separators = _has_phone_separators(number_text)
    starts_national_zero = digits.startswith("0")
    starts_mobile = _looks_like_german_mobile(digits)

    if dominant_context == "negative":
        return "DROP", "PHONE entfernt: nächster Kontext ist ein starkes Nicht-Telefon-Signal"
    if digit_count > PHONE_MAX_DIGITS:
        if 13 <= digit_count <= 19 and _luhn_checksum_is_valid(digits) and dominant_context != "positive":
            return "DROP", "PHONE entfernt: kreditkartenähnliche Nummer nach Luhn-Check"
        if dominant_context == "positive" and digit_count <= 19:
            return "KEEP", f"PHONE behalten: lange Zahlenfolge ({digit_count} Ziffern) mit nahem Telefonkontext"
        if dominant_context == "conflict":
            return "AMBIGUOUS", f"PHONE ambig: lange Zahlenfolge ({digit_count} Ziffern) mit widersprüchlichem Kontext"
        return "DROP", f"PHONE entfernt: zu viele Ziffern ({digit_count}) ohne klaren Telefonkontext"
    if 13 <= digit_count <= 19 and _luhn_checksum_is_valid(digits):
        if dominant_context == "positive":
            return "KEEP", "PHONE behalten: kreditkartenähnliche Länge, aber nächster Kontext ist Telefon"
        if dominant_context in {"negative", "none"} and not is_international:
            return "DROP", "PHONE entfernt: kreditkartenähnliche Nummer nach Luhn-Check"
        return "AMBIGUOUS", "PHONE ambig: kreditkartenähnliche Nummer mit uneindeutigem Kontext"
    if PHONE_SHORT_EXTENSION_MIN_DIGITS <= digit_count <= PHONE_SHORT_EXTENSION_MAX_DIGITS:
        if has_extension:
            return "KEEP", "PHONE behalten: kurze Durchwahl mit starkem Telefonkontext"
        if dominant_context == "positive":
            return "AMBIGUOUS", "PHONE ambig: kurze Nummer mit Telefonkontext"
        return "DROP", "PHONE entfernt: kurze Zahl ohne Telefonkontext"
    if digit_count < PHONE_MIN_DIGITS:
        return "DROP", f"PHONE entfernt: zu wenige Ziffern ({digit_count})"
    if is_international:
        if dominant_context == "negative":
            return "AMBIGUOUS", "PHONE ambig: internationale Struktur mit Negativkontext"
        return "KEEP", "PHONE behalten: internationale Telefonnummernstruktur"
    if dominant_context == "positive" or (has_positive and not has_negative):
        return "KEEP", "PHONE behalten: plausible Nummer mit Telefonkontext"
    if starts_mobile:
        return "KEEP", "PHONE behalten: deutsche Mobilnummernstruktur"
    if dominant_context == "conflict":
        return "AMBIGUOUS", "PHONE ambig: widersprüchlicher Telefon-/Negativkontext"
    if starts_national_zero and has_separators:
        return "AMBIGUOUS", "PHONE ambig: nationale Nummernstruktur ohne klaren Kontext"
    if starts_national_zero:
        return "AMBIGUOUS", "PHONE ambig: führende 0 und plausible Länge ohne Kontext"
    if has_separators:
        return "AMBIGUOUS", "PHONE ambig: getrennte Zifferngruppen ohne klaren Kontext"
    if source == "gliner":
        return "AMBIGUOUS", "PHONE ambig: GLiNER-Zahlenkandidat ohne eindeutige Struktur"
    return "DROP", "PHONE entfernt: keine ausreichende Telefonstruktur"


def _extract_phone_number_matches(text):
    text = str(text or "")
    for match in PHONE_NUMBER_CANDIDATE_RE.finditer(text):
        candidate = match.group("number").strip(PHONE_TRIM_CHARS)
        if candidate:
            yield match, candidate


def _phone_source_name(ent):
    src = str(ent.get("inference_source", "") or "").strip()
    if src:
        return src
    if ent.get("chunk_id", None) == -1:
        return "phone_regex_recall"
    return "gliner"


def _phone_entity_has_valid_absolute_span(raw_text, entity_text, start, end):
    try:
        start = int(start); end = int(end)
    except Exception:
        return False
    if start < 0 or end <= start or end > len(raw_text or ""):
        return False
    return (raw_text or "")[start:end] == str(entity_text or "")


def _find_phone_candidate_occurrences(raw_text, candidate):
    raw_text = raw_text or ""
    candidate = str(candidate or "").strip()
    if not candidate:
        return []
    return [(m.start(), m.end()) for m in re.finditer(re.escape(candidate), raw_text)]


def _phone_normalized_text(text):
    digits = _phone_digits(text)
    return digits or re.sub(r"\s+", " ", str(text or "").strip().lower())


def _build_phone_evidence_from_entity(ent, raw_text):
    original = str(ent.get("text", ""))
    evidences = []
    for match, candidate in _extract_phone_number_matches(original):
        rel_start = match.start("number")
        candidate = candidate.strip(PHONE_TRIM_CHARS)
        rel_end = rel_start + len(candidate)
        mapped_spans = []
        try:
            base_start = int(ent.get("start", 0))
            proposed_start = base_start + rel_start
            proposed_end = base_start + rel_end
            if _phone_entity_has_valid_absolute_span(raw_text, candidate, proposed_start, proposed_end):
                mapped_spans.append((proposed_start, proposed_end, False))
        except Exception:
            pass
        if not mapped_spans:
            occ = _find_phone_candidate_occurrences(raw_text, candidate)
            if len(occ) == 1:
                mapped_spans.append((occ[0][0], occ[0][1], False))
            elif 1 < len(occ) <= 20:
                mapped_spans.extend((s, e, True) for s, e in occ)
        for abs_start, abs_end, approximate in mapped_spans:
            evidences.append({
                "text": (raw_text or "")[abs_start:abs_end],
                "start": abs_start,
                "end": abs_end,
                "source": _phone_source_name(ent),
                "score": float(ent.get("score", 0.0) or 0.0),
                "label": ent.get("label", "phone number"),
                "config_key": ent.get("config_key", "phone number"),
                "entity": ent,
                "approximate_source_mapping": approximate,
            })
    return evidences


def _summarize_phone_sources(evidences):
    by_source = OrderedDict()
    for ev in evidences:
        src = ev.get("source", "unknown")
        by_source[src] = max(float(ev.get("score", 0.0) or 0.0), by_source.get(src, 0.0))
    return ", ".join(by_source.keys()), "; ".join(f"{src}:{score:.3f}" for src, score in by_source.items())


def _make_phone_group_entity(group, decision, detail):
    evidences = group["evidences"]
    best_ev = max(evidences, key=lambda ev: float(ev.get("score", 0.0) or 0.0))
    ent = dict(best_ev.get("entity", {}) or {})
    source_names, source_scores = _summarize_phone_sources(evidences)
    ent.update({
        "text": group["text"],
        "label": best_ev.get("label", "phone number"),
        "score": max(float(ev.get("score", 0.0) or 0.0) for ev in evidences),
        "start": group["start"],
        "end": group["end"],
        "chunk_id": -1,
        "inference_source": "phone_resolver",
        "config_key": best_ev.get("config_key", "phone number"),
        "tag": "[PHONE]",
        "filter_detail": detail,
        "phone_validation_decision": decision,
        "phone_resolver_decision": decision,
        "phone_resolver_detail": detail,
        "phone_sources": source_names,
        "phone_source_scores": source_scores,
        "phone_evidence_count": len(evidences),
        "span_specific_replacement": True,
    })
    return ent


def _phone_entity_occurrence_key(ent):
    try:
        return ("PHONE", _phone_normalized_text(ent.get("text", "")), int(ent.get("start")), int(ent.get("end")))
    except Exception:
        return make_entity_key(ent)


def resolve_phone_candidates(phone_entities, raw_text):
    kept, removed = [], []
    groups = OrderedDict()
    for ent in phone_entities:
        evidences = _build_phone_evidence_from_entity(ent, raw_text)
        if not evidences:
            ent_removed = dict(ent)
            ent_removed["filter_detail"] = "PHONE entfernt: keine konkrete Nummer im Treffer"
            ent_removed["phone_resolver_decision"] = "DROP"
            ent_removed["phone_sources"] = _phone_source_name(ent)
            removed.append(ent_removed)
            continue
        for ev in evidences:
            key = (int(ev["start"]), int(ev["end"]), _phone_normalized_text(ev["text"]))
            groups.setdefault(key, {"start": int(ev["start"]), "end": int(ev["end"]), "text": ev["text"], "evidences": []})
            groups[key]["evidences"].append(ev)
    for group in groups.values():
        decision, detail = _classify_phone_number_text(group["text"], raw_text=raw_text, start=group["start"], end=group["end"], source="phone_resolver")
        final_ent = _make_phone_group_entity(group, decision, detail)
        if decision == "KEEP":
            final_ent["phone_needs_llm"] = False
            kept.append(final_ent)
        elif decision == "DROP":
            final_ent["phone_needs_llm"] = False
            final_ent["removed_reason"] = "phone_specific"
            removed.append(final_ent)
        elif globals().get("PHONE_USE_LLM_FOR_AMBIGUOUS", True):
            final_ent["phone_needs_llm"] = True
            final_ent["llm_exempt_reason"] = "phone_ambiguous_pending_llm"
            kept.append(final_ent)
        else:
            policy = str(globals().get("PHONE_AMBIGUOUS_POLICY", "llm_or_review")).strip().lower()
            if policy in {"llm_or_keep", "llm_or_review"}:
                final_ent["phone_needs_llm"] = False
                final_ent["llm_checked"] = False
                final_ent["llm_decision"] = "REVIEW" if policy == "llm_or_review" else "ANON"
                final_ent["llm_reason"] = "PHONE ambig ohne LLM; gemäß Policy behalten"
                kept.append(final_ent)
            else:
                final_ent["phone_needs_llm"] = False
                final_ent["filter_detail"] = detail + " → PHONE entfernt gemäß PHONE_AMBIGUOUS_POLICY"
                final_ent["removed_reason"] = "phone_specific"
                removed.append(final_ent)
    return kept, removed


def _trim_phone_entity_to_number(ent, raw_text=None):
    evidences = _build_phone_evidence_from_entity(ent, raw_text or "")
    if not evidences:
        return "DROP", "PHONE entfernt: keine konkrete Nummer im Treffer"
    ev = evidences[0]
    ent["text"], ent["start"], ent["end"] = ev["text"], ev["start"], ev["end"]
    return _classify_phone_number_text(ev["text"], raw_text=raw_text or "", start=ev["start"], end=ev["end"], source="gliner")


def _passes_phone_specific_filter(ent, raw_text=None):
    if not globals().get("PHONE_REQUIRE_VALID_NUMBER_SYNTAX", True):
        ent["filter_detail"] = "PHONE behalten: Syntaxprüfung deaktiviert"
        return True
    decision, detail = _trim_phone_entity_to_number(ent, raw_text=raw_text)
    ent["filter_detail"] = detail
    ent["phone_validation_decision"] = decision
    ent["phone_needs_llm"] = decision == "AMBIGUOUS" and globals().get("PHONE_USE_LLM_FOR_AMBIGUOUS", True)
    return decision in {"KEEP", "AMBIGUOUS"}


def extract_phone_entities_from_raw_text(raw_text):
    if not globals().get("PHONE_ADD_REGEX_RECALL_PASS", True):
        return []
    if "PHO" not in globals().get("ENTITY_TYPES_ACTIVE", []):
        return []
    entities = []
    seen_spans = set()
    for match, candidate in _extract_phone_number_matches(raw_text):
        start = match.start("number")
        end = start + len(candidate)
        decision, detail = _classify_phone_number_text(candidate, raw_text=raw_text, start=start, end=end, source="regex_recall")
        if decision == "DROP":
            continue
        if (start, end) in seen_spans:
            continue
        seen_spans.add((start, end))
        entities.append({
            "text": candidate,
            "label": "phone number",
            "score": 0.999,
            "start": start,
            "end": end,
            "chunk_id": -1,
            "inference_source": "phone_regex_recall",
            "config_key": "phone number",
            "tag": "[PHONE]",
            "filter_detail": detail,
            "phone_validation_decision": decision,
            "phone_needs_llm": decision == "AMBIGUOUS" and globals().get("PHONE_USE_LLM_FOR_AMBIGUOUS", True),
            "span_specific_replacement": True,
        })
    return entities


def merge_phone_regex_recall_entities(existing_entities, regex_entities):
    existing_entities = list(existing_entities or [])
    regex_entities = list(regex_entities or [])
    seen_spans = {
        (str(ent.get("tag", "")), int(ent.get("start", -1)), int(ent.get("end", -1)))
        for ent in existing_entities
        if ent.get("tag") == "[PHONE]" and ent.get("start") is not None and ent.get("end") is not None
    }
    added = []
    for ent in regex_entities:
        key = (str(ent.get("tag", "")), int(ent.get("start", -1)), int(ent.get("end", -1)))
        if key in seen_spans:
            continue
        seen_spans.add(key)
        added.append(ent)
    return existing_entities + added, added


PHONE_LLM_ADJUDICATION_SYSTEM_PROMPT = """Du bist ein Prüfer für Telefonnummer-Erkennung in deutschen Interviewtranskripten.

Aufgabe:
Entscheide für jeden Kandidaten, ob der Kandidat wirklich eine Telefonnummer oder Durchwahl ist, die anonymisiert werden soll.

Scope:
Nur entity_type = PHONE.

Definitionen:
- ANON: Der Kandidat ist wahrscheinlich eine Telefonnummer, Mobilnummer, Rufnummer oder Durchwahl. Er soll anonymisiert werden.
- WHITEL: Der Kandidat ist eindeutig keine Telefonnummer, sondern z. B. Kreditkartennummer, IBAN, Materialnummer, Teilenummer, Seriennummer, Artikelnummer, Ticketnummer, Kundennummer, Bestellnummer, ID, Datum, Uhrzeit, Menge, Preis oder sonstige technische/geschäftliche Nummer. Er soll nicht anonymisiert werden.
- REVIEW: Der Kandidat ist unklar oder ambig. Wenn es plausibel eine Telefonnummer oder Durchwahl sein kann, wähle REVIEW. REVIEW wird im System anonymisiert und diagnostisch dokumentiert.

Wichtige Regeln:
1. Bei Datenschutz-Unsicherheit REVIEW, nicht WHITEL.
2. Begriffe ohne konkrete Nummer sind keine PHONE-Entity.
3. Berücksichtige den übergebenen linken und rechten Kontext.
4. Gib ausschließlich valides JSON im vorgegebenen Format zurück.

Antwortformat:
{
  "decisions": [
    {
      "candidate_id": "...",
      "decision": "ANON|WHITEL|REVIEW",
      "confidence": 0.0,
      "reason": "kurze Begründung"
    }
  ]
}
"""


def _collect_phone_occurrence_contexts(ent, raw_text):
    try:
        start = int(ent.get("start")); end = int(ent.get("end"))
    except Exception:
        return _collect_llm_contexts(ent.get("text", ""), raw_text, entities=[ent], left_chars=LLM_CONTEXT_CHARS_LEFT, right_chars=LLM_CONTEXT_CHARS_RIGHT, max_contexts=1)
    return [{
        "left": (raw_text or "")[max(0, start - LLM_CONTEXT_CHARS_LEFT):start],
        "entity": (raw_text or "")[start:end],
        "right": (raw_text or "")[end:min(len(raw_text or ""), end + LLM_CONTEXT_CHARS_RIGHT)],
        "start": start,
        "end": end,
    }]


def build_llm_phone_candidates(entities, raw_text):
    grouped = OrderedDict()
    for ent in entities:
        if ent.get("tag") != "[PHONE]" or not ent.get("phone_needs_llm"):
            continue
        grouped.setdefault(_phone_entity_occurrence_key(ent), []).append(ent)
    candidates = []
    for idx, key in enumerate(sorted(grouped.keys(), key=lambda k: tuple(str(x) for x in k)), 1):
        ents = grouped[key]
        best_ent = max(ents, key=lambda e: float(e.get("score", 0.0)))
        entity_text = best_ent.get("text", "").strip()
        candidates.append({
            "candidate_id": f"PHO_{idx:06d}",
            "entity_text": entity_text,
            "normalized_text": _phone_normalized_text(entity_text),
            "entity_type": "PHONE",
            "gliner_label": best_ent.get("label", ""),
            "max_score": round(float(best_ent.get("score", 0.0)), 6),
            "occurrence_count": 1,
            "start": best_ent.get("start", ""),
            "end": best_ent.get("end", ""),
            "phone_sources": best_ent.get("phone_sources", best_ent.get("inference_source", "")),
            "phone_source_scores": best_ent.get("phone_source_scores", ""),
            "phone_validation_detail": best_ent.get("filter_detail", ""),
            "contexts": _collect_phone_occurrence_contexts(best_ent, raw_text),
            "_key": key,
        })
    return candidates


def call_llm_phone_adjudication(candidates):
    """Sendet ambige PHONE-Kandidaten batchweise ans LLM."""
    if not candidates:
        return {}

    all_decisions = []
    batch_size = int(globals().get("PHONE_LLM_BATCH_SIZE", LLM_BATCH_SIZE))
    total_batches = (len(candidates) + batch_size - 1) // batch_size

    for batch_idx in range(0, len(candidates), batch_size):
        batch = candidates[batch_idx:batch_idx + batch_size]
        batch_num = batch_idx // batch_size + 1
        request_payload = {
            "task": "phone_adjudication",
            "entity_type_scope": ["PHONE"],
            "decision_labels": ["ANON", "WHITEL", "REVIEW"],
            "candidates": [_candidate_for_llm_payload(c) for c in batch],
        }

        print(f"PHONE-LLM-Adjudication Batch {batch_num}/{total_batches}: {len(batch)} Kandidaten")
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": PHONE_LLM_ADJUDICATION_SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(request_payload, ensure_ascii=False, indent=2)},
            ],
            temperature=0.0,
            max_tokens=4000,
            response_format={"type": "json_object"},
        )

        content = response.choices[0].message.content
        try:
            parsed = json.loads(content)
        except Exception as exc:
            raise ValueError(f"Ungültiges PHONE-LLM-JSON in Batch {batch_num}: {content[:500]}") from exc

        all_decisions.extend(_validate_llm_decisions_response(parsed, batch))

    decision_by_key = {}
    for decision in all_decisions:
        decision_by_key[decision["candidate_key"]] = decision
    return decision_by_key


def apply_phone_llm_decisions(entities, phone_llm_decision_by_key):
    kept = []
    removed = []
    policy = str(globals().get("PHONE_AMBIGUOUS_POLICY", "llm_or_review")).strip().lower()
    for ent in entities:
        if ent.get("tag") != "[PHONE]" or not ent.get("phone_needs_llm"):
            kept.append(ent)
            continue
        key = _phone_entity_occurrence_key(ent)
        decision = phone_llm_decision_by_key.get(key)
        if decision is None:
            if policy == "llm_or_drop":
                ent["filter_detail"] = "PHONE entfernt: ambig und keine LLM-Decision"
                ent["removed_reason"] = "phone_llm_whitel"
                removed.append(ent)
                continue
            ent["llm_checked"] = False
            ent["llm_decision"] = "REVIEW" if policy == "llm_or_review" else "ANON"
            ent["llm_reason"] = "PHONE ambig ohne LLM-Decision; gemäß Policy behalten"
            ent["phone_needs_llm"] = False
            kept.append(ent)
            continue
        ent["llm_checked"] = True
        ent["llm_decision"] = decision["decision"]
        ent["llm_confidence"] = decision["confidence"]
        ent["llm_reason"] = decision["reason"]
        ent["llm_exempt_reason"] = ""
        ent["phone_needs_llm"] = False
        if decision["decision"] == "WHITEL":
            ent["filter_detail"] = "PHONE entfernt durch LLM-WHITEL"
            ent["removed_reason"] = "phone_llm_whitel"
            removed.append(ent)
        else:
            kept.append(ent)
    return kept, removed

# ============================================================
# MAIL-spezifisches Post-Processing
# ============================================================
# Anders als PERSON ist MAIL präzisionsorientiert prüfbar: Eine echte
# E-Mail-Adresse muss eine valide localpart@domain.tld-Struktur haben.
# GLiNER-Begriffe wie "E-Mail-Prozess" oder "E-Mail-Vorlagen" werden
# dadurch verworfen, ohne die PERSON-Pipeline zu verändern.
EMAIL_ADDRESS_RE = re.compile(
    r"(?ix)"
    r"(?<![\w.+\-'])"
    r"([A-Z0-9._%+\-']{1,64}@"
    r"(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+"
    r"[A-Z]{2,63})"
    r"(?![A-Z0-9_-])"
)


def _trim_mail_entity_to_address(ent):
    """
    Schneidet bei MAIL-Treffern eine eingebettete echte E-Mail-Adresse aus.

    Beispiele:
    - "max.mustermann@bosch.com" bleibt unverändert.
    - "E-Mail: max.mustermann@bosch.com." wird auf die Adresse gekürzt.
    - "E-Mail-Prozess" liefert False und wird später entfernt.
    """
    original = str(ent.get("text", ""))
    match = EMAIL_ADDRESS_RE.search(original)
    if not match:
        return False

    email = match.group(1)
    original_stripped = original.strip()

    if email != original_stripped:
        try:
            original_start = int(ent.get("start", 0))
            ent["start"] = original_start + match.start(1)
            ent["end"] = original_start + match.end(1)
        except Exception:
            # Start/End sind Diagnose-/Kontextinformationen. Für die spätere
            # Regex-basierte Anonymisierung ist vor allem ent["text"] relevant.
            pass

        ent["text"] = email
        ent["mail_span_trimmed"] = True

    return True


def _passes_mail_specific_filter(ent):
    """
    Präzisionsorientierter MAIL-Filter nach GLiNER.

    Behalten werden nur echte E-Mail-Adressen mit valider Struktur.
    Semantische E-Mail-Begriffe ohne Adresse werden entfernt.
    """
    if not globals().get("MAIL_REQUIRE_VALID_ADDRESS_SYNTAX", True):
        ent["filter_detail"] = "MAIL behalten: Syntaxprüfung deaktiviert"
        return True

    if _trim_mail_entity_to_address(ent):
        if ent.get("mail_span_trimmed"):
            ent["filter_detail"] = "MAIL behalten: valide E-Mail-Adresse aus Span extrahiert"
        else:
            ent["filter_detail"] = "MAIL behalten: valide E-Mail-Adresse"
        return True

    ent["filter_detail"] = "MAIL entfernt: keine valide E-Mail-Adresse"
    return False

def apply_post_processing(entities, label_config, min_length,
                          pronoun_blocklist_lower, whitelist_lower,
                          chunks=None, raw_text=None):
    """
    Wendet Post-Processing an. PHONE wird ab v8.17 gesammelt und danach
    vom PHONE-Resolver vorkommensgenau final entschieden.
    """
    filtered = []
    phone_candidates = []
    removed = {
        "threshold": [],
        "min_length": [],
        "pronoun": [],
        "whitelist": [],
        "phone_specific": [],
        "mail_specific": [],
        "person_specific": [],
    }
    whitelist_normalized = {_normalize_entity_text(w) for w in whitelist_lower}
    for ent in entities:
        text_stripped = ent["text"].strip()
        text_lower = text_stripped.lower()
        text_normalized = _normalize_entity_text(text_stripped)
        config_key = ent.get("config_key", "")
        score = ent["score"]
        tag = ent.get("tag", "")
        label_threshold = label_config.get(config_key, {}).get("threshold", 0.5)
        if score < label_threshold:
            removed["threshold"].append(ent)
            continue
        if len(text_stripped) < min_length:
            removed["min_length"].append(ent)
            continue
        entity_words_lower = set(text_lower.split())
        pronoun_hit = entity_words_lower & pronoun_blocklist_lower
        if pronoun_hit:
            ent["filter_detail"] = f"Pronomen-Wort: {pronoun_hit}"
            removed["pronoun"].append(ent)
            continue
        if tag == "[PHONE]":
            phone_candidates.append(ent)
            continue
        if text_normalized in whitelist_normalized:
            ent["filter_detail"] = f"Whitelist-Exact-Norm: {text_normalized}"
            removed["whitelist"].append(ent)
            continue
        whitelist_hit = entity_words_lower & whitelist_lower
        if whitelist_hit:
            ent["filter_detail"] = f"Whitelist-Wort: {whitelist_hit}"
            removed["whitelist"].append(ent)
            continue
        if tag == "[MAIL]" and not _passes_mail_specific_filter(ent):
            removed["mail_specific"].append(ent)
            continue
        if tag == "[PERSON]" and not _passes_person_specific_filter(ent, chunks):
            removed["person_specific"].append(ent)
            continue
        filtered.append(ent)
    phone_kept, phone_removed = resolve_phone_candidates(phone_candidates, raw_text or "")
    filtered.extend(phone_kept)
    removed["phone_specific"].extend(phone_removed)
    return filtered, removed


# PHONE-Regex-Recall vor dem Post-Processing ergänzen.
# Die Treffer laufen anschließend durch denselben PHONE-Filter und ggf. PHONE-LLM.
phone_regex_recall_entities = []
phone_regex_recall_added = []
if PHONE_ADD_REGEX_RECALL_PASS and "PHO" in ENTITY_TYPES_ACTIVE:
    phone_regex_recall_entities = extract_phone_entities_from_raw_text(raw_text)
    entities_merged, phone_regex_recall_added = merge_phone_regex_recall_entities(
        entities_merged,
        phone_regex_recall_entities,
    )
    print(
        f"PHONE-Regex-Recall: {len(phone_regex_recall_added)} neue Kandidaten "
        f"aus {len(phone_regex_recall_entities)} Regex-Treffern ergänzt."
    )
else:
    print("PHONE-Regex-Recall: deaktiviert oder PHO nicht aktiv.")
phone_regex_recall_total_entities = len(phone_regex_recall_added)

# Temporären Speaker-Roster aus klaren Sprecherzeilen des aktuellen Transkripts bauen.
# Diese Liste wird nicht gespeichert und gilt nur für den aktuellen Lauf.
speaker_roster = build_speaker_roster_from_transcript(raw_text)
speaker_roster_full_names = speaker_roster.get("full_names", {})
speaker_roster_name_parts = speaker_roster.get("name_parts", {})
print(
    f"Speaker-Roster: {len(speaker_roster_full_names)} Vollnamen, "
    f"{len(speaker_roster_name_parts)} Namensbestandteile"
)
if speaker_roster_full_names:
    preview = list(speaker_roster_full_names.values())[:10]
    print("Speaker-Roster Vorschau:", preview)

# Post-Processing anwenden
entities_filtered, entities_removed = apply_post_processing(
    entities_merged,
    LABEL_CONFIG,
    MIN_ENTITY_LENGTH,
    PRONOUN_BLOCKLIST_LOWER,
    WHITELIST_LOWER,
    chunks=globals().get("INFERENCE_CONTEXT_CHUNKS", chunks),
    raw_text=raw_text,
)

entities_after_postprocessing = list(entities_filtered)

# Namensbestandteile aus sicheren Mehrwort-PERSON-Treffern ergänzen.
# Beispiel: "Tom Richter" -> zusätzlich "Tom" und "Richter".
entities_filtered, propagated_name_parts = propagate_person_name_parts_from_full_names(
    entities_filtered,
    raw_text,
    whitelist_lower=WHITELIST_LOWER,
    pronoun_blocklist_lower=PRONOUN_BLOCKLIST_LOWER,
)

entities_after_name_propagation = list(entities_filtered)
initialize_llm_diagnostics(entities_filtered)

# Externe aktive Whitelist nach Post-Processing und Name-Propagation anwenden.
external_whitelist_keys = load_external_entity_whitelist(EXTERNAL_ENTITY_WHITELIST_PATH)
entities_filtered, removed_by_external_whitelist = apply_external_entity_whitelist(
    entities_filtered,
    external_whitelist_keys,
)
entities_removed["external_whitelist"] = removed_by_external_whitelist

# PHONE-LLM-Adjudication nur für ambige PHONE-Kandidaten.
phone_llm_candidates = []
phone_llm_decision_by_key = {}
phone_llm_removed = []
phone_llm_decision_counts = Counter()

if PHONE_USE_LLM_FOR_AMBIGUOUS and "PHO" in ENTITY_TYPES_ACTIVE:
    phone_llm_candidates = build_llm_phone_candidates(entities_filtered, raw_text)
    print(f"PHONE-LLM-Adjudication: {len(phone_llm_candidates)} ambige PHONE-Kandidaten.")
    if phone_llm_candidates:
        phone_llm_decision_by_key = call_llm_phone_adjudication(phone_llm_candidates)
        phone_llm_decision_counts = Counter(d["decision"] for d in phone_llm_decision_by_key.values())
        entities_filtered, phone_llm_removed = apply_phone_llm_decisions(
            entities_filtered,
            phone_llm_decision_by_key,
        )
        entities_removed["phone_llm_whitel"] = phone_llm_removed
    else:
        entities_removed["phone_llm_whitel"] = []
else:
    entities_removed["phone_llm_whitel"] = []
    print("PHONE-LLM-Adjudication deaktiviert oder PHO nicht aktiv → kein PHONE-LLM-Call.")

# LLM-Adjudication nur für ambige PERSON-Kandidaten.
llm_candidates = []
llm_exempt_counts = Counter()
llm_decision_by_key = {}
whitel_by_llm = []
review_candidates_for_whitelist = []
active_whitelist_whitels_written = 0
whitelist_candidates_written = 0  # Ab dieser Version: neu geschriebene LLM-REVIEW-Kandidaten
llm_decision_counts = Counter()

if USE_LLM_ENTITY_ADJUDICATION and "PER" in LLM_ENTITY_TYPES and "PER" in ENTITY_TYPES_ACTIVE:
    llm_candidates, llm_exempt_counts = build_llm_entity_candidates(entities_filtered, raw_text, speaker_roster=speaker_roster)
    print(f"LLM-Adjudication: {len(llm_candidates)} deduplizierte PERSON-Kandidaten, {sum(llm_exempt_counts.values())} Exemptions.")

    if llm_candidates:
        llm_decision_by_key = call_llm_entity_adjudication(llm_candidates)
        llm_decision_counts = Counter(d["decision"] for d in llm_decision_by_key.values())
        entities_filtered, whitel_by_llm = apply_llm_entity_decisions(
            entities_filtered,
            llm_decision_by_key,
        )
        entities_removed["llm_whitel"] = whitel_by_llm

        # Planänderung:
        # - LLM-WHITELs werden direkt in die aktive entity_whitelist.csv übernommen.
        # - LLM-REVIEWs werden in whitelist_candidates.csv für spätere Prüfung gesammelt.
        review_candidates_for_whitelist = [
            ent for ent in entities_filtered
            if ent.get("llm_checked") and ent.get("llm_decision") == "REVIEW"
        ]
        active_whitelist_whitels_written = append_llm_whitels_to_active_whitelist(
            EXTERNAL_ENTITY_WHITELIST_PATH,
            whitel_by_llm,
            source_file=input_filename,
        )
        whitelist_candidates_written = append_whitelist_candidates(
            WHITELIST_CANDIDATES_PATH,
            review_candidates_for_whitelist,
            source_file=input_filename,
        )
    else:
        entities_removed["llm_whitel"] = []
        print("LLM-Adjudication: Keine Kandidaten → kein LLM-Call.")
else:
    entities_removed["llm_whitel"] = []
    print("LLM-Adjudication deaktiviert oder PER nicht aktiv → kein LLM-Call.")

# Schreibpfad/CSV-Schema validieren und Dateien sichtbar machen, auch wenn in diesem Run
# keine LLM-WHITELs oder REVIEWs angefallen sind. Ein echtes Schreibproblem wird hier sichtbar.
_ensure_csv_header(EXTERNAL_ENTITY_WHITELIST_PATH, ACTIVE_ENTITY_WHITELIST_COLUMNS)
_ensure_csv_header(WHITELIST_CANDIDATES_PATH, WHITELIST_CANDIDATE_COLUMNS)

# Zusammenfassung
print("=" * 60)
print("POST-PROCESSING ZUSAMMENFASSUNG")
print("=" * 60)
print(f"Entities nach Merge:       {len(entities_merged)}")
print(f"Speaker-Roster:            {len(speaker_roster_full_names)} Vollnamen / {len(speaker_roster_name_parts)} Namensbestandteile")
print(f"Short-Segment Recall:      {globals().get('short_segment_recall_total_entities', 0)} Entities aus {len(globals().get('short_segment_recall_chunks', []))} Segmenten")
print(f"PHONE-Regex-Recall:        {globals().get('phone_regex_recall_total_entities', 0)} neue Kandidaten")
print(f"  Entfernt durch Threshold:       {len(entities_removed['threshold'])}")
print(f"  Entfernt durch Mindestlänge:    {len(entities_removed['min_length'])}")
print(f"  Entfernt durch Pronomen:        {len(entities_removed['pronoun'])}")
print(f"  Entfernt durch Whitelist:       {len(entities_removed['whitelist'])}")
print(f"  Entfernt durch PHONE-Resolver:    {len(entities_removed.get('phone_specific', []))}")
print(f"  Entfernt durch PHONE-LLM:       {len(entities_removed.get('phone_llm_whitel', []))}")
print(f"  Entfernt durch MAIL-Regeln:     {len(entities_removed.get('mail_specific', []))}")
print(f"  Entfernt durch PERSON-Regeln:   {len(entities_removed['person_specific'])}")
print(f"  Entfernt durch externe Whitelist: {len(entities_removed.get('external_whitelist', []))}")
print(f"  Entfernt durch LLM-WHITEL:        {len(entities_removed.get('llm_whitel', []))}")
print(f"Entities nach Post-Processing: {len(entities_after_postprocessing)}")
print(f"Propagierte Namensbestandteile: {len(propagated_name_parts)}")
print(f"Entities nach Name-Propagation: {len(entities_after_name_propagation)}")
print(f"PHONE-LLM-Kandidaten:      {len(phone_llm_candidates)}")
print(f"PHONE-LLM-Decisions:       {dict(phone_llm_decision_counts)}")
print(f"LLM-Kandidaten gesendet:   {len(llm_candidates)}")
print(f"LLM-Decisions:             {dict(llm_decision_counts)}")
print(f"Aktive Whitelist neu (LLM-WHITELs): {active_whitelist_whitels_written}")
print(f"Whitelist-Kandidaten neu (LLM-REVIEWs): {whitelist_candidates_written}")
print(f"Entities final:            {len(entities_filtered)}")

if propagated_name_parts:
    print("\n--- Propagierte Namensbestandteile ---")
    for e in sorted(propagated_name_parts, key=lambda x: x["text"].lower())[:30]:
        print(
            f"  '{e['text']}' → {e.get('tag', '?')}  "
            f"(aus '{e.get('propagated_from_full_name', '')}', score={e.get('score', 0):.3f})"
        )
    if len(propagated_name_parts) > 30:
        print(f"  ... und {len(propagated_name_parts) - 30} weitere")

# Details zu entfernten Entities
for reason, ents in entities_removed.items():
    if ents:
        print(f"\n--- Entfernt ({reason}) ---")
        for e in sorted(ents, key=lambda x: x["score"], reverse=True)[:20]:
            tag = e.get("tag", "?")
            detail = e.get("filter_detail", "")
            detail_str = f"  [{detail}]" if detail else ""
            print(f"  '{e['text']}' → {tag}  "
                  f"(score={e['score']:.3f}){detail_str}")
        if len(ents) > 20:
            print(f"  ... und {len(ents) - 20} weitere")


## Zelle 8: Entity-Liste
Alle finalen Entities gruppiert nach Tag.
⚠ markiert Entities mit niedriger Konfidenz.
→ Diese Liste zur Whitelist-Pflege verwenden.

In [ ]:
print("=" * 70)
print("ENTITY-LISTE (für Prüfung und Whitelist-Ergänzung)")
print("=" * 70)

ENTITY_EXPORT_BASE_COLS = [
    "text", "label", "tag", "config_key", "score",
    "llm_checked", "llm_decision", "llm_confidence", "llm_reason",
    "llm_exempt_reason", "external_whitelist_hit",
]
ENTITY_EXPORT_OPTIONAL_COLS = [
    "start",
    "end",
    "inference_source",
    "filter_detail",
    "phone_resolver_decision",
    "phone_sources",
    "phone_source_scores",
    "phone_evidence_count",
    "span_specific_replacement",
    "propagated_name_part",
    "propagated_from_full_name",
    "removed_reason",
]


def _ensure_export_columns(df, columns):
    for col in columns:
        if col not in df.columns:
            df[col] = ""
    return df


def _flatten_removed_entities(entities_removed):
    rows = []
    for reason, ents in entities_removed.items():
        for ent in ents:
            row = dict(ent)
            row["removed_reason"] = reason
            row["is_final_entity"] = False
            rows.append(row)
    return rows


df_entities = pd.DataFrame(entities_filtered)
entity_list_path = None
removed_entity_list_path = None
df_entity_diagnostics_export = pd.DataFrame()
df_removed_diagnostics_export = pd.DataFrame()
write_entity_diagnostic_csvs = bool(globals().get("WRITE_ENTITY_DIAGNOSTIC_CSVS", False))

if not df_entities.empty:
    df_entities = _ensure_export_columns(df_entities, ENTITY_EXPORT_BASE_COLS + ENTITY_EXPORT_OPTIONAL_COLS)
    df_entities = df_entities.sort_values(["tag", "text"])

    for tag_name in sorted(df_entities["tag"].unique()):
        group = df_entities[df_entities["tag"] == tag_name]

        # Threshold für diese Gruppe finden
        config_keys = group["config_key"].dropna().unique()
        thresholds = [LABEL_CONFIG[ck]["threshold"] for ck in config_keys
                      if ck in LABEL_CONFIG]
        threshold_str = "/".join(str(t) for t in thresholds)

        print(f"\n{'─' * 70}")
        print(f" {tag_name} ({len(group)} Treffer, threshold={threshold_str})")
        print(f"{'─' * 70}")

        for _, row in group.iterrows():
            warn = " ⚠ niedrige Konfidenz" if row["score"] < LOW_CONFIDENCE_WARN else ""
            label_info = row["label"]
            llm_info = ""
            if bool(row.get("llm_checked", False)):
                llm_info = f" | LLM={row.get('llm_decision', '')}"
            elif row.get("llm_exempt_reason", ""):
                llm_info = f" | LLM-exempt={row.get('llm_exempt_reason', '')}"
            print(f"  {row['text']:45s} → {tag_name}  "
                  f"(label={label_info}, score={row['score']:.3f}{llm_info}){warn}")

    # Diagnose-DataFrame intern aufbauen. CSV-Ausgabe ist optional.
    export_cols = ENTITY_EXPORT_BASE_COLS + [
        col for col in ENTITY_EXPORT_OPTIONAL_COLS if col in df_entities.columns
    ]
    df_export = df_entities[export_cols].copy()
    df_export["low_confidence"] = df_export["score"] < LOW_CONFIDENCE_WARN
    df_export["is_final_entity"] = True
    df_entity_diagnostics_export = df_export

    if write_entity_diagnostic_csvs:
        entity_list_path = os.path.join(LOCAL_OUTPUT_DIR, f"{input_basename}_entity_list.csv")
        os.makedirs(os.path.dirname(entity_list_path), exist_ok=True)
        df_export.to_csv(entity_list_path, index=False, encoding="utf-8")
        print(f"\nEntity-Liste gespeichert: {entity_list_path}")
    else:
        entity_list_path = None
        print("\nEntity-Liste wurde intern aufgebaut, aber nicht als CSV geschrieben "
              "(WRITE_ENTITY_DIAGNOSTIC_CSVS=False).")
else:
    print("Keine finalen Entities gefunden.")

# Removed-/Diagnose-CSV: macht externe Whitelist-Treffer und LLM-WHITELs nachvollziehbar.
removed_rows = _flatten_removed_entities(entities_removed)
if removed_rows:
    df_removed = pd.DataFrame(removed_rows)
    df_removed = _ensure_export_columns(
        df_removed,
        ENTITY_EXPORT_BASE_COLS + ENTITY_EXPORT_OPTIONAL_COLS + ["is_final_entity"],
    )
    removed_export_cols = ["removed_reason"] + ENTITY_EXPORT_BASE_COLS + [
        col for col in ENTITY_EXPORT_OPTIONAL_COLS
        if col in df_removed.columns and col != "removed_reason"
    ] + ["is_final_entity"]
    df_removed_diagnostics_export = df_removed[removed_export_cols].copy()

    if write_entity_diagnostic_csvs:
        removed_entity_list_path = os.path.join(
            LOCAL_OUTPUT_DIR,
            f"{input_basename}_removed_entities.csv",
        )
        df_removed_diagnostics_export.to_csv(
            removed_entity_list_path,
            index=False,
            encoding="utf-8",
        )
        print(f"Removed-/Diagnose-Entity-Liste gespeichert: {removed_entity_list_path}")
    else:
        removed_entity_list_path = None
        print("Removed-/Diagnose-Entity-Liste wurde intern aufgebaut, aber nicht als CSV geschrieben "
              "(WRITE_ENTITY_DIAGNOSTIC_CSVS=False).")
else:
    print("Keine entfernten Entities für Diagnose-CSV vorhanden.")


## Zelle 9: Text anonymisieren
Ersetzt gefundene Entities im Originaltext durch Tags.
Mode `tags` → `[PERSON]`, Mode `mask` → `********`

In [ ]:
import re


def _entity_replacement_pattern(ent):
    """
    Für propagierte einzelne Namensbestandteile boundary-sicher ersetzen,
    damit z.B. "Tom" nicht in "Timing" anonymisiert wird.
    Für normale Entities bleibt das bisherige exakte Textmatching erhalten.
    """
    entity_text = ent["text"].strip()
    if ent.get("propagated_name_part"):
        return _compile_name_token_pattern(entity_text)
    return re.compile(re.escape(entity_text))


def _entity_has_valid_absolute_span(text, ent):
    """Prüft, ob start/end im Rohtext exakt auf den Entity-Text zeigen."""
    try:
        start = int(ent.get("start"))
        end = int(ent.get("end"))
    except Exception:
        return False
    if start < 0 or end <= start or end > len(text):
        return False
    return text[start:end] == str(ent.get("text", ""))


def _span_text_key(entity_text, tag):
    """Robuster Schlüssel, um span-spezifische PHONE-Treffer gegen globale Text-Ersetzungen abzugrenzen."""
    normalized = re.sub(r"\s+", " ", str(entity_text or "").strip().lower())
    return (normalized, str(tag or ""))


def anonymize_text(text, entities, mode="mask"):
    """
    Ersetzt Entities im Text durch Maskierung (*) bzw. Tags.

    Standard bleibt das bisherige textbasierte Ersetzen. Für vorkommensabhängige
    PHONE-Treffer aus dem Regex-Recall wird jedoch spanbasiert ersetzt, damit
    dieselbe Ziffernfolge je nach Kontext unterschiedlich behandelt werden kann
    (z. B. Rufnummer behalten/anonymisieren, Kreditkartennummer nicht anfassen).
    """
    entities = list(entities or [])

    # Wenn es für einen PHONE-Text mindestens einen validen span-spezifischen Treffer gibt,
    # darf derselbe PHONE-Text nicht zusätzlich global ersetzt werden. Sonst würde im Beispiel
    # "Rufnummer 4443 ..." / "Kreditkartennummer 4443 ..." wieder beide Vorkommen treffen.
    span_specific_phone_text_keys = set()
    for ent in entities:
        entity_text = str(ent.get("text", "")).strip()
        if (
            ent.get("tag") == "[PHONE]"
            and ent.get("span_specific_replacement")
            and entity_text
            and _entity_has_valid_absolute_span(text, ent)
        ):
            span_specific_phone_text_keys.add(_span_text_key(entity_text, ent.get("tag")))

    span_replacements = []
    text_replacements = []
    seen_text = set()
    seen_spans = set()

    for ent in entities:
        entity_text = ent["text"].strip()
        if not entity_text:
            continue

        tag = ent.get("tag", "[UNKNOWN]")
        replacement = "*" * len(entity_text) if mode == "mask" else tag

        if ent.get("span_specific_replacement") and _entity_has_valid_absolute_span(text, ent):
            start = int(ent.get("start"))
            end = int(ent.get("end"))
            span_key = (start, end, tag)
            if span_key in seen_spans:
                continue
            seen_spans.add(span_key)
            span_replacements.append((start, end, replacement, ent))
            continue

        # PHONE-Kontextfälle mit gleichem Text werden ausschließlich über die validen Spans ersetzt.
        if ent.get("tag") == "[PHONE]" and _span_text_key(entity_text, tag) in span_specific_phone_text_keys:
            continue

        is_propagated = bool(ent.get("propagated_name_part"))
        key = (entity_text, tag, is_propagated)
        if key in seen_text:
            continue
        seen_text.add(key)
        text_replacements.append((entity_text, replacement, ent, is_propagated))

    # 1) Span-spezifische Ersetzungen zuerst in einer Maske sammeln.
    # Überlappungen werden deterministisch vermieden: längerer Span gewinnt.
    span_replacements.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    selected_spans = []
    current_end = -1
    for start, end, replacement, ent in span_replacements:
        if start < current_end:
            continue
        selected_spans.append((start, end, replacement, ent))
        current_end = end

    parts = []
    last = 0
    span_replacement_count = 0
    for start, end, replacement, ent in selected_spans:
        parts.append(text[last:start])
        parts.append(replacement)
        last = end
        span_replacement_count += 1
    parts.append(text[last:])
    result = "".join(parts)

    # 2) Textbasierte Ersetzungen bleiben wie bisher für alle nicht span-spezifischen Entities.
    text_replacements.sort(key=lambda x: (len(x[0]), not x[3]), reverse=True)

    replacement_count = span_replacement_count
    for original, replacement, ent, _ in text_replacements:
        pattern = _entity_replacement_pattern(ent)
        result, count = pattern.subn(lambda _: replacement, result)
        replacement_count += count

    return result, replacement_count

# Anonymisierung durchführen
anonymized_text, total_replacements = anonymize_text(
    raw_text, entities_filtered, mode="mask"
)

print(f"Ersetzungen durchgeführt: {total_replacements}")
print(f"\nBeispiel (erste 500 Zeichen des anonymisierten Texts):")
print(anonymized_text[:500] + "...")


## Zelle 10: Ergebnisse speichern
Speichert den anonymisierten Text abhängig von `runtime_mode`:

- `local`: lokal im Unterverzeichnis `gliner_output`, keine Blob-Schreibzugriffe.
- `prod`: lokal im DBFS-Arbeitsverzeichnis und anschließend v7-kompatibel in Blob-Container `data`; Vergleichs-HTML optional nach `logs/html/YYYY-MM-DD/...`.


In [ ]:
import html

# ============================================================
# 1) RexGuard Output (TXT) speichern
# ============================================================
# runtime_mode=local: lokal nach gliner_output, kein Blob-Upload.
# runtime_mode=prod:  lokal ins DBFS-Arbeitsverzeichnis, danach Upload nach Container data.

# lokal schreiben
os.makedirs(os.path.dirname(local_output_path), exist_ok=True)
with open(local_output_path, "w", encoding="utf-8") as f:
    f.write(anonymized_text)

print(f"Anonymisierter Text (lokal): {local_output_path}")

# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# if ALLOW_BLOB_WRITES:
#     upload_file_to_blob_storage(
#         blob_service_client,
#         DATA_CONTAINER_NAME,
#         anonymized_output_filename,
#         local_output_path,
#     )
# else:
#     print("Blob-Upload TXT: deaktiviert; lokale Datei bleibt in gliner_output.")
print("Blob-Upload TXT: deaktiviert (Colab); lokale Datei bleibt in gliner_output.")

# ============================================================
# 2) Vergleichs-HTML erzeugen (optional)
# ============================================================
# runtime_mode=local: lokal nach gliner_output, kein Blob-Upload.
# runtime_mode=prod:  lokal ins DBFS-Arbeitsverzeichnis, danach Upload nach logs/html/YYYY-MM-DD/...
def create_score_table_html(entities):
    """
    Erzeugt eine HTML-Tabelle aller finalen Entities, sortiert nach Score (absteigend).
    Enthält zusätzlich LLM-Diagnosefelder.
    """
    if not entities:
        return "<p>Keine Entities gefunden.</p>"

    sorted_ents = sorted(entities, key=lambda e: e.get("score", 0), reverse=True)

    tag_colors = {
        "[PERSON]": "#FF6B6B",
        "[ORG]": "#4ECDC4",
        "[LOC]": "#45B7D1",
        "[PHONE]": "#96CEB4",
        "[MAIL]": "#FFEAA7",
    }

    rows = []
    for ent in sorted_ents:
        tag = ent.get("tag", "?")
        color = tag_colors.get(tag, "#DDD")
        score = float(ent.get("score", 0))
        label = ent.get("label", "?")
        text = html.escape(ent.get("text", "?").strip())
        llm_checked = bool(ent.get("llm_checked", False))
        llm_decision = html.escape(str(ent.get("llm_decision", "")))
        llm_confidence = ent.get("llm_confidence", "")
        llm_reason = html.escape(str(ent.get("llm_reason", "")))
        llm_exempt_reason = html.escape(str(ent.get("llm_exempt_reason", "")))
        source = html.escape(str(ent.get("inference_source", "")))
        start_pos = html.escape(str(ent.get("start", "")))
        end_pos = html.escape(str(ent.get("end", "")))
        phone_resolver = html.escape(str(ent.get("phone_resolver_decision", "")))
        phone_sources = html.escape(str(ent.get("phone_sources", "")))
        filter_detail = html.escape(str(ent.get("filter_detail", "")))

        # Score-Farbe: grün > 0.8, gelb 0.6-0.8, rot < 0.6
        if score >= 0.8:
            score_color = "#27ae60"
        elif score >= 0.6:
            score_color = "#f39c12"
        else:
            score_color = "#e74c3c"

        if llm_checked:
            llm_status = llm_decision
        elif llm_exempt_reason:
            llm_status = f"exempt: {llm_exempt_reason}"
        else:
            llm_status = ""

        if isinstance(llm_confidence, (int, float)):
            llm_confidence_str = f"{float(llm_confidence):.2f}"
        else:
            llm_confidence_str = html.escape(str(llm_confidence))

        rows.append(
            f'<tr>'
            f'<td>{text}</td>'
            f'<td><span style="background-color:{color};padding:2px 6px;'
            f'border-radius:3px;">{html.escape(str(tag))}</span></td>'
            f'<td>{html.escape(str(label))}</td>'
            f'<td style="color:{score_color};font-weight:bold;text-align:right;">'
            f'{score:.3f}</td>'
            f'<td>{source}</td>'
            f'<td style="text-align:right;">{start_pos}</td>'
            f'<td style="text-align:right;">{end_pos}</td>'
            f'<td>{phone_resolver}</td>'
            f'<td>{phone_sources}</td>'
            f'<td>{filter_detail}</td>'
            f'<td>{html.escape(str(llm_status))}</td>'
            f'<td style="text-align:right;">{llm_confidence_str}</td>'
            f'<td>{llm_reason}</td>'
            f'</tr>'
        )

    return (
        '<table class="score-table">\n'
        '<thead><tr>'
        '<th>Entity</th><th>Tag</th><th>Label</th><th>Score</th>'
        '<th>Source</th><th>Start</th><th>End</th>'
        '<th>PHONE Resolver</th><th>PHONE Sources</th><th>Filter Detail</th>'
        '<th>LLM</th><th>LLM Conf.</th><th>LLM Reason</th>'
        '</tr></thead>\n'
        '<tbody>\n' + '\n'.join(rows) + '\n</tbody>\n</table>'
    )


def create_removed_entities_table_html(entities_removed):
    """
    Erzeugt eine Diagnose-Tabelle für entfernte Entities, inklusive LLM-WHITELs.

    Whitelist-Entfernungen werden absichtlich nicht mehr einzeln im HTML
    aufgeführt: Die Begriffe sind bereits kuratiert/erwartet und würden die
    Diagnoseansicht bei wachsender aktiver Whitelist unnötig aufblasen.
    Die Zählwerte bleiben oben in der HTML-Statistik sichtbar.
    """
    rows = []
    hidden_whitelist_reasons = {"whitelist", "external_whitelist"}
    hidden_whitelist_counts = {
        "Interne Whitelist": len(entities_removed.get("whitelist", [])),
        "Externe Whitelist": len(entities_removed.get("external_whitelist", [])),
    }
    hidden_whitelist_total = sum(hidden_whitelist_counts.values())

    for reason, ents in entities_removed.items():
        if reason in hidden_whitelist_reasons:
            continue

        for ent in ents:
            score = ent.get("score", 0)
            try:
                score = float(score)
                score_str = f"{score:.3f}"
            except Exception:
                score_str = html.escape(str(score))
            llm_confidence = ent.get("llm_confidence", "")
            if isinstance(llm_confidence, (int, float)):
                llm_confidence_str = f"{float(llm_confidence):.2f}"
            else:
                llm_confidence_str = html.escape(str(llm_confidence))

            rows.append(
                f'<tr>'
                f'<td>{html.escape(str(reason))}</td>'
                f'<td>{html.escape(str(ent.get("text", "")))}</td>'
                f'<td>{html.escape(str(ent.get("tag", "")))}</td>'
                f'<td style="text-align:right;">{score_str}</td>'
                f'<td>{html.escape(str(ent.get("inference_source", "")))}</td>'
                f'<td style="text-align:right;">{html.escape(str(ent.get("start", "")))}</td>'
                f'<td style="text-align:right;">{html.escape(str(ent.get("end", "")))}</td>'
                f'<td>{html.escape(str(ent.get("phone_resolver_decision", "")))}</td>'
                f'<td>{html.escape(str(ent.get("phone_sources", "")))}</td>'
                f'<td>{html.escape(str(ent.get("filter_detail", "")))}</td>'
                f'<td>{html.escape(str(ent.get("llm_decision", "")))}</td>'
                f'<td style="text-align:right;">{llm_confidence_str}</td>'
                f'<td>{html.escape(str(ent.get("llm_reason", "")))}</td>'
                f'</tr>'
            )

    hidden_note = ""
    if hidden_whitelist_total:
        hidden_parts = [
            f"{label}: {count}"
            for label, count in hidden_whitelist_counts.items()
            if count
        ]
        hidden_note = (
            '<p><em>Whitelist-Entfernungen werden hier nicht einzeln gelistet. '
            f'Ausgeblendete Whitelist-Treffer: {html.escape(", ".join(hidden_parts))}.</em></p>'
        )

    if not rows:
        if hidden_note:
            return hidden_note + "<p>Keine weiteren entfernten Entities.</p>"
        return "<p>Keine entfernten Entities.</p>"

    table = (
        '<table class="score-table">\n'
        '<thead><tr>'
        '<th>Removed Reason</th><th>Entity</th><th>Tag</th><th>Score</th>'
        '<th>Source</th><th>Start</th><th>End</th>'
        '<th>PHONE Resolver</th><th>PHONE Sources</th>'
        '<th>Filter Detail</th><th>LLM</th><th>LLM Conf.</th><th>LLM Reason</th>'
        '</tr></thead>\n'
        '<tbody>\n' + '\n'.join(rows) + '\n</tbody>\n</table>'
    )
    return hidden_note + table


def _collect_non_overlapping_entity_matches(original_text, entities):
    """
    Sammelt alle Vorkommen der Entities im Originaltext und wählt eine
    nicht-überlappende Menge. Dadurch werden propagierte Einzel-Namen wie
    "Tom" nicht in bereits markierten Vollnamen wie "Tom Richter" erneut
    in HTML-Markup ersetzt.
    """
    candidates = []

    for ent in entities:
        entity_text = ent.get("text", "").strip()
        if not entity_text:
            continue

        if ent.get("span_specific_replacement"):
            try:
                start = int(ent.get("start"))
                end = int(ent.get("end"))
            except Exception:
                start = end = -1
            if 0 <= start < end <= len(original_text) and original_text[start:end] == entity_text:
                candidates.append({
                    "start": start,
                    "end": end,
                    "ent": ent,
                    "length": end - start,
                })
                continue

        pattern = _entity_replacement_pattern(ent)
        for match in pattern.finditer(original_text):
            if match.start() == match.end():
                continue
            candidates.append({
                "start": match.start(),
                "end": match.end(),
                "ent": ent,
                "length": match.end() - match.start(),
            })

    # Gleicher Start: längster/highest-score zuerst; normale Treffer vor propagierten.
    candidates.sort(
        key=lambda c: (
            c["start"],
            -c["length"],
            bool(c["ent"].get("propagated_name_part")),
            -float(c["ent"].get("score", 0)),
        )
    )

    selected = []
    current_end = -1
    for cand in candidates:
        if cand["start"] < current_end:
            continue
        selected.append(cand)
        current_end = cand["end"]

    return selected


def _render_marked_text_html(original_text, entities, label_config, tag_colors):
    """Rendert den Originaltext mit sicheren, nicht-überlappenden Highlights."""
    matches = _collect_non_overlapping_entity_matches(original_text, entities)

    parts = []
    pos = 0
    for match in matches:
        start = match["start"]
        end = match["end"]
        ent = match["ent"]

        parts.append(html.escape(original_text[pos:start]))

        matched_text = original_text[start:end]
        tag = ent.get("tag", "[UNKNOWN]")
        config_key = ent.get("config_key", "")
        threshold = label_config.get(config_key, {}).get("threshold", "?")
        color = tag_colors.get(tag, "#DDD")
        score = float(ent.get("score", 0))
        warn_style = "border: 2px dashed orange;" if score < LOW_CONFIDENCE_WARN else ""
        propagated_info = " | propagated=true" if ent.get("propagated_name_part") else ""
        propagated_from = ent.get("propagated_from_full_name", "")
        if propagated_from:
            propagated_info += f" | from={propagated_from}"

        llm_info = ""
        if ent.get("llm_checked"):
            llm_info = (
                f" | LLM={ent.get('llm_decision', '')}"
                f" | LLM_conf={ent.get('llm_confidence', '')}"
                f" | LLM_reason={ent.get('llm_reason', '')}"
            )
        elif ent.get("llm_exempt_reason"):
            llm_info = f" | LLM_exempt={ent.get('llm_exempt_reason', '')}"

        phone_info = ""
        if tag == "[PHONE]":
            phone_info = (
                f" | PHONE_resolver={ent.get('phone_resolver_decision', '')}"
                f" | PHONE_sources={ent.get('phone_sources', '')}"
                f" | detail={ent.get('filter_detail', '')}"
            )

        title = (
            f"{tag} | label={ent.get('label', '?')} | score={score:.3f} "
            f"| threshold={threshold}{propagated_info}{phone_info}{llm_info}"
        )

        highlight = (
            f'<span style="background-color:{color};padding:2px 4px;border-radius:3px;'
            f'{warn_style}cursor:help;" '
            f'title="{html.escape(title, quote=True)}">'
            f'{html.escape(matched_text)}</span>'
        )
        parts.append(highlight)
        pos = end

    parts.append(html.escape(original_text[pos:]))
    return "".join(parts).replace("\n", "<br>\n")


def create_comparison_html(original_text, entities, label_config, output_path):
    """
    HTML-Datei mit:
    1. Score-Übersichtstabelle (alle Entities, sortiert nach Score)
    2. Farblich markierter Originaltext mit Tooltips
    """
    tag_colors = {
        "[PERSON]": "#FF6B6B",
        "[ORG]": "#4ECDC4",
        "[LOC]": "#45B7D1",
        "[PHONE]": "#96CEB4",
        "[MAIL]": "#FFEAA7",
    }

    # --- Score-/Diagnose-Tabellen ---
    score_table = create_score_table_html(entities)
    removed_table = create_removed_entities_table_html(entities_removed)

    # --- Markierter Text ---
    marked_text = _render_marked_text_html(original_text, entities, label_config, tag_colors)

    # --- Legende ---
    legend_items = []
    for config_key, cfg in label_config.items():
        tag = cfg["tag"]
        color = tag_colors.get(tag, "#DDD")
        sublabels = ", ".join(l.strip() for l in config_key.split(","))
        neg_labels = cfg.get("negative_labels", [])
        neg_info = f" (neg: {', '.join(neg_labels)})" if neg_labels else ""
        legend_items.append(
            f'<span style="background-color:{color};padding:4px 8px;border-radius:4px;'
            f'margin-right:10px;">{tag} [{sublabels}]{neg_info} '
            f'(≥{cfg["threshold"]})</span>'
        )
    legend_items.append(
        '<span style="border:2px dashed orange;padding:4px 8px;border-radius:4px;'
        'margin-right:10px;">⚠ niedrige Konfidenz</span>'
    )
    legend = " ".join(legend_items)

    # --- Filter-Statistik ---
    filter_stats = (
        f"Threshold: {len(entities_removed['threshold'])} | "
        f"Mindestlänge: {len(entities_removed['min_length'])} | "
        f"Pronomen: {len(entities_removed['pronoun'])} | "
        f"Interne Whitelist: {len(entities_removed['whitelist'])} | "
        f"PHONE-Resolver: {len(entities_removed.get('phone_specific', []))} | "
        f"PHONE-LLM: {len(entities_removed.get('phone_llm_whitel', []))} | "
        f"MAIL-Regeln: {len(entities_removed.get('mail_specific', []))} | "
        f"PERSON-Regeln: {len(entities_removed.get('person_specific', []))} | "
        f"Externe Whitelist: {len(entities_removed.get('external_whitelist', []))} | "
        f"LLM-WHITEL: {len(entities_removed.get('llm_whitel', []))} | "
        f"Aktive Whitelist neu: {globals().get('active_whitelist_whitels_written', 0)} | "
        f"REVIEW-Kandidaten neu: {globals().get('whitelist_candidates_written', 0)} | "
        f"LLM-Kandidaten: {len(globals().get('llm_candidates', []))} | "
        f"Short-Segment Recall: {globals().get('short_segment_recall_total_entities', 0)} | "
        f"PHONE-Regex-Recall: {globals().get('phone_regex_recall_total_entities', 0)} | "
        f"Namensbestandteile propagiert: {len(globals().get('propagated_name_parts', []))}"
    )

    # --- Durchlauf-Info ---
    pass_parts = []
    for i, (ck, cfg_item) in enumerate(label_config.items(), 1):
        pos = ", ".join(l.strip() for l in ck.split(","))
        neg = cfg_item.get("negative_labels", [])
        neg_str = f" + neg:[{', '.join(neg)}]" if neg else ""
        pass_parts.append(f"Durchlauf {i}: [{pos}]{neg_str}")
    pass_info = " → ".join(pass_parts)
    recall_stats = globals().get("short_segment_recall_stats", [])
    if recall_stats:
        recall_parts = [
            f"Recall {s.get('entity_type')}: {s.get('entities')} Entities / {s.get('segments')} Segmente"
            for s in recall_stats
        ]
        pass_info += " | Short-Segment " + " ; ".join(recall_parts)

    # --- Negativ-Label Statistik ---
    neg_stats = ""
    if total_neg_rejected > 0:
        neg_stats = (
            f'<div class="neginfo">'
            f'<strong>Negativ-Labels:</strong> {total_neg_rejected} Entities '
            f'umgelenkt (korrekt als Negativ-Label klassifiziert und verworfen)'
            f'</div>'
        )

    html = f"""<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <title>NER Vergleichsansicht — {input_basename}</title>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px;
               margin: 40px auto; padding: 0 20px; line-height: 1.6; color: #333; }}
        h1 {{ color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
        h2 {{ color: #34495e; margin-top: 30px; }}
        .legend {{ background: #f8f9fa; padding: 15px; border-radius: 8px;
                   margin-bottom: 15px; }}
        .stats {{ background: #eaf2f8; padding: 15px; border-radius: 8px;
                  margin-bottom: 15px; }}
        .filters {{ background: #fef9e7; padding: 15px; border-radius: 8px;
                    margin-bottom: 15px; font-size: 0.9em; }}
        .passes {{ background: #eafaf1; padding: 15px; border-radius: 8px;
                   margin-bottom: 15px; font-size: 0.9em; }}
        .neginfo {{ background: #fdebd0; padding: 15px; border-radius: 8px;
                    margin-bottom: 20px; font-size: 0.9em; }}
        .content {{ background: #fff; padding: 20px; border: 1px solid #e0e0e0;
                    border-radius: 8px; }}
        .score-table {{ width: 100%; border-collapse: collapse; margin-bottom: 20px;
                        font-size: 0.9em; }}
        .score-table th {{ background: #2c3e50; color: white; padding: 8px 12px;
                           text-align: left; position: sticky; top: 0; }}
        .score-table td {{ padding: 6px 12px; border-bottom: 1px solid #e0e0e0; }}
        .score-table tr:hover {{ background: #f5f5f5; }}
        .score-table tr:nth-child(even) {{ background: #fafafa; }}
        .score-table tr:nth-child(even):hover {{ background: #f0f0f0; }}
        .table-container {{ max-height: 500px; overflow-y: auto; border: 1px solid #e0e0e0;
                            border-radius: 8px; margin-bottom: 30px; }}
    </style>
</head>
<body>
    <h1>NER Vergleichsansicht (v8.17 - PHONE Resolver + PERSON Recall + MAIL Filter)</h1>
    <div class="stats">
        <strong>Datei:</strong> {input_basename} |
        <strong>Modell:</strong> {MODEL_LOAD_SOURCE} |
        <strong>Entities final:</strong> {len(entities)} |
        <strong>Ersetzungen:</strong> {total_replacements}
    </div>
    <div class="passes">
        <strong>Inferenz-Strategie:</strong> {pass_info}
    </div>
    {neg_stats}
    <div class="legend">
        <strong>Legende:</strong> {legend}
    </div>
    <div class="filters">
        <strong>Entfernte Entities:</strong> {filter_stats}
    </div>

    <h2>Score-Übersicht ({len(entities)} finale Entities)</h2>
    <div class="table-container">
        {score_table}
    </div>

    <h2>Entfernte Entities / Diagnose</h2>
    <div class="table-container">
        {removed_table}
    </div>

    <h2>Markierter Text</h2>
    <div class="content">
        {marked_text}
    </div>
</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)


html_path_local = None
html_blob_path = None

if GENERATE_HTML:
    html_path_local = os.path.join(PRIMARY_OUTPUT_DIR, f"{input_basename}_RexGuard_compare.html")
    create_comparison_html(raw_text, entities_filtered, LABEL_CONFIG, html_path_local)
    print(f"Vergleichs-HTML (lokal): {html_path_local}")

    # --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
    # if ALLOW_BLOB_WRITES:
    #     today_str = datetime.utcnow().strftime("%Y-%m-%d")
    #     html_blob_path = f"{HTML_PREFIX}/{today_str}/{input_basename}_RexGuard_compare.html"
    #     upload_file_to_blob_storage(
    #         blob_service_client,
    #         LOGS_CONTAINER_NAME,
    #         html_blob_path,
    #         html_path_local,
    #     )
    # else:
    #     print("Blob-Upload HTML: deaktiviert; lokale HTML-Datei bleibt im gliner_output-Verzeichnis.")
    print("Blob-Upload HTML: deaktiviert (Colab); lokale HTML-Datei bleibt im gliner_output-Verzeichnis.")
else:
    print("GENERATE_HTML=False → Vergleichs-HTML wird nicht erzeugt.")

# ============================================================
# 3) Zusammenfassung
# ============================================================
print("\n" + "=" * 60)
print("ZUSAMMENFASSUNG")
print("=" * 60)
# Colab-Portierung: Input ist immer lokal.
print(f"Input (lokal):       {local_input_path}")
# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# if globals().get("INPUT_SOURCE_MODE") == "latest_blob":
#     print(f"Input (Blob):        {input_blob_name}")
# else:
#     print(f"Input (lokal):       {local_input_path}")
print(f"Modell:              {MODEL_LOAD_SOURCE}")
print(f"Wörter im Text:      {word_count:,}")
print(f"Chunks verarbeitet:  {len(chunks)}")
print(f"Short-Segment Recall: {globals().get('short_segment_recall_total_entities', 0)} Entities aus {len(globals().get('short_segment_recall_chunks', []))} Segmenten")
print(f"PHONE-Regex-Recall: {globals().get('phone_regex_recall_total_entities', 0)} neue Kandidaten")
print(f"Inferenz-Durchläufe: {len(LABEL_CONFIG)}")

for i, (ck, cfg) in enumerate(LABEL_CONFIG.items(), 1):
    pos = [l.strip() for l in ck.split(",")]
    neg = cfg.get("negative_labels", [])
    _, pass_ents = normal_pass_results[i - 1] if "normal_pass_results" in globals() else all_pass_results[i - 1]
    neg_str = f"  + neg:{neg}" if neg else ""
    print(f"  Durchlauf {i}: {pos} → {cfg['tag']}  "
          f"({len(pass_ents)} Entities){neg_str}")

if globals().get("short_segment_recall_stats"):
    print("\nShort-Segment Recall:")
    for s in short_segment_recall_stats:
        print(
            f"  {s['entity_type']} {s['tag']}: {s['entities']} Entities "
            f"aus {s['segments']} Segmenten (threshold={s['threshold']})"
        )

if total_neg_rejected > 0:
    print(f"\nDurch Negativ-Labels umgelenkt: {total_neg_rejected}")

print(f"\nEntities nach Merge: {len(entities_merged)}")
print(f"Entities final:      {len(entities_filtered)}")
print(f"  davon entfernt:")
print(f"    Threshold:       {len(entities_removed['threshold'])}")
print(f"    Mindestlänge:    {len(entities_removed['min_length'])}")
print(f"    Pronomen:        {len(entities_removed['pronoun'])}")
print(f"    Interne Whitelist: {len(entities_removed['whitelist'])}")
print(f"    PHONE-Resolver:    {len(entities_removed.get('phone_specific', []))}")
print(f"    PHONE-LLM:       {len(entities_removed.get('phone_llm_whitel', []))}")
print(f"    MAIL-Regeln:     {len(entities_removed.get('mail_specific', []))}")
print(f"    PERSON-Regeln:     {len(entities_removed.get('person_specific', []))}")
print(f"    Externe Whitelist: {len(entities_removed.get('external_whitelist', []))}")
print(f"    LLM-WHITEL:          {len(entities_removed.get('llm_whitel', []))}")
print(f"  Namensbestandteile propagiert: {len(globals().get('propagated_name_parts', []))}")
print(f"  LLM-Kandidaten gesendet:       {len(globals().get('llm_candidates', []))}")
print(f"  Aktive Whitelist neu (LLM-WHITELs): {globals().get('active_whitelist_whitels_written', 0)}")
print(f"  Whitelist-Kandidaten neu (LLM-REVIEWs): {globals().get('whitelist_candidates_written', 0)}")
print(f"Ersetzungen im Text: {total_replacements}")

print(f"\nOutput (lokal, Blob-Upload deaktiviert):")
print(f"  TXT lokal:         {local_output_path}")

print(f"\nZusatz-Artefakte (lokal):")
if entity_list_path:
    print(f"  Entity-Liste (CSV): {entity_list_path}")

if globals().get("removed_entity_list_path"):
    print(f"  Removed-/Diagnose-CSV: {removed_entity_list_path}")

if globals().get("EXTERNAL_ENTITY_WHITELIST_PATH"):
    print(f"  Aktive Entity-Whitelist: {EXTERNAL_ENTITY_WHITELIST_PATH}")

if globals().get("WHITELIST_CANDIDATES_PATH"):
    print(f"  Whitelist-Kandidaten/REVIEWs: {WHITELIST_CANDIDATES_PATH}")

if GENERATE_HTML and html_path_local:
    print(f"  Vergleichs-HTML lokal: {html_path_local}")

# --- AZURE-SPEZIFISCH: deaktiviert (Colab-Portierung) ---
# if GENERATE_HTML and html_blob_path:
#     print(f"  Vergleichs-HTML Blob: {LOGS_CONTAINER_NAME}/{html_blob_path}")

print("=" * 60)


## Nächste Schritte

1. **Entity-Liste prüfen** (`entity_list.csv`): Zeigt alle finalen Entities mit Score, Label, Konfidenz-Flag und `filter_detail`.
2. **Vergleichs-HTML öffnen**: Originaltext mit farblich markierten Entities. Tooltip zeigt Score, Label UND Threshold.
3. **PERSON-Recall-Filter prüfen**: `PERSON_MULTI_TOKEN_MIN_SCORE` und `PERSON_SINGLE_TOKEN_MIN_SCORE` stehen bewusst auf `0.5`. Einwort-PERSON-Treffer werden im Zweifel behalten.
4. **Whitelist kritisch prüfen**: Begriffe in `WHITELIST` werden weiterhin entfernt. Achtung: Dort enthaltene echte Namen, z. B. bekannte Vornamen, werden nicht anonymisiert.
5. **Harte Non-Person-Muster erweitern**: Wenn technische Begriffe weiterhin als PERSON auftauchen, zuerst `_looks_like_hard_non_person(...)` oder die `WHITELIST` ergänzen.
6. **Threshold nur vorsichtig erhöhen**: Ein höherer GLiNER-Threshold für PERSON kann echte Einzelnamen wieder entfernen.
7. **Mask-Mode**: Für Sternchen-Anonymisierung in Zelle 9 `mode="tags"` → `mode="mask"` ändern.
